# Non-Recurrent Graph Neural Network Method

This notebook's objective is to completely rewrite the pipeline for the Deep Learning project task. 
The implementation is based on:
- Version 2 of the project
- PyTorch workflow implementation from **Deeplearning.ai**

In [1]:
# Deep Learning and PyTorch
import torch
import torch_geometric
from torch_geometric.data import Data, Dataset
from torch_geometric.loader import DataLoader
import torchmetrics
from torch_geometric.utils import dense_to_sparse
from torchinfo import summary

# Data Processing
import numpy as np

# Add specific path for local imports
import sys
helper_utils_path = "/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/non_recurrent_gnn/flexible_gatconv2"
if helper_utils_path not in sys.path:
    sys.path.insert(0, helper_utils_path)

import helper_utils

# System and File Operations
import os
import glob

# Type Hints
from typing import Dict, Tuple, Union, List, Optional

# Utilities
import warnings
import configparser
import matplotlib.pyplot as plt
import pickle
import importlib
importlib.reload(helper_utils)
helper_utils.set_seed(42)
import optuna
from pprint import pprint
import inspect
from pathlib import Path

# Configuration
warnings.filterwarnings("ignore", category=UserWarning)

# Version Information
print(f"Torch version: {torch.__version__}")
print(f"Cuda available: {torch.cuda.is_available()}")
print(f"Torch geometric version: {torch_geometric.__version__}")

Torch version: 2.6.0+cu124
Cuda available: True
Torch geometric version: 2.7.0


## 1. Dataset

### 1.1. Features

#### Node Features

In [2]:
def get_node_features(data: np.ndarray, channels_first: bool = True) -> torch.Tensor:
    """
    Extract node features from fNIRS data using statistical measures.

    Args:
        data: Input fNIRS data array
        channels_first: If True, data shape is (C, T), else (T, C)

    Returns:
        all_node_feats: Tensor of shape (C, F) with statistical features per channel
    """
    stats = helper_utils.compute_statistical_features(data, channels_first=channels_first)

    FEATURE_KEYS = ("mean", "min", "max", "skewness", "kurtosis", "variance")

    C = len(stats["mean"])
    F = len(FEATURE_KEYS)
    all_node_feats = np.empty((C, F), dtype=np.float64)

    for i in range(C):
        all_node_feats[i, 0] = stats["mean"][i]
        all_node_feats[i, 1] = stats["min"][i]
        all_node_feats[i, 2] = stats["max"][i]
        all_node_feats[i, 3] = stats["skewness"][i]
        all_node_feats[i, 4] = stats["kurtosis"][i]
        all_node_feats[i, 5] = stats["variance"][i]

    return torch.tensor(all_node_feats, dtype=torch.float)

#### Edge Features

| Graph Type | Formula | Example (N=23) |
|------------|---------|----------------|
| **Undirected, no self-loops** | $\frac{N(N-1)}{2}$ | 253 |
| **Undirected, with self-loops** | $\frac{N(N+1)}{2}$ | 276 |
| **Directed, no self-loops** | $N(N-1)$ | 506 |
| **Directed, with self-loops** | $N^2$ | 529 |

Where:
- **Undirected**: Each edge `(i, j)` appears once
- **Directed**: Each edge can be `(i→j)` and `(j→i)` separately
- **Self-loops**: Edges where `i = j` (e.g., `(0→0)`)

In [3]:
def get_edge_features(hb_data: np.ndarray, fs: float, directed: bool = False,
                     corr_threshold: float = 0.0, self_loops: bool = False) -> Tuple[torch.Tensor, torch.Tensor]:
    """
    Extract edge features from fNIRS data using correlation and coherence.

    Args:
        hb_data: fNIRS hemoglobin data array
        fs: Sampling frequency
        directed: If True, create directed graph; if False, undirected
        corr_threshold: Minimum absolute correlation threshold for edge inclusion.
                       Only edges with |correlation| > corr_threshold are kept.
                       Default 0.0 includes all edges.
        self_loops: If True, include self-loops (i=j); if False, exclude them.
                   Default False excludes self-loops.

    Returns:
        edge_index: Tensor of shape (2, E) with edge indices
        edge_attr: Tensor of shape (E, 2) with [abs_correlation, coherence] features
    """
    # Ensure data is in correct orientation (channels x time)
    if hb_data.shape[0] != 23 and hb_data.shape[1] == 23:
        hb_data = hb_data.T

    # Compute Pearson correlation matrix
    R = helper_utils.pearson_correlation_matrix(hb_data, channels_first=True)

    # Take absolute value (like fMRI code)
    R_abs = np.abs(R)

    # Compute coherence matrix
    coh_mat, _, _ = helper_utils.coherence_matrix(hb_data, fs=fs, coherence_ratio='1/3',
                                                      channels_first=True, return_spectrum=False)

    assert R_abs.shape == coh_mat.shape and R_abs.ndim == 2 and R_abs.shape[0] == R_abs.shape[1], \
        "R_abs and coh_mat must be square matrices of the same shape"

    C = R_abs.shape[0]

    if not directed:
        # Undirected graph: take upper triangle
        k = 0 if self_loops else 1  # k=0 includes diagonal, k=1 excludes it
        i, j = np.triu_indices(C, k=k)

        # Filter edges: absolute correlation > threshold
        mask = R_abs[i, j] > corr_threshold
        i, j = i[mask], j[mask]

        edge_index = np.vstack([i, j])                    # (2, E)
        # Store absolute correlation values
        all_edge_feats = np.column_stack([R_abs[i, j], coh_mat[i, j]])  # (E, 2)
    else:
        # Directed graph: include all pairs (or all except diagonal)
        i_list, j_list = [], []
        for i in range(C):
            for j in range(C):
                # Skip self-loops if not enabled
                if i == j and not self_loops:
                    continue

                # Only include edge if absolute correlation > threshold
                if R_abs[i, j] > corr_threshold:
                    i_list.append(i)
                    j_list.append(j)

        i = np.asarray(i_list, dtype=int)
        j = np.asarray(j_list, dtype=int)
        edge_index = np.vstack([i, j])                    # (2, E)
        # Store absolute correlation values
        all_edge_feats = np.column_stack([R_abs[i, j], coh_mat[i, j]])  # (E, 2)

    return torch.tensor(edge_index, dtype=torch.long), torch.tensor(all_edge_feats, dtype=torch.float)

### 1.2. Graph Dataset

In [4]:
class fNIRSGraphDatasetNonRecurrent(Dataset):
    def __init__(
        self,
        root: Union[str, Path],
        data_type: str,
        max_trials: int,
        directed: bool = False,
        corr_threshold: float = 0.1,
        self_loops: bool = False,
        transform=None,
        pre_transform=None,
        pre_filter=None
    ):
        self.root = root
        self.data_type = data_type
        self.max_trials = max_trials
        self.directed = directed
        self.corr_threshold = corr_threshold
        self.self_loops = self_loops
        self.data_list = []

        super().__init__(
            root,
            transform,
            pre_transform,
            pre_filter
        )

    @property
    def raw_file_names(self):
        return []

    @property
    def processed_file_names(self):
        return []

    def process(self):
        base = Path(self.root)
        label = {"healthy": 0, "anxiety": 1}

        for label_name in ("healthy", "anxiety"):
            label_dir = base / label_name

            for subj_dir in sorted([p for p in label_dir.iterdir() if p.is_dir()]):
                subj_id = subj_dir.name

                # Load metadata to get sampling frequency
                sfreq = None
                ini_path = subj_dir / f"{subj_id}.data"
                if ini_path.exists():
                    cfg = configparser.ConfigParser()
                    cfg.read(ini_path)
                    sfreq = float(cfg["metadata"]["sfreq"])
                fs = float(sfreq) if (sfreq is not None and np.isfinite(sfreq)) else 10.1

                hb_dir = subj_dir / self.data_type
                if not hb_dir.exists():
                    continue

                trial_files = sorted(hb_dir.glob("*.npy"),  key=lambda p: int(p.stem))
                if self.max_trials is not None and self.max_trials > 0:
                    trial_files = trial_files[: self.max_trials]

                for trial_path in trial_files:
                    trial_idx = int(trial_path.stem)

                    # Load one trial → (C, T) or (T, C)
                    arr = np.load(trial_path)
                    if arr.ndim != 2:
                        raise ValueError(f"Expected 2D array, got shape {arr.shape} at {trial_path}")

                    # Node features
                    node_feats = get_node_features(arr, channels_first=True)

                    # Spatial features
                    edge_index, edge_feats = get_edge_features(arr, fs=fs, directed=self.directed, corr_threshold=self.corr_threshold, self_loops=self.self_loops)

                    # Label
                    y = torch.tensor([label[label_name]], dtype=torch.long)

                    # Create Data object
                    data = Data(
                        x=node_feats,
                        edge_index=edge_index, edge_attr=edge_feats,
                        y=y,
                        subject_id=subj_id, trial_idx=trial_idx, label_str=label_name
                    )

                    # If transform is defined, apply it
                    if self.transform is not None:
                        data = self.transform(data)

                    self.data_list.append(data)

    def len(self) -> int:
        return len(self.data_list)

    def get(self, idx: int) -> Data:
        return self.data_list[idx]

In [5]:
root_dir = r'/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/data/processed-old/GNG'
DATA_TYPE = 'hbo'
MAX_TRIALS = 2
dataset = fNIRSGraphDatasetNonRecurrent(
    root=root_dir,
    data_type=DATA_TYPE,
    max_trials=MAX_TRIALS,
    directed=True,
    corr_threshold=0.1,
    self_loops=True
)

data = dataset[1]
print(f"Dataset Keys: {data.keys()}")
print("-" * 50)
print(f"Node Features Shape: {data.x.shape}")
print(f"Example of Node Features:\n{data.x}")
print("-" * 50)
print(f"Edge Features Shape: {data.edge_attr.shape}")
print(f"Example of Edge Features:\n{data.edge_attr}")

# # sample_data.edge_attr to txt
# tx = data.edge_attr.numpy()
# np.savetxt('edge_attr.txt', tx, fmt='%f')

Dataset Keys: ['y', 'edge_attr', 'edge_index', 'trial_idx', 'subject_id', 'x', 'label_str']
--------------------------------------------------
Node Features Shape: torch.Size([23, 6])
Example of Node Features:
tensor([[ 4.3592e-17, -1.3772e+00,  1.5530e+00,  4.0232e-01,  2.0257e+00,
          6.8655e-01],
        [ 4.3592e-17, -1.1432e+00,  1.7385e+00,  5.4092e-01,  2.3534e+00,
          6.8852e-01],
        [ 0.0000e+00, -1.5214e+00,  1.3608e+00, -2.8467e-01,  1.8703e+00,
          8.6849e-01],
        [-1.9616e-16, -1.7255e+00,  1.0347e+00, -7.3868e-01,  2.2695e+00,
          6.8724e-01],
        [ 8.7183e-17, -1.9393e+00,  1.2182e+00, -7.4729e-01,  2.6944e+00,
          7.8803e-01],
        [-4.3592e-17, -2.4483e+00,  8.6379e-01, -1.1384e+00,  3.3323e+00,
          8.0510e-01],
        [ 8.7183e-17, -1.8479e+00,  1.2345e+00, -2.6708e-01,  1.7591e+00,
          8.8658e-01],
        [-2.6155e-16, -2.1843e+00,  1.3869e+00, -4.0625e-01,  2.3358e+00,
          8.3282e-01],
        [ 8.71

Processing...
Done!


#### Find `mean` and `std` of `Dataset`

In [6]:
def get_mean_std(dataset: Dataset) -> Tuple[Dict[str, torch.Tensor], Dict[str, torch.Tensor]]:
    """
    Calculate mean and standard deviation of node features and edge features across all graphs in the dataset.

    Args:
        dataset: fNIRSGraphDatasetNonRecurrent instance

    Returns:
        mean_dict: Dictionary with 'node_features' and 'edge_features' means
        std_dict: Dictionary with 'node_features' and 'edge_features' stds
    """
    node_features_list = []
    edge_features_list = []

    for graph in dataset:
        # graph.x shape: (num_nodes, num_node_features)
        node_features_list.append(graph.x)

        # graph.edge_attr shape: (num_edges, num_edge_features)
        if graph.edge_attr is not None:
            edge_features_list.append(graph.edge_attr)

    # Stack all node features: (total_node_instances, num_node_features)
    all_node_features = torch.cat(node_features_list, dim=0)

    # Stack all edge features: (total_edge_instances, num_edge_features)
    all_edge_features = torch.cat(edge_features_list, dim=0) if edge_features_list else None

    # Compute statistics
    node_mean = all_node_features.mean(dim=0)
    node_std = all_node_features.std(dim=0)

    mean_dict = {'node_features': node_mean}
    std_dict = {'node_features': node_std}

    if all_edge_features is not None:
        edge_mean = all_edge_features.mean(dim=0)
        edge_std = all_edge_features.std(dim=0)
        mean_dict['edge_features'] = edge_mean
        std_dict['edge_features'] = edge_std

    return mean_dict, std_dict

In [7]:
# # Calculate dataset statistics
# mean_dict, std_dict = get_mean_std(dataset)

# print("Node Features Statistics:")
# print(f"  Mean shape: {mean_dict['node_features'].shape}")
# print(f"  Mean: {mean_dict['node_features']}")
# print(f"  Std shape: {std_dict['node_features'].shape}")
# print(f"  Std: {std_dict['node_features']}")

# if 'edge_features' in mean_dict:
#     print("\nEdge Features Statistics:")
#     print(f"  Mean shape: {mean_dict['edge_features'].shape}")
#     print(f"  Mean: {mean_dict['edge_features']}")
#     print(f"  Std shape: {std_dict['edge_features'].shape}")
#     print(f"  Std: {std_dict['edge_features']}")

### 1.3. Transformations Pipelines `get_transformations` 

#### 1.3.1. Standardization, Augmentation Pipeline

In [8]:
from torch_geometric.data import Data
from torch_geometric.transforms import BaseTransform, AddRandomWalkPE
from torch_geometric.utils import dropout_edge, mask_feature

class StandardizeGraphFeatures(BaseTransform):
    """
    Standardizes node and edge features using pre-computed statistics.
    Implements column-wise (feature-wise) standardization.
    """

    def __init__(self, mean_dict: Dict[str, torch.Tensor],
                 std_dict: Dict[str, torch.Tensor],
                 eps: float = 1e-8):
        """
        Args:
            mean_dict: Dict with 'node_features' and 'edge_features' means
            std_dict: Dict with 'node_features' and 'edge_features' stds
            eps: Small constant for numerical stability
        """
        self.mean_dict = mean_dict
        self.std_dict = std_dict
        self.eps = eps

    def forward(self, data: Data) -> Data:
        """Apply standardization to a single graph."""

        # Standardize node features: (23, 6) → (23, 6)
        if data.x is not None:
            node_mean = self.mean_dict['node_features']
            node_std = self.std_dict['node_features']

            # Column-wise normalization
            # Each feature f ∈ {0,1,2,3,4,5} normalized independently
            data.x = (data.x - node_mean) / (node_std + self.eps)

        # Standardize edge features: (E, 2) → (E, 2)
        if data.edge_attr is not None and 'edge_features' in self.mean_dict:
            edge_mean = self.mean_dict['edge_features']
            edge_std = self.std_dict['edge_features']

            # Column-wise normalization
            # Each feature f ∈ {0,1} normalized independently
            data.edge_attr = (data.edge_attr - edge_mean) / (edge_std + self.eps)

        return data

class DropoutEdgeAugmentation(BaseTransform):
    """
    Training augmentation that randomly removes edges from the graph.
    Uses PyTorch Geometric's dropout_edge utility.

    Reference: TRAINING_AUGMENTATION.md
    Recommendation: MODERATE PRIORITY (🟡) - Use cautiously for functional connectivity

    This augmentation:
    - Randomly drops edges during training to prevent over-reliance on specific connections
    - Forces model to learn robust patterns from incomplete graph structure
    - Simulates variability in functional connectivity strength across subjects
    - ONLY applied during training, NOT during validation/testing

    IMPORTANT: Use conservative dropout rates (0.1-0.2) for fNIRS brain connectivity

    Args:
        p: Edge dropout probability (0.0-1.0, recommended: 0.1-0.2)
        force_undirected: If True, maintains graph symmetry for undirected graphs
    """

    def __init__(self, p: float = 0.1, force_undirected: bool = True):
        self.p = p
        self.force_undirected = force_undirected

    def forward(self, data: Data) -> Data:
        """Apply edge dropout to graph data."""
        if self.p > 0:
            edge_index, edge_mask = dropout_edge(
                data.edge_index,
                p=self.p,
                force_undirected=self.force_undirected,
                training=True
            )
            data.edge_index = edge_index

            # Also filter edge attributes if they exist
            if data.edge_attr is not None:
                data.edge_attr = data.edge_attr[edge_mask]

        return data

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(p={self.p}, force_undirected={self.force_undirected})"

class MaskFeatureAugmentation(BaseTransform):
    """
    Training augmentation that randomly masks node features.
    Uses PyTorch Geometric's mask_feature utility.

    Reference: TRAINING_AUGMENTATION.md
    Recommendation: HIGH PRIORITY (🟢) - Ideal for fNIRS applications

    This augmentation:
    - Randomly masks (sets to zero) portions of node feature values
    - Prevents over-reliance on specific statistical features
    - Simulates realistic fNIRS data quality issues (motion artifacts, signal dropout)
    - Acts as feature-level dropout for robust pattern learning
    - ONLY applied during training, NOT during validation/testing

    Biologically interpretable: Models real-world sensor failures and data quality variability

    Args:
        p: Masking probability (0.0-1.0, recommended: 0.1-0.2)
        mode: Masking strategy - 'all' (individual values), 'col' (entire features), 'row' (entire nodes)
        fill_value: Value to use for masked positions (default: 0)
    """

    def __init__(self, p: float = 0.1, mode: str = 'all', fill_value: float = 0):
        self.p = p
        self.mode = mode
        self.fill_value = fill_value

    def forward(self, data: Data) -> Data:
        """Apply feature masking to graph data."""
        if self.p > 0 and data.x is not None:
            masked_x, feature_mask = mask_feature(
                data.x,
                p=self.p,
                mode=self.mode,
                fill_value=self.fill_value,
                training=True
            )
            data.x = masked_x

        return data

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(p={self.p}, mode='{self.mode}', fill_value={self.fill_value})"

class RandomWalkPEAugmentation(BaseTransform):
    """
    Node feature augmentation that adds Random Walk Positional Encoding (RWPE).
    Uses PyTorch Geometric's AddRandomWalkPE transform.

    Reference: NODE_FEATURE_AUGMENTATION.md
    Recommendation: HIGH PRIORITY (🟢) - Excellent for distinguishing brain regions

    This augmentation:
    - Adds positional encodings based on random walk return probabilities
    - Helps distinguish nodes with similar features but different connectivity roles
    - Captures local-to-medium range structural context
    - Interpretable: encodes "how information flows" through brain network
    - MUST be applied to BOTH training and inference (permanent feature augmentation)

    IMPORTANT: When using RWPE, update model input_dim = 6 + walk_length

    Args:
        walk_length: Number of random walk steps (3-5 recommended for fNIRS)
        attr_name: Attribute name for encoding (None = concatenate to existing features)
    """

    def __init__(self, walk_length: int = 4, attr_name: Optional[str] = None):
        self.walk_length = walk_length
        self.attr_name = attr_name
        # Create the PyG transform
        self.rwpe_transform = AddRandomWalkPE(
            walk_length=walk_length, attr_name=attr_name
        )

    def forward(self, data: Data) -> Data:
        """Apply random walk positional encoding to graph data."""
        return self.rwpe_transform(data)

    def __repr__(self) -> str:
        return f"{self.__class__.__name__}(walk_length={self.walk_length}, attr_name={self.attr_name})"

#### 1.3.2. Transformation Pipeline 

In [9]:
from torch_geometric.transforms import Compose

def get_transformations(
    mean: Dict[str, torch.Tensor],
    std: Dict[str, torch.Tensor],
    augment: bool = False,
    # Training augmentation parameters (TRAINING_AUGMENTATION.md)
    edge_dropout_p: float = 0.0,
    feature_mask_p: float = 0.0,
    feature_mask_mode: str = 'all',
    # Positional encoding parameters (NODE_FEATURE_AUGMENTATION.md)
    use_positional_encoding: bool = False,
    pe_walk_length: int = 4,
):
    """
    Returns a transformation pipeline for standardizing and optionally augmenting graph features.

    Based on recommendations from GRAPH_AUGMENTATION_RECOMMENDATION.md:
    - HIGH PRIORITY (🟢): Mask Feature (p=0.1-0.2, mode='all')
    - HIGH PRIORITY (🟢): Random Walk PE (walk_length=3-5)
    - MODERATE PRIORITY (🟡): Dropout Edge (p=0.1-0.2, cautious)

    Args:
        mean: Dictionary with 'node_features' and optionally 'edge_features' means.
        std: Dictionary with 'node_features' and optionally 'edge_features' stds.
        augment: Whether to apply training augmentation (True for training, False for validation).

        # Training Augmentation (only applied if augment=True)
        edge_dropout_p: Edge dropout probability (0.0-1.0, recommended: 0.1-0.2).
        feature_mask_p: Feature masking probability (0.0-1.0, recommended: 0.1-0.2).
        feature_mask_mode: Masking mode - 'all' (recommended), 'col', or 'row'.

        # Positional Encoding (applied to both training and validation)
        use_positional_encoding: Whether to add Random Walk Positional Encoding.
        pe_walk_length: Number of random walk steps (recommended: 3-5 for fNIRS).

    Returns:
        transform: Transformation pipeline with standardization and optional augmentation.

    Note:
        - Standardization is ALWAYS applied first
        - Positional Encoding (if enabled) is applied to BOTH train and val
        - Training augmentations (edge dropout, feature masking) are ONLY applied when augment=True
        - Order: Standardization → Positional Encoding → Training Augmentation
    """
    transformations = []

    # Step 1: Always apply standardization first
    transformations.append(StandardizeGraphFeatures(mean_dict=mean, std_dict=std))

    # Step 2: Optionally add positional encoding (applies to both train and val)
    if use_positional_encoding:
        transformations.append(
            RandomWalkPEAugmentation(
                walk_length=pe_walk_length,
                attr_name=None  # Concatenate to existing features
            )
        )

    # Step 3: Optionally add training augmentation (only for training data)
    if augment:
        # Edge Dropout (MODERATE PRIORITY - use cautiously)
        if edge_dropout_p > 0:
            transformations.append(
                DropoutEdgeAugmentation(
                    p=edge_dropout_p,
                    force_undirected=True  # fNIRS graphs are undirected
                )
            )

        # Feature Masking (HIGH PRIORITY - highly recommended)
        if feature_mask_p > 0:
            transformations.append(
                MaskFeatureAugmentation(
                    p=feature_mask_p,
                    mode=feature_mask_mode,
                    fill_value=0
                )
            )

    # Compose all transformations into a single pipeline
    transform = Compose(transformations)

    return transform

In [10]:
# val_transform = get_transformations(
#     mean_dict,
#     std_dict,
#     augment=False,
#     use_positional_encoding=False  # Set to True to enable RWPE
# )

# # Example 2: Training transform with recommended augmentations (Phase 1)
# # Based on GRAPH_AUGMENTATION_RECOMMENDATION.md
# train_transform = get_transformations(
#     mean_dict,
#     std_dict,
#     augment=True,
#     # HIGH PRIORITY: Feature Masking
#     feature_mask_p=0.15,  # Recommended: 0.1-0.2
#     feature_mask_mode='all',  # Recommended mode
#     # MODERATE PRIORITY: Edge Dropout (use cautiously)
#     edge_dropout_p=0.1,  # Recommended: 0.1-0.2
#     # Positional Encoding (set to True for Phase 2 experiments)
#     use_positional_encoding=False,  # Set to True to enable RWPE
#     pe_walk_length=4  # Recommended: 3-5 for fNIRS
# )

# print("=" * 70)
# print("VALIDATION TRANSFORM (No Augmentation)")
# print("=" * 70)
# print(f"Transform: {val_transform}")
# print()

# print("=" * 70)
# print("TRAINING TRANSFORM (With Augmentation)")
# print("=" * 70)
# print(f"Transform: {train_transform}")
# print()

# sample_graph = dataset[0]
# print("=" * 70)
# print("ORIGINAL GRAPH")
# print("=" * 70)
# print(f"Nodes: {sample_graph.x.size(0)}, Edges: {sample_graph.edge_index.size(1)}")
# print()

# for i in range(3):
#     train_graph = train_transform(sample_graph.clone())
#     print(
#         f"Augmentation {i + 1}: Nodes: {train_graph.x.size(0)}, Edges: {train_graph.edge_index.size(1)}"
#     )

### 1.4. DataLoaders

In [11]:
# # Test get_holdout_subject_loaders_v2
# print("=" * 60)
# print("Testing get_holdout_subject_loaders_v2")
# print("=" * 60)

# train_loader_v2, val_loader_v2 = helper_utils.get_holdout_subject_loaders_v2(
#     dataset,
#     batch_size=8,
#     shuffle_train=True,
#     num_workers=0,
#     pin_memory=False,
#     val_ratio=0.2,
#     random_state=42,
#     show_subjects=False,
#     transform=transform
# )

# print(f"\nTrain loader batches: {len(train_loader_v2)}")
# print(f"Val loader batches: {len(val_loader_v2)}")

# # Test a batch from train loader
# batch = next(iter(train_loader_v2))
# print(f"\nBatch from train_loader_v2:")
# print(f"  Number of graphs: {batch.num_graphs}")
# print(f"  Node features shape: {batch.x.shape}")
# print(f"  Edge features shape: {batch.edge_attr.shape}")

# print("=" * 60)
# print("Testing get_kfold_subject_loaders_v2")
# print("=" * 60)

# fold_loaders_v2 = helper_utils.get_kfold_subject_loaders_v2(
#     dataset,
#     n_splits=5,
#     batch_size=8,
#     shuffle_train=True,
#     num_workers=0,
#     pin_memory=False,
#     random_state=42,
#     show_subjects=True,
#     train_transform=train_transform,
#     val_transform=val_transform
# )

# print(f"\nTotal folds: {len(fold_loaders_v2)}")

# # Test first fold
# train_loader_fold1, val_loader_fold1 = fold_loaders_v2[0]
# print(f"\nFold 1:")
# print(f"  Train loader batches: {len(train_loader_fold1)}")
# print(f"  Val loader batches: {len(val_loader_fold1)}")

# # Test a batch from first fold
# batch = next(iter(train_loader_fold1))
# print(f"\n  Batch from fold 1 train_loader:")
# print(f"    Number of graphs: {batch.num_graphs}")
# print(f"    Node features shape: {batch.x.shape}")
# print(f"    Edge features shape: {batch.edge_attr.shape}")
# print(f"    Features are standardized (means near 0): {batch.x.mean(dim=0)[:3]}")


### 1.5. Compute Class Weights

In [12]:
from sklearn.utils.class_weight import compute_class_weight

def calculate_class_weights(data_source, device, fold_idx=None, use_sqrt=False):
    """
    Calculates class weights for handling imbalanced datasets.

    Supports both single DataLoader and fold loaders from K-fold cross-validation.

    Args:
        data_source (DataLoader or list):
            - If DataLoader: A single training data loader
            - If list: fold_loaders from get_kfold_subject_loaders_v2(),
              which is a list of (train_loader, val_loader) tuples
        device (torch.device):
            The device (e.g., 'cuda' or 'cpu') to place the final tensor on.
        fold_idx (int, optional):
            If data_source is fold_loaders, specify which fold to use.
            If None and data_source is fold_loaders, uses fold 0 (first fold).
            Ignored if data_source is a single DataLoader.
        use_sqrt (bool, optional):
            If True, applies square root transformation to reduce overly aggressive
            weighting of minority classes. Default is False.

    Returns:
        torch.Tensor: A 1D tensor of class weights.

    Examples:
        # For holdout training (single loader)
        weights = calculate_class_weights(train_loader, device)

        # For K-fold training (use first fold)
        fold_loaders = get_kfold_subject_loaders_v2(dataset, n_splits=5, ...)
        weights = calculate_class_weights(fold_loaders, device, fold_idx=0)

        # For K-fold training (compute from specific fold)
        weights = calculate_class_weights(fold_loaders, device, fold_idx=2)

        # With square root transformation to reduce minority class weight
        weights = calculate_class_weights(fold_loaders, device, fold_idx=0, use_sqrt=True)
    """
    # Determine the input type and extract the appropriate train_loader
    if isinstance(data_source, list):
        # data_source is fold_loaders: list of (train_loader, val_loader) tuples
        if fold_idx is None:
            fold_idx = 0  # Default to first fold
        if fold_idx >= len(data_source):
            raise ValueError(f"fold_idx={fold_idx} out of range. Only {len(data_source)} folds available.")
        train_loader, _ = data_source[fold_idx]
    else:
        # data_source is a single DataLoader
        train_loader = data_source

    # Extract all labels from the training loader
    train_labels_list = []
    for batch in train_loader:
        train_labels_list.extend(batch.y.cpu().numpy().flatten().tolist())

    train_labels_list = np.array(train_labels_list)

    # Use scikit-learn's utility to automatically calculate class weights
    class_weights = compute_class_weight(
        class_weight='balanced',
        classes=np.unique(train_labels_list),
        y=train_labels_list
    )

    # Apply square root transformation if requested
    if use_sqrt:
        # Apply square root to reduce aggressive weighting
        class_weights = np.sqrt(class_weights)

        # Re-normalize to maintain relative scale
        class_weights = class_weights / np.min(class_weights)

    # Convert the NumPy array of weights into a PyTorch tensor of type float
    class_weights_tensor = torch.tensor(class_weights, dtype=torch.float)

    return class_weights_tensor.to(device)


In [13]:
# # Verify the class weights calculation
# print("=" * 70)
# print("Verifying Class Weights Calculation")
# print("=" * 70)

# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# class_weights = calculate_class_weights(fold_loaders_v2, device, use_sqrt=True)

# # Define label mapping for our dataset
# label_map = {0: "healthy", 1: "anxiety"}

# print("\nBreakdown of weights per category:")
# print(f"{'Category':<25} {'Class ID':<12} {'Weight':<12}")
# print("-" * 50)

# for label_id, category_name in label_map.items():
#     # .item() extracts the scalar value from the tensor
#     weight = class_weights[label_id].item()
#     print(f"  {category_name:<23} {label_id:<12} {weight:.4f}")

# print("-" * 50)
# print(f"Total classes: {len(class_weights)}")
# print(f"Weights tensor: {class_weights}")
# print(f"Weights device: {class_weights.device}")
# print("=" * 70 + "\n")

### 1.6. Focal Loss for Class Imbalance

In [14]:
import torch
import torch.nn as nn
import torch.nn.functional as F

class FocalLoss(nn.Module):
    """
    Focal Loss for binary/multi-class classification to handle class imbalance.

    This loss function down-weights easy-to-classify examples and focuses training
    on hard negative examples, making it particularly effective for imbalanced datasets.

    Formula:
        FL(p_t) = -alpha_t * (1 - p_t)^gamma * log(p_t)

    where:
        - p_t: predicted probability for the true class
        - alpha: balancing factor in range [0, 1] for positive/negative classes
        - gamma: focusing parameter in range [0, inf) to down-weight easy examples

    Args:
        alpha (float): Balancing factor for positive class.
            - alpha=0.25 gives more weight to positive class (recommended for imbalanced data)
            - alpha=0.5 treats both classes equally
            - alpha=0.75 gives more weight to negative class
            Default: 0.25

        gamma (float): Focusing parameter to reduce loss for well-classified examples.
            - gamma=0: Focal Loss reduces to Cross-Entropy Loss
            - gamma=1: Mild focusing on hard examples
            - gamma=2: Standard focusing (recommended starting point)
            - gamma=5: Aggressive focusing on hard examples
            Default: 2.0

        reduction (str): Specifies the reduction to apply to the output.
            - 'none': No reduction, returns loss per sample
            - 'mean': Returns the mean of losses (recommended for training)
            - 'sum': Returns the sum of losses
            Default: 'mean'

    Shape:
        - Input: (N, C) where N is batch size and C is number of classes
        - Target: (N,) with class indices in range [0, C-1]
        - Output: scalar if reduction='mean' or 'sum', (N,) if reduction='none'

    Examples:
        >>> # Binary classification with default parameters
        >>> loss_fn = FocalLoss(alpha=0.25, gamma=2.0)
        >>> logits = torch.randn(8, 2, requires_grad=True)  # batch_size=8, n_classes=2
        >>> targets = torch.randint(0, 2, (8,))             # binary labels
        >>> loss = loss_fn(logits, targets)
        >>> loss.backward()

        >>> # Multi-class classification with custom parameters
        >>> loss_fn = FocalLoss(alpha=0.5, gamma=1.5, reduction='mean')
        >>> logits = torch.randn(16, 5)  # 5 classes
        >>> targets = torch.randint(0, 5, (16,))
        >>> loss = loss_fn(logits, targets)

        >>> # For Optuna hyperparameter search
        >>> def objective(trial):
        ...     alpha = trial.suggest_float('focal_alpha', 0.1, 0.9)
        ...     gamma = trial.suggest_float('focal_gamma', 0.0, 5.0)
        ...     loss_fn = FocalLoss(alpha=alpha, gamma=gamma)
        ...     # ... training code

    Reference:
        Lin, T. Y., et al. (2017). "Focal Loss for Dense Object Detection."
        Proceedings of the IEEE International Conference on Computer Vision.
        https://arxiv.org/abs/1708.02002
    """

    def __init__(self, alpha: float = 0.25, gamma: float = 2.0, reduction: str = 'mean'):
        super(FocalLoss, self).__init__()
        self.alpha = alpha
        self.gamma = gamma
        self.reduction = reduction

        # Validate parameters
        if not 0 <= alpha <= 1:
            raise ValueError(f"alpha must be in [0, 1], got {alpha}")
        if gamma < 0:
            raise ValueError(f"gamma must be non-negative, got {gamma}")
        if reduction not in ['none', 'mean', 'sum']:
            raise ValueError(f"reduction must be 'none', 'mean', or 'sum', got {reduction}")

    def forward(self, logits: torch.Tensor, targets: torch.Tensor) -> torch.Tensor:
        """
        Compute focal loss.

        Args:
            logits (torch.Tensor): Predicted logits with shape (N, C)
            targets (torch.Tensor): Ground truth class indices with shape (N,)

        Returns:
            torch.Tensor: Focal loss value (scalar if reduction='mean'/'sum', (N,) if 'none')
        """
        # Compute cross-entropy loss without reduction (per-sample loss)
        # F.cross_entropy computes: -log(softmax(logits)[target])
        ce_loss = F.cross_entropy(logits, targets, reduction='none')

        # Get predicted probabilities for the true class
        # Step 1: Apply softmax to logits to get probabilities
        probs = F.softmax(logits, dim=1)

        # Step 2: Gather probabilities for the true class indices
        # probs shape: (N, C), targets shape: (N,)
        # We need to gather probs[i, targets[i]] for each sample i
        p_t = probs.gather(dim=1, index=targets.unsqueeze(1)).squeeze(1)

        # Compute focal modulating factor: (1 - p_t)^gamma
        # This down-weights easy examples (high p_t) more aggressively as gamma increases
        focal_weight = (1 - p_t) ** self.gamma

        # Compute alpha weighting factor
        # For binary classification:
        # - alpha for positive class (class 1)
        # - (1 - alpha) for negative class (class 0)
        alpha_t = self.alpha * targets + (1 - self.alpha) * (1 - targets)

        # Compute final focal loss: alpha_t * focal_weight * ce_loss
        focal_loss = alpha_t * focal_weight * ce_loss

        # Apply reduction
        if self.reduction == 'none':
            return focal_loss
        elif self.reduction == 'mean':
            return focal_loss.mean()
        elif self.reduction == 'sum':
            return focal_loss.sum()

    def __repr__(self):
        return f"FocalLoss(alpha={self.alpha}, gamma={self.gamma}, reduction='{self.reduction}')"

# Quick sanity check
print("✓ FocalLoss class defined successfully")
print(f"  Default parameters: alpha=0.25, gamma=2.0, reduction='mean'")
print(f"  Compatible with binary (2 classes) and multi-class classification")
print(f"  Usage: loss_fn = FocalLoss(alpha=0.25, gamma=2.0)")

✓ FocalLoss class defined successfully
  Default parameters: alpha=0.25, gamma=2.0, reduction='mean'
  Compatible with binary (2 classes) and multi-class classification
  Usage: loss_fn = FocalLoss(alpha=0.25, gamma=2.0)


## 2. Model Definition

### 2.1 GATv2Conv

In [15]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.nn import Linear, Sequential, BatchNorm1d, LayerNorm, ReLU
from torch_geometric.nn import GATv2Conv, global_mean_pool, GINEConv
from typing import Optional, Dict, Any, Union

class GAT(nn.Module):
    def __init__(
        self,
        in_channels,
        hidden_channels,
        out_channels,
        heads=4,
        dropout=0.6,
        edge_dim=None,
        n_classes=2,
        use_residual=True  # <-- Add this argument
    ):
        super().__init__()
        self.dropout = dropout
        self.use_residual = use_residual  # <-- Store it as instance variable

        # First GAT layer
        self.conv1 = GATv2Conv(
            in_channels,
            hidden_channels,
            heads=heads,
            dropout=dropout,
            edge_dim=edge_dim,
            concat=True  # outputs hidden_channels * heads
        )

        # Residual projection for conv1 (if dimensions don't match)
        # Only create if use_residual is True
        if self.use_residual and in_channels != hidden_channels * heads:
            self.residual1 = Linear(in_channels, hidden_channels * heads)
        else:
            self.residual1 = None

        # Second GAT layer
        self.conv2 = GATv2Conv(
            hidden_channels * heads,  # input from conv1
            out_channels,
            heads=1,
            dropout=dropout,
            edge_dim=edge_dim,
            concat=False
        )

        # Residual projection for conv2 (if dimensions don't match)
        # Only create if use_residual is True
        if self.use_residual and hidden_channels * heads != out_channels:
            self.residual2 = Linear(hidden_channels * heads, out_channels)
        else:
            self.residual2 = None

        # Classification head
        self.fc1 = Linear(out_channels, n_classes)

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: Optional[torch.Tensor] = None,
        batch: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:

        if batch is None:
            batch = x.new_zeros(x.size(0), dtype=torch.long)

        # First GAT layer - conditionally apply residual
        if self.use_residual:
            identity1 = x  # Save the input for skip connection

        x = self.conv1(x, edge_index, edge_attr)

        if self.use_residual:
            # Match dimensions if needed
            if self.residual1 is not None:
                identity1 = self.residual1(identity1)
            # Add residual connection BEFORE activation
            x = x + identity1

        x = F.elu(x)  # Activation
        x = F.dropout(x, p=self.dropout, training=self.training)

        # Second GAT layer - conditionally apply residual
        if self.use_residual:
            identity2 = x  # Save the input for skip connection

        x = self.conv2(x, edge_index, edge_attr)

        if self.use_residual:
            # Match dimensions if needed
            if self.residual2 is not None:
                identity2 = self.residual2(identity2)
            # Add residual connection
            x = x + identity2

        # Global pooling
        graph_embeddings = global_mean_pool(x, batch)

        # Classification
        logits = self.fc1(graph_embeddings)

        return logits

In [16]:
# batch = next(iter(train_loader_fold1))

# model_gat = GAT(
#     in_channels=6,
#     hidden_channels=32,
#     out_channels=32,
#     heads=4,
#     dropout=0.2,
#     edge_dim=2,
#     n_classes=2,
#     use_residual=True
# )

# with torch.no_grad():
#     output_gat = model_gat(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

# total_params_gat = sum(p.numel() for p in model_gat.parameters())
# print(f"✓ GAT Model created successfully")
# print(f"  Output shape: {output_gat.shape}")
# print(f"  Total parameters: {total_params_gat:,}")

### 2.2. Flexible GATv2Conv

In [17]:
class FlexibleGATNet(nn.Module):
    """
    Flexible Graph Attention Network (GATv2) for graph classification.

    Supports both scalar and list of `n_filters` and `heads` for layer-wise control:
    1. Feature Extractor: Multiple GATv2Conv layers with attention mechanism
    2. Global Pooling: Mean pooling on final layer
    3. Classifier: Linear head for classification

    Hyperparameters:
    - n_layers: Number of GATv2Conv layers
    - n_filters: Number of filters per layer (int or list of ints)
    - heads: Number of attention heads per layer (int or list of ints)
    - fc_size: Output dimension of final GAT layer before pooling
    - dropout: Dropout probability (default: 0.6 as per GATv2 paper)
    - use_residual: Enable/disable residual connections
    - use_norm: Enable/disable normalization between layers (default: False)
    - norm_type: Type of normalization - 'batch' or 'layer' (default: 'batch')
    - n_classes: Number of output classes (default: 2 for binary classification)
    - use_gine_first_layer: Use GINEConv for first layer instead of GATv2Conv (default: False)
    - gine_train_eps: If using GINE first layer, make epsilon learnable (default: True)

    Note on GINE Hybrid Architecture:
    When use_gine_first_layer=True, the first layer uses GINEConv which:
    - Fully utilizes edge features (both correlation AND coherence)
    - Provides maximum expressiveness (WL-test equivalent)
    - Outputs n_filters[0] dimensions (NO head multiplication)
    - Uses ReLU activation and BatchNorm1d (GINE standard)
    - heads[0] parameter is ignored when using GINE first layer
    Subsequent layers remain GATv2Conv with attention mechanism.

    Note: According to GATv2 paper experiments, normalization is optional and
    not always necessary. Test with and without to find optimal configuration.

    Usage (scalar):
        model = FlexibleGATNet(
            in_channels=6,
            n_layers=3,
            n_filters=64,  # All layers have 64 filters
            heads=8,             # All layers have 8 heads
            fc_size=64,
            dropout=0.6,
            edge_dim=2,
            n_classes=2,
            use_residual=True
        )

    Usage (list):
        model = FlexibleGATNet(
            in_channels=6,
            n_layers=3,
            n_filters=[16, 32, 64],  # Each layer has different filters
            heads=[8, 8, 4],               # Each layer has different heads
            fc_size=64,
            dropout=0.6,
            edge_dim=2,
            n_classes=2,
            use_residual=True
        )

    Usage (GINE-GAT hybrid):
        model = FlexibleGATNet(
            in_channels=6,
            n_layers=3,
            n_filters=[16, 32, 64],
            heads=[None, 8, 4],  # heads[0] ignored when use_gine_first_layer=True
            fc_size=32,
            dropout=0.4,
            edge_dim=2,
            n_classes=2,
            use_residual=True,
            use_gine_first_layer=True,  # Enable GINE first layer
            gine_train_eps=True,
        )
    """

    def __init__(
        self,
        in_channels: int,
        n_layers: int = 2,
        n_filters: Union[int, list] = 64,
        heads: Union[int, list] = 8,
        fc_size: int = 64,
        dropout: float = 0.6,
        edge_dim: Optional[int] = None,
        n_classes: int = 2,
        use_residual: bool = True,
        use_norm: bool = False,
        norm_type: str = "batch",
        use_gine_first_layer: bool = False,  # Enable GINE hybrid architecture
        gine_train_eps: bool = True,  # Learnable epsilon for GINE
    ):
        super().__init__()

        self.dropout = dropout
        self.use_residual = use_residual
        self.n_classes = n_classes
        self.use_norm = use_norm
        self.norm_type = norm_type.lower() if norm_type is not None else None
        self.use_gine_first_layer = use_gine_first_layer
        self.gine_train_eps = gine_train_eps

        # Validate norm_type
        if self.use_norm and self.norm_type not in ["batch", "layer"]:
            raise ValueError(f"norm_type must be 'batch' or 'layer', got '{norm_type}'")

        # Handle n_filters: convert scalar to list if needed
        if isinstance(n_filters, int):
            self.n_filters_list = [n_filters] * n_layers
        elif isinstance(n_filters, (list, tuple)):
            if len(n_filters) != n_layers:
                raise ValueError(
                    f"Length of n_filters list ({len(n_filters)}) must match n_layers ({n_layers})"
                )
            self.n_filters_list = list(n_filters)
        else:
            raise TypeError(f"n_filters must be int or list, got {type(n_filters)}")

        # Handle heads: convert scalar to list if needed
        if isinstance(heads, int):
            self.heads_list = [heads] * n_layers
        elif isinstance(heads, (list, tuple)):
            if len(heads) != n_layers:
                raise ValueError(
                    f"Length of heads list ({len(heads)}) must match n_layers ({n_layers})"
                )
            self.heads_list = list(heads)
        else:
            raise TypeError(f"heads must be int or list, got {type(heads)}")

        # Build GATv2Conv layers, residual projections, and normalization layers
        self.convs, self.residuals, self.norms = self._build_gat_layers(
            in_channels=in_channels,
            n_filters_list=self.n_filters_list,
            heads_list=self.heads_list,
            fc_size=fc_size,
            edge_dim=edge_dim,
            use_norm=self.use_norm,
            norm_type=self.norm_type,
            use_gine_first_layer=self.use_gine_first_layer,
            gine_train_eps=self.gine_train_eps,
        )

        # Classification head
        self.fc1 = Linear(fc_size, n_classes)

    def _build_gat_layers(
        self,
        in_channels: int,
        n_filters_list: list,
        heads_list: list,
        fc_size: int,
        edge_dim: Optional[int],
        use_norm: bool,
        norm_type: str,
        use_gine_first_layer: bool,
        gine_train_eps: bool,
    ) -> tuple:
        """
        Build GATv2Conv layers, optional residual projections, and optional normalization layers using nn.ModuleList().

        Args:
            in_channels: Input node feature dimension
            n_filters_list: List of hidden dimensions for each layer
            heads_list: List of attention heads for each layer
            fc_size: Output dimension of final layer
            edge_dim: Edge feature dimension
            use_norm: Whether to use normalization
            norm_type: Type of normalization ('batch' or 'layer')

        Returns:
            (convs, residuals, norms): ModuleLists of GATv2Conv, residual Linear, and normalization layers
        """
        convs = nn.ModuleList()
        residuals = nn.ModuleList()
        norms = nn.ModuleList()

        # First layer: GINE or GAT based on use_gine_first_layer
        if use_gine_first_layer:
            # GINE first layer
            convs.append(
                GINEConv(
                    nn=self._build_gine_mlp(in_channels, n_filters_list[0]),
                    edge_dim=edge_dim,
                    train_eps=gine_train_eps,
                )
            )
            first_layer_out_dim = n_filters_list[0]  # GINE: no head multiplication

            # GINE always uses BatchNorm (best practice), ignore use_norm parameter
            norms.append(BatchNorm1d(first_layer_out_dim))
        else:
            # GAT first layer
            convs.append(
                GATv2Conv(
                    in_channels,
                    n_filters_list[0],
                    heads=heads_list[0],
                    dropout=self.dropout,
                    edge_dim=edge_dim,
                    concat=True,  # Concatenate multi-head outputs
                )
            )
            first_layer_out_dim = (
                n_filters_list[0] * heads_list[0]
            )  # GAT: head multiplication

            # Normalization layer for GAT first layer (optional)
            if use_norm:
                if norm_type == "batch":
                    norms.append(BatchNorm1d(first_layer_out_dim))
                elif norm_type == "layer":
                    norms.append(LayerNorm(first_layer_out_dim))
            else:
                norms.append(None)

        # Residual projection for first layer (common for both)
        if self.use_residual and in_channels != first_layer_out_dim:
            residuals.append(Linear(in_channels, first_layer_out_dim))
        else:
            residuals.append(None)

        # Middle layers: handle dimension based on previous layer type
        for i in range(len(n_filters_list) - 1):
            # Calculate previous layer output dimension
            if i == 0 and use_gine_first_layer:
                # First layer was GINE: no head multiplication
                prev_dim = n_filters_list[i]
            else:
                # Previous layer was GAT: head multiplication
                prev_dim = n_filters_list[i] * heads_list[i]

            curr_dim = n_filters_list[i + 1]

            convs.append(
                GATv2Conv(
                    prev_dim,
                    curr_dim,
                    heads=heads_list[i + 1],
                    dropout=self.dropout,
                    edge_dim=edge_dim,
                    concat=True,
                )
            )

            # Residual projection for middle layer
            curr_out_dim = curr_dim * heads_list[i + 1]
            if self.use_residual and prev_dim != curr_out_dim:
                residuals.append(Linear(prev_dim, curr_out_dim))
            else:
                residuals.append(None)

            # Normalization layer for middle layer
            if use_norm:
                if norm_type == "batch":
                    norms.append(BatchNorm1d(curr_out_dim))
                elif norm_type == "layer":
                    norms.append(LayerNorm(curr_out_dim))
            else:
                norms.append(None)

        # Final layer: Calculate input dimension based on previous layer type
        # Determine what the last layer's output dimension is
        if len(n_filters_list) == 1 and use_gine_first_layer:
            # Only one layer and it's GINE: no head multiplication
            final_in_dim = n_filters_list[-1]
        else:
            # Last layer is GAT (either single GAT or last of multiple layers)
            final_in_dim = n_filters_list[-1] * heads_list[-1]

        convs.append(
            GATv2Conv(
                final_in_dim,
                fc_size,
                heads=1,  # Single head for output layer (best practice)
                dropout=self.dropout,
                edge_dim=edge_dim,
                concat=False,  # Average instead of concatenate
            )
        )

        # Residual projection for final layer
        if self.use_residual and final_in_dim != fc_size:
            residuals.append(Linear(final_in_dim, fc_size))
        else:
            residuals.append(None)

        # No normalization on final layer (before global pooling)
        norms.append(None)

        return convs, residuals, norms

    def _build_gine_mlp(self, in_dim: int, out_dim: int) -> nn.Sequential:
        """
        Build MLP for GINEConv: Linear → BN → ReLU → Linear → ReLU

        This follows the GIN paper's recommendation for maximum expressiveness.

        Args:
            in_dim: Input dimension
            out_dim: Output dimension

        Returns:
            Sequential module for GINE MLP
        """
        return Sequential(
            Linear(in_dim, out_dim),
            BatchNorm1d(out_dim),
            ReLU(),
            Linear(out_dim, out_dim),
            ReLU(),
        )

    def forward(
        self,
        x: torch.Tensor,
        edge_index: torch.Tensor,
        edge_attr: Optional[torch.Tensor] = None,
        batch: Optional[torch.Tensor] = None,
    ) -> torch.Tensor:
        """
        Forward pass: GATv2Conv layers → Global pooling → Classification

        Args:
            x: Node features [N, in_channels]
            edge_index: Edge connectivity [2, E]
            edge_attr: Edge attributes [E, edge_dim] (optional)
            batch: Graph batch indices [N] (None = single graph)

        Returns:
            Logits [B, n_classes] for multi-class classification
        """
        # Handle single graph case
        if batch is None:
            batch = x.new_zeros(x.size(0), dtype=torch.long)

        # Feature extraction: Pass through conv layers with residual connections and normalization
        for i, (conv, residual, norm) in enumerate(
            zip(self.convs, self.residuals, self.norms)
        ):
            # Save identity for residual connection
            if self.use_residual:
                identity = x

            # Convolutional layer (GINE or GAT)
            x = conv(x, edge_index, edge_attr)

            # Add residual connection BEFORE normalization and activation (pre-activation pattern)
            if self.use_residual:
                if residual is not None:
                    identity = residual(identity)
                x = x + identity

            # Apply normalization AFTER residual and BEFORE activation
            if norm is not None:
                x = norm(x)

            # Activation and dropout (skip on last layer)
            if i < len(self.convs) - 1:  # Not the last layer
                # Use layer-specific activation
                if i == 0 and self.use_gine_first_layer:
                    # GINE first layer: ReLU
                    x = F.relu(x)
                else:
                    # GAT layers: ELU
                    x = F.elu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)

        # Global pooling: Mean pooling aggregates node features per graph
        graph_embeddings = global_mean_pool(x, batch)

        # Classification: Linear layer
        logits = self.fc1(graph_embeddings)

        # Output shape: (B, n_classes)
        return logits

In [18]:
# # Test FlexibleGATNet with list configuration
# model_flexible_gat_list = FlexibleGATNet(
#     in_channels=6,
#     n_layers=2,
#     n_filters=[112, 48],  # Layer-specific dimensions
#     heads=[8, 6],               # Layer-specific heads
#     fc_size=64,
#     dropout=0.4,
#     edge_dim=2,
#     n_classes=2,
#     use_residual=True,
#     use_gine_first_layer=True
# )

# with torch.no_grad():
#     output_flexible_gat_list = model_flexible_gat_list(batch.x, batch.edge_index, batch.edge_attr, batch.batch)

# total_params_flexible_gat_list = sum(p.numel() for p in model_flexible_gat_list.parameters())
# print(f"✓ FlexibleGATNet (list) Model created successfully")
# print(f"  Output shape: {output_flexible_gat_list.shape}")
# print(f"  Total parameters: {total_params_flexible_gat_list:,}")

# summary(
#     model_flexible_gat_list,
#     input_data=(batch.x, batch.edge_index, batch.edge_attr, batch.batch),
#     depth=3,
#     col_names=["input_size", "output_size", "num_params", "kernel_size", "trainable"],
#     row_settings=["var_names"]  # Shows variable names
# )

## 3. Hyperparameter Tuning with Optuna

### 3.1. Simple Training and Validation Pipeline

In [19]:
def training_epoch(model, train_loader, optimizer, loss_fn, device, epoch=None, n_epochs=None, verbose=True, log_freq=1):
    """
    Performs a single training epoch for a given model with configurable logging granularity.

    Args:
        model: The GNN model to train
        train_loader: DataLoader for the training set
        optimizer: Optimizer for updating model parameters
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        device: Device to run the training on ('cpu' or 'cuda')
        epoch: Current epoch number (for logging)
        n_epochs: Total number of epochs (for logging)
        verbose: If True, print progress (default: True)
        log_freq: Log every N batches (default: 1, meaning log every batch)
                 Examples: log_freq=5 logs every 5 batches, log_freq=10 logs every 10 batches

    Returns:
        avg_loss: Average training loss over all batches
        train_acc: Training accuracy
        train_f1: Training F1 score
    """
    model.train()
    total_loss = 0.0

    # Introspect model signature once
    sig = inspect.signature(model.forward)
    param_names = set(sig.parameters.keys())
    use_x          = ('x'          in param_names)
    use_edge_index = ('edge_index' in param_names)
    use_edge_attr  = ('edge_attr'  in param_names)
    use_batch      = ('batch'      in param_names)

    train_accuracy_metric = torchmetrics.Accuracy(task='binary').to(device)
    train_f1_metric = torchmetrics.F1Score(task='binary').to(device)

    # Determine total batches for logging
    total_batches = len(train_loader)
    epoch_str = f"{epoch}/{n_epochs}" if (epoch is not None and n_epochs is not None) else "1/1"

    for batch_idx, batch in enumerate(train_loader):
        batch = batch.to(device)

        # Forward pass - build kwargs based on model signature
        optimizer.zero_grad()
        kwargs = {}
        if use_x:          kwargs['x'] = batch.x
        if use_edge_index: kwargs['edge_index'] = batch.edge_index
        if use_edge_attr and hasattr(batch, 'edge_attr'):
            kwargs['edge_attr'] = batch.edge_attr
        if use_batch and hasattr(batch, 'batch'):
            kwargs['batch'] = batch.batch
        logits = model(**kwargs)

        y_gt = batch.y.view(-1).long()

        # Compute loss
        loss = loss_fn(logits, y_gt)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update metrics
        batch_preds = torch.argmax(logits, dim=-1)
        train_accuracy_metric.update(batch_preds, y_gt)
        train_f1_metric.update(batch_preds, y_gt)

        # Log progress at specified frequency
        current_batch = batch_idx + 1
        if verbose and (current_batch % log_freq == 0 or current_batch == total_batches):
            avg_loss = total_loss / current_batch
            current_acc = train_accuracy_metric.compute().item()
            current_f1 = train_f1_metric.compute().item()
            print(f"Epoch [{epoch_str}], Step [{current_batch}/{total_batches}], Loss: {avg_loss:.4f}, Acc: {current_acc:.4f}, F1: {current_f1:.4f}")

    avg_loss = total_loss / len(train_loader)
    train_acc = train_accuracy_metric.compute().item()
    train_f1 = train_f1_metric.compute().item()

    return avg_loss, train_acc, train_f1

def evaluate_model(model, val_loader, loss_fn, device, epoch=None, n_epochs=None, verbose=True, log_final_only=True):
    """
    Evaluates the performance of a model on a validation dataset with configurable logging.
    Returns final performance metrics including accuracy, F1 score, predictions, and targets.

    Args:
        model: The GNN model to evaluate
        val_loader: DataLoader for the validation set
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        device: Device to run the evaluation on ('cpu' or 'cuda')
        epoch: Current epoch number (for logging)
        n_epochs: Total number of epochs (for logging)
        verbose: If True, print results (default: True)
        log_final_only: If True, only log final results after all batches (default: True).
                       If False, log per-batch progress like training.

    Returns:
        avg_loss: Average validation loss over all batches
        val_acc: Validation accuracy
        val_f1: Validation F1 score for binary classification
        preds: Numpy array of all predictions
        targets: Numpy array of all ground truth labels
    """
    model.eval()
    total_loss = 0.0
    logit_list, label_list = [], []

    # Introspect model signature once
    sig = inspect.signature(model.forward)
    param_names = set(sig.parameters.keys())
    use_x          = ('x'          in param_names)
    use_edge_index = ('edge_index' in param_names)
    use_edge_attr  = ('edge_attr'  in param_names)
    use_batch      = ('batch'      in param_names)

    val_accuracy_metric = torchmetrics.Accuracy(task='binary').to(device)
    val_f1_metric = torchmetrics.F1Score(task='binary').to(device)

    # Determine total batches for logging
    total_batches = len(val_loader)
    epoch_str = f"{epoch}/{n_epochs}" if (epoch is not None and n_epochs is not None) else "1/1"

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            batch = batch.to(device)

            # Forward pass - build kwargs based on model signature
            kwargs = {}
            if use_x:          kwargs['x'] = batch.x
            if use_edge_index: kwargs['edge_index'] = batch.edge_index
            if use_edge_attr and hasattr(batch, 'edge_attr'):
                kwargs['edge_attr'] = batch.edge_attr
            if use_batch and hasattr(batch, 'batch'):
                kwargs['batch'] = batch.batch
            logits = model(**kwargs)

            y_gt = batch.y.view(-1).long()

            # Compute loss
            batch_loss = loss_fn(logits, y_gt).item()
            total_loss += batch_loss

            # Get predictions and update metrics
            batch_preds = torch.argmax(logits, dim=-1)
            val_accuracy_metric.update(batch_preds, y_gt)
            val_f1_metric.update(batch_preds, y_gt)

            logit_list.append(batch_preds.detach().cpu().numpy().reshape(-1))
            label_list.append(y_gt.detach().cpu().numpy().reshape(-1))

            # Log per-batch progress if log_final_only=False
            if verbose and not log_final_only:
                current_batch = batch_idx + 1
                current_loss = total_loss / current_batch
                current_acc = val_accuracy_metric.compute().item()
                current_f1 = val_f1_metric.compute().item()
                print(f"Epoch [{epoch_str}], Step [{current_batch}/{total_batches}], Loss: {current_loss:.4f}, Acc: {current_acc:.4f}, F1: {current_f1:.4f}")

    avg_loss = total_loss / len(val_loader)
    val_acc = val_accuracy_metric.compute().item()
    val_f1 = val_f1_metric.compute().item()

    preds = np.hstack(logit_list).ravel()
    targets = np.hstack(label_list).ravel()

    # Log final results only once (or only time if log_final_only=True)
    if verbose and log_final_only:
        print(f"Epoch [{epoch_str}], Step [{total_batches}/{total_batches}], Loss: {avg_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

    # ✅ FIX: Return aggregated arrays, not just last batch
    return avg_loss, val_acc, val_f1, preds, targets

### 3.2. Design Search Space

#### 3.2.1. Hyperparameter Search

In [20]:
def design_search_space(trial, use_fl=False):
    """
    Design the search space for hyperparameter optimization of the GNN model.
    This function uses Optuna to suggest hyperparameters for the GNN architecture.

    Args:
        trial (optuna.Trial): An Optuna trial object used to suggest hyperparameters.
        use_fl (bool): Whether to use Focal Loss. If True, suggests focal loss hyperparameters.

    Returns:
        dict: A dictionary containing the suggested hyperparameters.
    """
    # GNN Architecture Hyperparameters

    # Use trial.suggest_int to set n_layers. Name it "n_layers", and set it to be an integer between 1 and 10.
    n_layers = trial.suggest_int("n_layers", 1, 3, step=1)

    # Use trial.suggest_int to set n_filters for each layer.
    # Name each filter size as "n_filters_layer{i}" where i is the layer index
    n_filters = [
        trial.suggest_int(f"n_filters_layer{i}", 16, 128, step=16) for i in range(n_layers)
    ]

    # Use trial.suggest_int to set n_heads for each layer.
    # Name each head count as "n_heads_layer{i}" where i is the layer index
    heads = [
        trial.suggest_int(f"n_heads_layer{i}", 2, 8, step=2) for i in range(n_layers)
    ]

    # Use trial.suggest_categorical to set dropout_rate.
    dropout = trial.suggest_float("dropout", 0.1, 0.8, step=0.1)

    # Use trial.suggest_int to set fc_size, name it "fc_size",
    fc_size = trial.suggest_int("fc_size", 32, 128, step=32)

    # Use trial.suggest_categorical to set use_norm, name it "use_norm",
    use_norm = trial.suggest_categorical("use_norm", [True, False])

    # If use_norm is True, suggest norm_type
    if use_norm:
        norm_type = trial.suggest_categorical("norm_type", ["batch", "layer"])

    # Use trial.suggest_float to set learning_rate, name it "learning_rate",
    learning_rate = trial.suggest_float("learning_rate", 1e-5, 1e-1, log=True)

    # FocalLoss hyperparameters
    if use_fl:
        focal_alpha = trial.suggest_float('focal_alpha', 0.1, 0.9)
        focal_gamma = trial.suggest_float('focal_gamma', 0.0, 5.0)
    else:
        focal_alpha = None
        focal_gamma = None

    return {
        "n_layers": n_layers,
        "n_filters": n_filters,
        "heads": heads,
        "dropout": dropout,
        "fc_size": fc_size,
        "use_norm": use_norm,
        "norm_type": norm_type if use_norm else None,
        "learning_rate": learning_rate,
        "focal_alpha": focal_alpha,
        "focal_gamma": focal_gamma,
    }


In [21]:
fixed_params = {
    "n_layers": 3,
    "n_filters_layer0": 32,
    "n_filters_layer1": 32,
    "n_filters_layer2": 32,
    "n_heads_layer0": 4,
    "n_heads_layer1": 4,
    "n_heads_layer2": 4,
    "dropout": 0.1,
    "fc_size": 64,
    "use_norm": True,
    "norm_type": "batch",
    "learning_rate": 0.001,
}

toy_trial = optuna.trial.FixedTrial(fixed_params)
pprint(design_search_space(toy_trial))

{'dropout': 0.1,
 'fc_size': 64,
 'focal_alpha': None,
 'focal_gamma': None,
 'heads': [4, 4, 4],
 'learning_rate': 0.001,
 'n_filters': [32, 32, 32],
 'n_layers': 3,
 'norm_type': 'batch',
 'use_norm': True}


#### 3.2.2. Augmentation Search

In [22]:
def design_search_space_for_augmentation(trial):
    """
    Design the search space for graph augmentation hyperparameter optimization.
    This function tests combinations across Phase 1 (training augmentation),
    Phase 2 (positional encoding), and Phase 3 (combined) configurations.

    Based on recommendations from GRAPH_AUGMENTATION_RECOMMENDATION.md:
    - HIGH PRIORITY (🟢): Mask Feature (p=0.1-0.2, mode='all')
    - HIGH PRIORITY (🟢): Random Walk PE (walk_length=3-5)
    - MODERATE PRIORITY (🟡): Dropout Edge (p=0.1-0.2)

    Args:
        trial (optuna.Trial): An Optuna trial object used to suggest hyperparameters.

    Returns:
        dict: A dictionary containing the suggested augmentation hyperparameters.
    """
    # === Phase Selection ===
    # Optuna will explore which phase(s) work best

    # Use positional encoding (Phase 2/3)
    use_positional_encoding = trial.suggest_categorical(
        "use_positional_encoding", [True, False]
    )

    # If using PE, suggest walk_length
    if use_positional_encoding:
        pe_walk_length = trial.suggest_int("pe_walk_length", 3, 6, step=1)
        # Calculate input dimension for model
        input_dim = 6 + pe_walk_length
    else:
        pe_walk_length = 4  # Default (not used)
        input_dim = 6

    # === Training Augmentation (Phase 1/3) ===

    # Feature Masking (HIGH PRIORITY)
    # Test range from no masking to moderate masking
    feature_mask_p = trial.suggest_float("feature_mask_p", 0.0, 0.25, step=0.05)

    # Feature masking mode (usually 'all' is best, but test others)
    feature_mask_mode = trial.suggest_categorical(
        "feature_mask_mode", ["all", "col", "row"]
    )

    # Edge Dropout (MODERATE PRIORITY - use cautiously)
    # Test range from no dropout to conservative dropout
    edge_dropout_p = trial.suggest_float("edge_dropout_p", 0.0, 0.3, step=0.05)

    return {
        # Positional encoding
        "use_positional_encoding": use_positional_encoding,
        "pe_walk_length": pe_walk_length,
        "input_dim": input_dim,
        # Training augmentation
        "feature_mask_p": feature_mask_p,
        "feature_mask_mode": feature_mask_mode,
        "edge_dropout_p": edge_dropout_p,
    }

In [23]:
# print("=" * 80)
# print("TESTING AUGMENTATION SEARCH SPACE")
# print("=" * 80)

# # Example 1: No augmentation (baseline)
# test_params_baseline = {
#     "use_positional_encoding": False,
#     "feature_mask_p": 0.0,
#     "feature_mask_mode": "all",
#     "edge_dropout_p": 0.0,
# }

# # Example 2: Phase 1 - Training augmentation only
# test_params_phase1 = {
#     "use_positional_encoding": False,
#     "feature_mask_p": 0.15,
#     "feature_mask_mode": "all",
#     "edge_dropout_p": 0.1,
# }

# # Example 3: Phase 2 - Positional encoding only
# test_params_phase2 = {
#     "use_positional_encoding": True,
#     "pe_walk_length": 4,
#     "feature_mask_p": 0.0,
#     "feature_mask_mode": "all",
#     "edge_dropout_p": 0.0,
# }

# # Example 4: Phase 3 - Combined
# test_params_phase3 = {
#     "use_positional_encoding": True,
#     "pe_walk_length": 4,
#     "feature_mask_p": 0.15,
#     "feature_mask_mode": "all",
#     "edge_dropout_p": 0.1,
# }

# print("\nExample Configuration 1 (Baseline - No Augmentation):")
# trial_baseline = optuna.trial.FixedTrial(test_params_baseline)
# aug_params_baseline = design_search_space_for_augmentation(trial_baseline)
# pprint(aug_params_baseline)

# print("\nExample Configuration 2 (Phase 1 - Training Augmentation):")
# trial_phase1 = optuna.trial.FixedTrial(test_params_phase1)
# aug_params_phase1 = design_search_space_for_augmentation(trial_phase1)
# pprint(aug_params_phase1)

# print("\nExample Configuration 3 (Phase 2 - Positional Encoding):")
# trial_phase2 = optuna.trial.FixedTrial(test_params_phase2)
# aug_params_phase2 = design_search_space_for_augmentation(trial_phase2)
# pprint(aug_params_phase2)
# print(f"  → Model input_dim must be: {aug_params_phase2['input_dim']}")

# print("\nExample Configuration 4 (Phase 3 - Combined):")
# trial_phase3 = optuna.trial.FixedTrial(test_params_phase3)
# aug_params_phase3 = design_search_space_for_augmentation(trial_phase3)
# pprint(aug_params_phase3)
# print(f"  → Model input_dim must be: {aug_params_phase3['input_dim']}")

# print("\n" + "=" * 80)
# print("SEARCH SPACE OVERVIEW")
# print("=" * 80)
# print("\nHyperparameters being optimized:")
# print("  1. use_positional_encoding: [True, False]")
# print("  2. pe_walk_length: [3, 4, 5, 6] (if PE enabled)")
# print("  3. feature_mask_p: [0.0, 0.05, 0.10, 0.15, 0.20, 0.25]")
# print("  4. feature_mask_mode: ['all', 'col', 'row']")
# print("  5. edge_dropout_p: [0.0, 0.05, 0.10, ..., 0.30]")
# print("\nFixed model hyperparameters:")
# print("  - n_layers: 2")
# print("  - n_filters: [128, 80]")
# print("  - n_heads: [6, 6]")
# print("  - dropout: 0.8")
# print("  - fc_size: 32")
# print("  - learning_rate: 0.004989")
# print("  - use_norm: False")
# print("=" * 80)

### 3.3. Objective Function

#### 3.3.1. Hyperparameter Search

In [24]:
def objective(dataset, mean_dict, std_dict, trial, epoch, device, use_fl=False):
    """
    Multi-objective optimization: Maximize F1 and Maximize Recall.

    This refactored version:
    - Removes redundant sklearn imports
    - Uses torchmetrics for consistency with codebase
    - Calculates precision/recall only once from final evaluation
    - Leverages preds/targets returned by evaluate_model
    - Implements early stopping based on validation loss

    Args:
        dataset: The dataset to use for training and validation
        mean_dict: Dictionary containing mean values for normalization
        std_dict: Dictionary containing std values for normalization
        trial: Optuna trial object
        epoch: Number of epochs to train
        device: Device to train on (CPU or CUDA)
        use_fl: Whether to use Focal Loss. If True, uses FocalLoss with tuned hyperparameters.

    Returns: (f1_score, recall_score) - both to be maximized
    """
    # Step 1: Sample hyperparameters
    hparams = design_search_space(trial, use_fl=use_fl)
    n_layers = hparams["n_layers"]
    n_filters = hparams["n_filters"]
    heads = hparams["heads"]
    dropout = hparams["dropout"]
    fc_size = hparams["fc_size"]
    use_norm = hparams["use_norm"]
    norm_type = hparams["norm_type"] if use_norm else None
    learning_rate = hparams["learning_rate"]
    focal_alpha = hparams.get("focal_alpha", None)
    focal_gamma = hparams.get("focal_gamma", None)

    val_transform = get_transformations(
        mean=mean_dict,
        std=std_dict,
        augment=False,
    )

    train_loader_v2, val_loader_v2 = helper_utils.get_holdout_subject_loaders_v2(
        dataset,
        batch_size=8,
        shuffle_train=True,
        num_workers=0,
        pin_memory=False,
        val_ratio=0.2,
        random_state=42,
        show_subjects=False,
        transform=val_transform,
        verbose=False
    )

    # Step 2: Setup
    # class_weights = calculate_class_weights(train_loader_v2, device)

    model = FlexibleGATNet(
        in_channels=6,
        n_layers=n_layers,
        n_filters=n_filters,
        heads=heads,
        dropout=dropout,
        fc_size=fc_size,
        use_norm=use_norm,
        norm_type=norm_type,
        edge_dim=2,
        n_classes=2,
        use_gine_first_layer=True
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)

    # Choose loss function based on use_fl parameter
    if use_fl and focal_alpha is not None and focal_gamma is not None:
        # Create FocalLoss with tuned parameters
        loss_fn = FocalLoss(
            alpha=focal_alpha,
            gamma=focal_gamma,
            reduction='mean'
        )
    else:
        loss_fn = torch.nn.CrossEntropyLoss()

    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=False
    )

    # Step 3: Training with best-epoch tracking based on F1 and early stopping based on loss
    best_val_f1 = 0.0
    best_epoch = 0
    best_model_state = None

    # Early stopping parameters
    min_delta = 0.001
    patience = 10
    best_val_loss = float('inf')
    patience_counter = 0

    for epoch in range(epoch):
        # Train
        train_loss, train_acc, train_f1 = training_epoch(
            model=model,
            train_loader=train_loader_v2,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            epoch=epoch + 1,
            n_epochs=epoch,
            verbose=False,
            log_freq=3
        )

        # Validate
        val_loss, val_acc, val_f1, val_preds, val_targets = evaluate_model(
            model=model,
            val_loader=val_loader_v2,
            loss_fn=loss_fn,
            device=device,
            epoch=epoch + 1,
            n_epochs=epoch,
            verbose=False,
            log_final_only=True
        )

        scheduler.step(val_f1)

        # Early stopping check based on validation loss
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f'Early stopping at epoch {epoch + 1}')
                break

        # Report intermediate value to Optuna for pruning
        trial.report(val_f1, epoch)

        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        # Save best model based on F1 (primary metric)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = epoch + 1
            # Deep copy model state
            best_model_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}

    # Step 4: Restore best model
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    # Step 5: Final evaluation on best model
    final_val_loss, final_val_acc, final_val_f1, final_preds, final_targets = evaluate_model(
        model=model,
        val_loader=val_loader_v2,
        loss_fn=loss_fn,
        device=device,
        verbose=False,
        log_final_only=True
    )

    # Step 6: Calculate precision and recall using torchmetrics (following codebase pattern)
    # Convert numpy arrays to torch tensors
    preds_tensor = torch.from_numpy(final_preds).long().to(device)
    targets_tensor = torch.from_numpy(final_targets).long().to(device)

    # Use torchmetrics for consistency with codebase
    precision_metric = torchmetrics.Precision(task='binary').to(device)
    recall_metric = torchmetrics.Recall(task='binary').to(device)

    final_precision = precision_metric(preds_tensor, targets_tensor).item()
    final_recall = recall_metric(preds_tensor, targets_tensor).item()

    # Step 7: Log metrics
    trial.set_user_attr("val_accuracy", final_val_acc)
    trial.set_user_attr("val_loss", final_val_loss)
    trial.set_user_attr("val_f1", final_val_f1)
    trial.set_user_attr("val_precision", final_precision)
    trial.set_user_attr("val_recall", final_recall)
    trial.set_user_attr("train_f1", train_f1)
    trial.set_user_attr("train_accuracy", train_acc)
    trial.set_user_attr("total_params", total_params)
    trial.set_user_attr("trainable_params", trainable_params)
    trial.set_user_attr("best_epoch", best_epoch)

    # Step 8: Return objectives
    return final_val_f1

#### 3.3.2. Augmentation Search

In [25]:
def objective_for_augmentation(dataset, mean_dict, std_dict, trial, epoch, device):
    """
    Objective function for optimizing graph augmentation hyperparameters.

    This function uses FIXED model architecture (based on best configuration from
    previous experiments) and focuses ONLY on finding optimal augmentation settings.

    Fixed Model Configuration:
        - n_layers: 2
        - n_filters_layer0: 128
        - n_filters_layer1: 80
        - n_heads_layer0: 6
        - n_heads_layer1: 6
        - dropout: 0.8
        - fc_size: 32
        - use_norm: False
        - learning_rate: 0.004989

    Optimized Augmentation Parameters:
        - use_positional_encoding: True/False
        - pe_walk_length: 3-6 (if PE enabled)
        - feature_mask_p: 0.0-0.25
        - feature_mask_mode: 'all', 'col', 'row'
        - edge_dropout_p: 0.0-0.3

    Args:
        trial (optuna.Trial): Optuna trial object
        epoch (int): Number of training epochs
        device (torch.device): Device to run training on
        mean_dict (dict): Dictionary with node/edge feature means
        std_dict (dict): Dictionary with node/edge feature stds
        dataset: fNIRS graph dataset

    Returns:
        float: Validation F1 score (to be maximized)
    """
    # === Step 1: Sample Augmentation Hyperparameters ===
    aug_params = design_search_space_for_augmentation(trial)

    use_positional_encoding = aug_params["use_positional_encoding"]
    pe_walk_length = aug_params["pe_walk_length"]
    input_dim = aug_params["input_dim"]
    feature_mask_p = aug_params["feature_mask_p"]
    feature_mask_mode = aug_params["feature_mask_mode"]
    edge_dropout_p = aug_params["edge_dropout_p"]

    # === Step 3: Create Transforms ===

    # Training transform (with augmentation)
    train_transform = get_transformations(
        mean=mean_dict,
        std=std_dict,
        augment=True,  # Enable training augmentation
        edge_dropout_p=edge_dropout_p,
        feature_mask_p=feature_mask_p,
        feature_mask_mode=feature_mask_mode,
        use_positional_encoding=use_positional_encoding,
        pe_walk_length=pe_walk_length,
    )

    # Validation transform (NO training augmentation, but same PE)
    val_transform = get_transformations(
        mean=mean_dict,
        std=std_dict,
        augment=False,  # NO training augmentation for validation
        use_positional_encoding=use_positional_encoding,
        pe_walk_length=pe_walk_length,
    )

    # === Step 4: Create Data Loaders with Transforms ===

    train_loader_v2, val_loader_v2 = helper_utils.get_holdout_subject_loaders_v2(
        dataset,
        batch_size=8,
        shuffle_train=True,
        num_workers=0,
        pin_memory=False,
        val_ratio=0.2,
        random_state=42,
        show_subjects=False,
        train_transform=train_transform,  # Apply augmented transform to training
        val_transform=val_transform,  # Apply non-augmented transform to validation
        verbose=False,
    )

    # === Step 5: Fixed Model Architecture (from best configuration) ===

    # FIXED hyperparameters from provided configuration
    n_layers = 2
    n_filters = [112, 32]  # layer0=128, layer1=80
    heads = [6, 4]  # layer0=6, layer1=6
    dropout = 0.4
    fc_size = 96
    use_norm = True
    norm_type = 'batch'  # Not used when use_norm=False
    learning_rate = 0.036652

    ## Calculate class weights
    # class_weights = calculate_class_weights(train_loader_v2, device)

    # Create model with FIXED architecture
    # IMPORTANT: input_dim changes based on positional encoding
    model = FlexibleGATNet(
        in_channels=input_dim,  # 6 or (6 + pe_walk_length)
        n_layers=n_layers,
        n_filters=n_filters,
        heads=heads,
        dropout=dropout,
        fc_size=fc_size,
        use_norm=use_norm,
        norm_type=norm_type,
        edge_dim=2,
        n_classes=2,
        use_gine_first_layer=True,
    ).to(device)

    total_params = sum(p.numel() for p in model.parameters())
    trainable_params = sum(p.numel() for p in model.parameters() if p.requires_grad)

    # Optimizer and loss function
    optimizer = torch.optim.Adam(model.parameters(), lr=learning_rate)
    loss_fn = torch.nn.CrossEntropyLoss()
    scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode="max", factor=0.5, patience=3, verbose=False
    )

    # === Step 6: Training Loop with Early Stopping ===
    best_val_f1 = 0.0
    best_epoch = 0
    best_model_state = None

    # Early stopping parameters
    min_delta = 0.001
    patience = 10
    best_val_loss = float("inf")
    patience_counter = 0

    for e in range(epoch):
        # Train
        train_loss, train_acc, train_f1 = training_epoch(
            model=model,
            train_loader=train_loader_v2,
            optimizer=optimizer,
            loss_fn=loss_fn,
            device=device,
            epoch=e + 1,
            n_epochs=epoch,
            verbose=False,
            log_freq=3,
        )

        # Validate
        val_loss, val_acc, val_f1, val_preds, val_targets = evaluate_model(
            model=model,
            val_loader=val_loader_v2,
            loss_fn=loss_fn,
            device=device,
            epoch=e + 1,
            n_epochs=epoch,
            verbose=False,
            log_final_only=True,
        )

        scheduler.step(val_f1)

        # Early stopping check based on validation loss
        if val_loss < best_val_loss - min_delta:
            best_val_loss = val_loss
            patience_counter = 0
        else:
            patience_counter += 1
            if patience_counter >= patience:
                print(f"Early stopping at epoch {e + 1}")
                break

        # Report intermediate value to Optuna for pruning
        trial.report(val_f1, e)

        # Handle pruning based on the intermediate value
        if trial.should_prune():
            raise optuna.exceptions.TrialPruned()

        # Save best model based on F1 (primary metric)
        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            best_epoch = e + 1
            # Deep copy model state
            best_model_state = {
                k: v.cpu().clone() for k, v in model.state_dict().items()
            }

    # === Step 7: Restore Best Model ===
    if best_model_state is not None:
        model.load_state_dict({k: v.to(device) for k, v in best_model_state.items()})

    # === Step 8: Final Evaluation on Best Model ===
    final_val_loss, final_val_acc, final_val_f1, final_preds, final_targets = (
        evaluate_model(
            model=model,
            val_loader=val_loader_v2,
            loss_fn=loss_fn,
            device=device,
            verbose=False,
            log_final_only=True,
        )
    )

    # === Step 9: Calculate Precision and Recall ===
    preds_tensor = torch.from_numpy(final_preds).long().to(device)
    targets_tensor = torch.from_numpy(final_targets).long().to(device)

    precision_metric = torchmetrics.Precision(task="binary").to(device)
    recall_metric = torchmetrics.Recall(task="binary").to(device)

    final_precision = precision_metric(preds_tensor, targets_tensor).item()
    final_recall = recall_metric(preds_tensor, targets_tensor).item()

    # === Step 9: Log Metrics ===
    trial.set_user_attr("val_accuracy", final_val_acc)
    trial.set_user_attr("val_loss", final_val_loss)
    trial.set_user_attr("val_f1", final_val_f1)
    trial.set_user_attr("val_precision", final_precision)
    trial.set_user_attr("val_recall", final_recall)
    trial.set_user_attr("train_f1", train_f1)
    trial.set_user_attr("train_accuracy", train_acc)
    trial.set_user_attr("total_params", total_params)
    trial.set_user_attr("trainable_params", trainable_params)
    trial.set_user_attr("best_epoch", best_epoch)

    # Log augmentation configuration
    trial.set_user_attr("input_dim", input_dim)
    trial.set_user_attr("augmentation_config", {
        "use_positional_encoding": use_positional_encoding,
        "pe_walk_length": pe_walk_length if use_positional_encoding else None,
        "feature_mask_p": feature_mask_p,
        "feature_mask_mode": feature_mask_mode,
        "edge_dropout_p": edge_dropout_p,
    })

    # === Step 10: Return F1 Score (to be maximized) ===
    return final_val_f1

### 3.4. Run Optuna Search

#### 3.4.1. Callback Logging Setup

In [26]:
import logging
import sys
from datetime import datetime

class ProgressCallback:
    """
    Custom Optuna callback to provide periodic progress updates without flooding the notebook output.
    """
    def __init__(self, n_trials, update_interval=50, log_file=None):
        """
        Args:
            n_trials: Total number of trials
            update_interval: Print progress every N trials
            log_file: Optional file path to write progress logs
        """
        self.n_trials = n_trials
        self.update_interval = update_interval
        self.log_file = log_file
        self.start_time = datetime.now()

    def __call__(self, study, trial):
        """Called after each trial completes."""
        trial_number = trial.number + 1

        # Only print at specified intervals or on last trial
        if trial_number % self.update_interval == 0 or trial_number == self.n_trials:
            elapsed = datetime.now() - self.start_time
            elapsed_str = str(elapsed).split('.')[0]  # Remove microseconds

            # Get best value so far
            try:
                best_value = study.best_value
                best_trial_num = study.best_trial.number
            except:
                best_value = "N/A"
                best_trial_num = "N/A"

            # Get current trial info
            trial_value = trial.value if trial.value is not None else "Pruned"

            message = (f"Progress: {trial_number}/{self.n_trials} trials | "
                      f"Elapsed: {elapsed_str} | "
                      f"Current F1: {trial_value if isinstance(trial_value, str) else f'{trial_value:.4f}'} | "
                      f"Best F1: {best_value if isinstance(best_value, str) else f'{best_value:.4f}'} "
                      f"(Trial #{best_trial_num})")

            print(message)

            # Also write to log file if specified
            if self.log_file:
                with open(self.log_file, 'a') as f:
                    f.write(f"{datetime.now().strftime('%Y-%m-%d %H:%M:%S')} - {message}\n")

def setup_optuna_logging(save_dir, verbosity=logging.WARNING):
    """
    Configure Optuna logging to reduce notebook output.

    Args:
        save_dir: Directory to save log files
        verbosity: Logging level (logging.WARNING or logging.ERROR to suppress most output)

    Returns:
        log_file_path: Path to the detailed log file
    """
    # Set Optuna's verbosity (WARNING suppresses INFO and DEBUG messages)
    optuna.logging.set_verbosity(verbosity)

    # Create a detailed log file for Optuna
    log_file = os.path.join(save_dir, "optuna_detailed.log")

    # Configure Python's logging to write to file
    logger = logging.getLogger("optuna")

    # Remove existing handlers to avoid duplicate logs
    logger.handlers.clear()

    # Set logger level to capture everything
    logger.setLevel(logging.DEBUG)

    # Add file handler
    file_handler = logging.FileHandler(log_file)
    file_handler.setLevel(logging.DEBUG)  # Capture everything in the file
    formatter = logging.Formatter('%(asctime)s - %(name)s - %(levelname)s - %(message)s')
    file_handler.setFormatter(formatter)
    logger.addHandler(file_handler)

    print(f"Optuna detailed logs will be saved to: {log_file}")

    return log_file

#### 3.4.2. Hyperparameter Search

In [27]:
# # ============================================================================
# # IMPORTS - Required Libraries
# # ============================================================================
# import optuna
# from optuna.trial import TrialState

# # ============================================================================
# # CONFIGURATION - Experimental Parameters
# # ============================================================================
# # Dataset Configuration
# DATA_TYPE = 'hbo'  # Hemoglobin type: 'hbo' (oxygenated), 'hbr' (deoxygenated), or 'hbt' (total)
# MAX_TRIALS = 2  # Number of trials per subject to load from dataset
# root_dir = r'/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/data/processed_data/GNG'

# # Optimization Configuration
# N_TRIALS = 2500  # Total number of Optuna optimization trials
# N_EPOCHS = 100  # Training epochs per trial
# seed = 42

# # Device Configuration
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

# # ============================================================================
# # INITIALIZATION - Random Seed & Sampler
# # ============================================================================
# helper_utils.set_seed(seed)
# sampler = optuna.samplers.TPESampler(seed=seed)

# # ============================================================================
# # DIRECTORY SETUP - Experiment Paths & Storage
# # ============================================================================
# STUDY_NAME = f'{DATA_TYPE}_mt{MAX_TRIALS}_n_epoch_{N_EPOCHS}_n_trials_{N_TRIALS}_w_Reduce_lr_scheduler_w_Focal_Loss'
# base_dir = "/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments"
# save_dir = helper_utils.get_experiment_dir(base_dir=base_dir, experiment_name=STUDY_NAME, overwrite=False)
# storage_name = f"sqlite:///{save_dir}/optuna_study.db"

# # ============================================================================
# # DATA LOADING - Dataset & Statistics Computation
# # ============================================================================
# # Load and preprocess fNIRS graph dataset - done ONCE to avoid redundant loading
# dataset = fNIRSGraphDatasetNonRecurrent(
#     root=root_dir,
#     data_type=DATA_TYPE,
#     max_trials=MAX_TRIALS,
#     directed=True,
#     corr_threshold=0.1,
#     self_loops=True
# )

# # Compute dataset statistics for normalization - done ONCE
# mean_dict, std_dict = get_mean_std(dataset)

# # ============================================================================
# # LOGGING SETUP - Configure Output & Progress Tracking
# # ============================================================================
# print(f"Experiment Directory: {save_dir}")
# print(f"Study Name: {STUDY_NAME}")
# print(f"Running {N_TRIALS} trials with {N_EPOCHS} epochs each...\n")

# # Setup Optuna logging (redirects detailed logs to file)
# optuna_log_file = setup_optuna_logging(save_dir, verbosity=logging.WARNING)

# # Create progress callback (prints every 50 trials instead of every trial)
# progress_log_file = os.path.join(save_dir, "progress.log")
# progress_callback = ProgressCallback(
#     n_trials=N_TRIALS,
#     update_interval=50,
#     log_file=progress_log_file
# )

# print(f"Progress updates will appear every 50 trials.")
# print(f"Progress log file: {progress_log_file}")
# print(f"=" * 80)
# print()

# # ============================================================================
# # STUDY CREATION - Optuna Optimization Study
# # ============================================================================
# study = optuna.create_study(
#     study_name=STUDY_NAME,
#     storage=storage_name,
#     direction="maximize",  # Maximize F1 score
#     pruner=optuna.pruners.MedianPruner(
#         n_startup_trials=15,  # Don't prune first 15 trials
#         n_warmup_steps=20,    # Wait 20 epochs before pruning
#         interval_steps=1      # Check every epoch
#     ),
#     sampler=optuna.samplers.TPESampler(seed=42),  # Tree-structured Parzen Estimator
#     load_if_exists=True  # Resume if study exists
# )

# # ============================================================================
# # OPTIMIZATION EXECUTION - Run Hyperparameter Search
# # ============================================================================
# study.optimize(
#     lambda trial: objective(
#         dataset, mean_dict, std_dict, trial, N_EPOCHS, DEVICE, use_fl=False
#     ),
#     n_trials=N_TRIALS,
#     callbacks=[progress_callback],
#     show_progress_bar=False
# )

# # ============================================================================
# # RESULTS SUMMARY - Final Statistics & Logs
# # ============================================================================
# print()
# print(f"=" * 80)
# print("OPTIMIZATION COMPLETE")
# print(f"=" * 80)

# pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
# complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

# print("Study statistics: ")
# print("  Number of finished trials: ", len(study.trials))
# print("  Number of pruned trials: ", len(pruned_trials))
# print("  Number of complete trials: ", len(complete_trials))
# print(f"\nDetailed logs saved to: {optuna_log_file}")
# print(f"Progress logs saved to: {progress_log_file}")

#### 3.4.3. Augmentation Search

In [28]:
# import optuna
# from optuna.trial import TrialState

# # ============================================================================
# # 1. EXPERIMENTAL CONFIGURATION
# # ============================================================================
# # Define core experimental parameters for augmentation optimization
# DATA_TYPE = 'hbo'  # Hemoglobin type: 'hbo' (oxygenated), 'hbr' (deoxygenated), or 'hbt' (total)
# MAX_TRIALS = 4  # Number of trials per subject to load from dataset
# N_TRIALS = 2500  # Total number of Optuna optimization trials
# N_EPOCHS = 100  # Training epochs per trial
# DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')  # Use GPU if available
# root_dir = r'/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/data/processed_data/GNG'

# # ============================================================================
# # 2. EXPERIMENT DIRECTORY SETUP
# # ============================================================================
# # Create unique experiment name and directory structure
# STUDY_NAME = f"augmentation_optimization_{DATA_TYPE}_mt{MAX_TRIALS}_n_epoch_{N_EPOCHS}_n_trials_{N_TRIALS}_w_Reduce_lr_scheduler"
# base_dir = "/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments"
# save_dir = helper_utils.get_experiment_dir(base_dir=base_dir, experiment_name=STUDY_NAME, overwrite=False)
# storage_name = f"sqlite:///{save_dir}/optuna_study.db"

# # ============================================================================
# # 3. DATASET INITIALIZATION (ONCE - NOT PER TRIAL)
# # ============================================================================
# # Load and preprocess fNIRS graph dataset - done ONCE to avoid redundant loading
# dataset = fNIRSGraphDatasetNonRecurrent(
#     root=root_dir,
#     data_type=DATA_TYPE,
#     max_trials=MAX_TRIALS,
#     directed=True,
#     corr_threshold=0.1,
#     self_loops=True
# )

# # Compute dataset statistics for normalization - done ONCE
# mean_dict, std_dict = get_mean_std(dataset)

# # ============================================================================
# # 4. LOGGING CONFIGURATION
# # ============================================================================
# # Setup Optuna logging (redirects detailed logs to file)
# optuna_log_file = setup_optuna_logging(save_dir, verbosity=logging.WARNING)

# # Create progress callback (prints every 50 trials instead of every trial)
# progress_log_file = os.path.join(save_dir, "progress.log")
# progress_callback = ProgressCallback(
#     n_trials=N_TRIALS,
#     update_interval=50,  # Adjust this value: higher = less output
#     log_file=progress_log_file
# )

# print(f"Progress updates will appear every 50 trials.")
# print(f"Progress log file: {progress_log_file}")
# print(f"=" * 80)
# print()

# # ============================================================================
# # 5. OPTUNA STUDY CREATION
# # ============================================================================
# # Configure optimization study with pruning and sampling strategies
# study = optuna.create_study(
#     study_name=STUDY_NAME,
#     storage=storage_name,
#     direction="maximize",  # Maximize F1 score
#     pruner=optuna.pruners.MedianPruner(
#         n_startup_trials=15,  # Don't prune first 15 trials
#         n_warmup_steps=20,    # Wait 20 epochs before pruning
#         interval_steps=1      # Check every epoch
#     ),
#     sampler=optuna.samplers.TPESampler(seed=42),  # Tree-structured Parzen Estimator
#     load_if_exists=True  # Resume if study exists
# )

# # ============================================================================
# # 6. RUN HYPERPARAMETER OPTIMIZATION
# # ============================================================================
# # Execute optimization trials using pre-loaded dataset and statistics
# study.optimize(
#     lambda trial: objective_for_augmentation(
#         dataset, mean_dict, std_dict, trial, N_EPOCHS, DEVICE
#     ),
#     n_trials=N_TRIALS,
#     callbacks=[progress_callback],
#     show_progress_bar=False
# )

# # ============================================================================
# # 7. REPORT FINAL STATISTICS
# # ============================================================================
# print()
# print(f"=" * 80)
# print("OPTIMIZATION COMPLETE")
# print(f"=" * 80)

# # Collect trial state statistics
# pruned_trials = study.get_trials(deepcopy=False, states=[TrialState.PRUNED])
# complete_trials = study.get_trials(deepcopy=False, states=[TrialState.COMPLETE])

# # Display summary
# print("Study statistics: ")
# print("  Number of finished trials: ", len(study.trials))
# print("  Number of pruned trials: ", len(pruned_trials))
# print("  Number of complete trials: ", len(complete_trials))
# print(f"\nDetailed logs saved to: {optuna_log_file}")
# print(f"Progress logs saved to: {progress_log_file}")

### 3.5. Analyze Optuna Study

#### 3.5.1. `Top-k` Trials

In [29]:
from prettytable import PrettyTable
from datetime import datetime
import os

def get_best_k_trials(df_trials, by, k, study_name, mode="hyperparameter", ascending=False, show_best_trial=True, save_study=False, study_obj=None, trial_id=None):
    """
    Get the best k trials from the trials dataframe based on a specified column.
    Always displays results in a formatted table and optionally saves the study to disk.

    Compatible with both single-objective and multi-objective Optuna studies.
    Supports reporting for both hyperparameter search and augmentation search.

    Args:
        df_trials (pd.DataFrame): The trials dataframe from study.trials_dataframe()
        by (str): The column name to sort by (e.g., 'values_0' for F1, 'values_1' for Recall)
        k (int): Number of top trials to return
        study_name (str): Name of the study folder for saving (default: "optuna_study")
        mode (str): Type of search to report on. Options:
            - "hyperparameter": Hyperparameter optimization search (from design_search_space)
            - "augmentation": Graph augmentation search (from design_search_space_for_augmentation)
            Default: "hyperparameter"
        ascending (bool): If True, sort in ascending order (default: False for descending)
        show_best_trial (bool): If True, display best trial summary (default: True)
        save_study (bool): If True, save study to base_dir/YYYYMMDD/study_id (default: False)
        study_obj: Optuna study object for displaying best trial (required if show_best_trial=True)
        trial_id (int, optional): If provided, display a summary of the specific trial after the Best Trial Summary

    Returns:
        pd.DataFrame: The top k trials sorted by the specified column

    Raises:
        ValueError: If mode is not "hyperparameter" or "augmentation"
    """
    # Validate mode parameter
    if mode not in ["hyperparameter", "augmentation"]:
        raise ValueError(f"mode must be 'hyperparameter' or 'augmentation', got '{mode}'")

    df_sorted = df_trials.sort_values(by=by, ascending=ascending)
    best_k = df_sorted.head(k)

    # Define display columns based on mode
    if mode == "hyperparameter":
        # Hyperparameter search columns from design_search_space
        display_cols = ['number', 'value', 'values_0', 'values_1', 'params_n_layers',
                       'params_n_filters_layer0', 'params_n_heads_layer0',
                       'params_dropout', 'params_fc_size', 'params_use_norm',
                       'params_norm_type', 'params_learning_rate',
                       'user_attrs_val_accuracy', 'user_attrs_val_loss',
                       'user_attrs_best_epoch', 'user_attrs_total_params']
    elif mode == "augmentation":
        # Augmentation search columns from design_search_space_for_augmentation
        display_cols = ['number', 'value', 'values_0', 'values_1',
                       'params_use_positional_encoding', 'params_pe_walk_length',
                       'params_input_dim', 'params_feature_mask_p',
                       'params_feature_mask_mode', 'params_edge_dropout_p',
                       'user_attrs_val_accuracy', 'user_attrs_val_loss',
                       'user_attrs_best_epoch', 'user_attrs_total_params']

    # Filter to only columns that exist in the dataframe
    available_cols = [col for col in display_cols if col in best_k.columns]

    # Create PrettyTable
    table = PrettyTable()
    table.field_names = available_cols

    # Add rows
    for idx, row in best_k.iterrows():
        values = [row[col] for col in available_cols]
        # Format numeric values
        formatted_values = []
        for val in values:
            if isinstance(val, float):
                formatted_values.append(f"{val:.4f}")
            else:
                formatted_values.append(str(val))
        table.add_row(formatted_values)

    # Display mode in header
    mode_display = "Hyperparameter Search" if mode == "hyperparameter" else "Graph Augmentation Search"
    print(f"\n{'='*80}")
    print(f"Top {k} Trials - {mode_display}")
    print(f"{'='*80}")
    print(table)

    if show_best_trial and study_obj is not None:
        # Check if the study is multi-objective or single-objective
        is_multi_objective = len(study_obj.directions) > 1

        if is_multi_objective:
            # For multi-objective, use best_trials (Pareto front)
            best_trials_list = study_obj.best_trials
            if len(best_trials_list) > 0:
                # Display the first best trial from the Pareto front
                best_trial = best_trials_list[0]
                print(f"\n{'='*70}")
                print(f"Best Trial Summary (First from Pareto Front) - {mode_display}")
                print(f"{'='*70}")
                print(f"  Trial Number: {best_trial.number}")

                # Multi-objective has multiple values
                for idx, val in enumerate(best_trial.values or []):
                    print(f"  Objective {idx}: {val:.4f}")

                print(f"\n  Hyperparameters ({mode}):")
                for key, value in best_trial.params.items():
                    print(f"    {key}: {value}")
                print(f"{'='*70}")
        else:
            # For single-objective, use best_trial
            best_trial = study_obj.best_trial
            print(f"\n{'='*70}")
            print(f"Best Trial Summary (Single-Objective) - {mode_display}")
            print(f"{'='*70}")
            print(f"  Trial Number: {best_trial.number}")

            # Single-objective has a single value
            if best_trial.value is not None:
                print(f"  Best Value: {best_trial.value:.4f}")

            print(f"\n  Hyperparameters ({mode}):")
            for key, value in best_trial.params.items():
                print(f"    {key}: {value}")
            print(f"{'='*70}")

    # Display Trial ID Summary if trial_id is provided
    if trial_id is not None:
        trial_row = df_trials[df_trials['number'] == trial_id]

        if not trial_row.empty:
            trial_row = trial_row.iloc[0]

            # Display Trial ID Summary
            print(f"\n{'='*70}")
            print(f"Trial ID: {trial_id} Summary ({mode_display})")
            print(f"{'='*70}")
            print(f"  Trial Number: {trial_row['number']}")

            # Display value(s)
            if 'value' in trial_row.index and trial_row['value'] is not None:
                print(f"  Value: {trial_row['value']:.4f}")

            # Display multi-objective values if they exist
            for i in range(10):
                if f'values_{i}' in trial_row.index and trial_row[f'values_{i}'] is not None:
                    print(f"  Objective {i}: {trial_row[f'values_{i}']:.4f}")

            # Display hyperparameters
            print(f"\n  Hyperparameters ({mode}):")
            for col in trial_row.index:
                if col.startswith('params_'):
                    param_name = col.replace('params_', '')
                    print(f"    {param_name}: {trial_row[col]}")
            print(f"{'='*70}")

    # Save study if requested
    if save_study and study_obj is not None:
        # Create date-based folder structure using local get_experiment_dir
        base_dir = "/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments"
        save_dir = helper_utils.get_experiment_dir(base_dir=base_dir, experiment_name=study_name, overwrite=False)

        # Save trials dataframe to CSV
        df_file = os.path.join(save_dir, f"trials_dataframe.csv")
        df_trials.to_csv(df_file, index=False)

        print(f"\nTrials dataframe saved to: {df_file}")

    return best_k

#### 3.5.2. Visualization

In [30]:
def save_optuna_visualizations(study, study_name, params_to_plot=None,
                                objective_names=None,
                                base_dir="/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments"):
    """
    Generate and save all Optuna visualization plots for single-objective or multi-objective studies.

    Automatically detects if the study is single-objective or multi-objective and generates
    appropriate visualizations. Uses current Optuna API with proper parameter handling.

    Args:
        study: Optuna study object
        study_name (str): Name of the study (e.g., "n_epoch_50_n_trials_100")
        params_to_plot (list, optional): List of parameter names to include in parallel coordinate plot
                                         If None, uses default parameters based on search space
        objective_names (list, optional): List of objective names for display.
                                         - For single-objective: provide one name (e.g., ['F1 Score'])
                                         - For multi-objective: provide names for each objective (e.g., ['F1 Score', 'Recall'])
                                         If None, uses default names based on number of objectives
        base_dir (str): Base directory for experiments

    Returns:
        dict: Dictionary with figure names as keys and saved paths as values

    Examples:
        # Single-objective study (hyperparameter search)
        saved_paths = save_optuna_visualizations(
            study,
            "hyperparameter_study",
            params_to_plot=['n_layers', 'dropout', 'learning_rate'],
            objective_names=['Validation F1 Score']
        )

        # Single-objective study (augmentation search)
        saved_paths = save_optuna_visualizations(
            study,
            "augmentation_study",
            params_to_plot=['use_positional_encoding', 'feature_mask_p', 'edge_dropout_p'],
            objective_names=['Validation F1 Score']
        )
    """
    # Create date-based folder structure
    date_str = datetime.now().strftime("%Y%m%d")
    save_dir = os.path.join(base_dir, date_str, study_name)
    os.makedirs(save_dir, exist_ok=True)

    # Default parameters if not specified
    # Infer from study trials if available
    if params_to_plot is None:
        # Try to extract parameters from the first trial
        try:
            first_trial = study.best_trial if study.best_trial else study.trials[0]
            all_params = list(first_trial.params.keys())
            # Filter to commonly visualized parameters (prioritize first 5-6 parameters)
            params_to_plot = all_params[:6] if len(all_params) > 6 else all_params
        except (IndexError, AttributeError):
            # Fallback to empty list if no trials exist
            params_to_plot = []

    # Detect number of objectives
    n_objectives = len(study.directions)
    is_multi_objective = n_objectives > 1

    # Set default objective names if not provided
    if objective_names is None:
        if is_multi_objective:
            objective_names = [f"Objective {i}" for i in range(n_objectives)]
        else:
            objective_names = ["Objective Value"]

    # Validate objective_names
    if len(objective_names) != n_objectives:
        raise ValueError(
            f"Number of objective_names ({len(objective_names)}) must match "
            f"number of study objectives ({n_objectives})"
        )

    # Dictionary to store saved paths
    saved_paths = {}

    print("=" * 80)
    print(f"Generating and Saving Optuna Visualizations")
    print(f"Study Type: {'Multi-Objective' if is_multi_objective else 'Single-Objective'}")
    print(f"Number of Objectives: {n_objectives}")
    print(f"Parameters to Plot: {params_to_plot}")
    print("=" * 80)

    if is_multi_objective:
        # =====================================================================
        # MULTI-OBJECTIVE OPTIMIZATION
        # =====================================================================

        for obj_idx, obj_name in enumerate(objective_names):
            print(f"\n--- Processing Objective {obj_idx}: {obj_name} ---")

            # Create a target function that extracts the specific objective value
            # This is the correct way to specify a target for multi-objective studies
            def make_target_func(idx):
                def target_func(trial):
                    if trial.values is None or len(trial.values) <= idx:
                        return float('nan')
                    return trial.values[idx]
                return target_func

            target_func = make_target_func(obj_idx)

            # 1. Optimization History
            try:
                fig = optuna.visualization.matplotlib.plot_optimization_history(
                    study,
                    target=target_func,
                    target_name=obj_name,
                )
                plt.title(f'Optimization History - {obj_name}')
                plt.legend(loc='best')
                plt.tight_layout()

                fig_path = os.path.join(save_dir, f"optimization_history_obj{obj_idx}.png")
                plt.savefig(fig_path, dpi=300, bbox_inches='tight')
                saved_paths[f'optimization_history_obj{obj_idx}'] = fig_path
                print(f"✓ Saved: optimization_history_obj{obj_idx}.png")
                plt.close()
            except Exception as e:
                print(f"✗ Failed to save optimization_history_obj{obj_idx}: {e}")

            # 2. Parameter Importances
            try:
                fig = optuna.visualization.matplotlib.plot_param_importances(
                    study,
                    target=target_func,
                    target_name=obj_name
                )
                plt.legend(loc='best')
                plt.tight_layout()

                fig_path = os.path.join(save_dir, f"param_importances_obj{obj_idx}.png")
                plt.savefig(fig_path, dpi=300, bbox_inches='tight')
                saved_paths[f'param_importances_obj{obj_idx}'] = fig_path
                print(f"✓ Saved: param_importances_obj{obj_idx}.png")
                plt.close()
            except Exception as e:
                print(f"✗ Failed to save param_importances_obj{obj_idx}: {e}")

            # 3. Parallel Coordinate
            try:
                ax = optuna.visualization.matplotlib.plot_parallel_coordinate(
                    study,
                    params=params_to_plot if params_to_plot else None,
                    target=target_func,
                    target_name=obj_name
                )
                fig = plt.gcf()  # Get current figure instead of ax.figure
                fig.set_size_inches(12, 6, forward=True)
                plt.title(f'Parallel Coordinate - {obj_name}', pad=20)
                fig.tight_layout()

                fig_path = os.path.join(save_dir, f"parallel_coordinate_obj{obj_idx}.png")
                plt.savefig(fig_path, dpi=300, bbox_inches='tight')
                saved_paths[f'parallel_coordinate_obj{obj_idx}'] = fig_path
                print(f"✓ Saved: parallel_coordinate_obj{obj_idx}.png")
                plt.close()
            except Exception as e:
                print(f"✗ Failed to save parallel_coordinate_obj{obj_idx}: {e}")

        # 4. Pareto Front (only for multi-objective)
        try:
            print(f"\n--- Generating Pareto Front ---")
            optuna.visualization.matplotlib.plot_pareto_front(
                study,
                target_names=objective_names
            )
            pareto_title = ' vs '.join(objective_names)
            plt.title(f'Pareto Front - {pareto_title}')
            plt.legend(loc='best')
            plt.tight_layout()

            fig_path = os.path.join(save_dir, "pareto_front.png")
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            saved_paths['pareto_front'] = fig_path
            print(f"✓ Saved: pareto_front.png")
            plt.close()
        except Exception as e:
            print(f"✗ Failed to save pareto_front: {e}")

    else:
        # =====================================================================
        # SINGLE-OBJECTIVE OPTIMIZATION
        # =====================================================================

        obj_name = objective_names[0]
        print(f"\n--- Processing Single Objective: {obj_name} ---")

        # 1. Optimization History (target=None for single-objective)
        try:
            fig = optuna.visualization.matplotlib.plot_optimization_history(
                study,
                target=None,  # None for single-objective automatically uses objective values
                target_name=obj_name,
            )
            plt.title(f'Optimization History - {obj_name}')
            plt.legend(loc='best')
            plt.tight_layout()

            fig_path = os.path.join(save_dir, "optimization_history.png")
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            saved_paths['optimization_history'] = fig_path
            print(f"✓ Saved: optimization_history.png")
            plt.close()
        except Exception as e:
            print(f"✗ Failed to save optimization_history: {e}")

        # 2. Parameter Importances
        try:
            fig = optuna.visualization.matplotlib.plot_param_importances(
                study,
                target=None,  # None for single-objective
                target_name=obj_name
            )
            plt.legend(loc='best')
            plt.tight_layout()

            fig_path = os.path.join(save_dir, "param_importances.png")
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            saved_paths['param_importances'] = fig_path
            print(f"✓ Saved: param_importances.png")
            plt.close()
        except Exception as e:
            print(f"✗ Failed to save param_importances: {e}")

        # 3. Parallel Coordinate
        try:
            ax = optuna.visualization.matplotlib.plot_parallel_coordinate(
                study,
                params=params_to_plot if params_to_plot else None,
                target=None,  # None for single-objective
                target_name=obj_name
            )
            fig = plt.gcf()  # Get current figure instead of ax.figure
            fig.set_size_inches(12, 6, forward=True)
            plt.title(f'Parallel Coordinate - {obj_name}', pad=20)
            fig.tight_layout()

            fig_path = os.path.join(save_dir, "parallel_coordinate.png")
            plt.savefig(fig_path, dpi=300, bbox_inches='tight')
            saved_paths['parallel_coordinate'] = fig_path
            print(f"✓ Saved: parallel_coordinate.png")
            plt.close()
        except Exception as e:
            print(f"✗ Failed to save parallel_coordinate: {e}")

        print(f"\nNote: Pareto front is not applicable for single-objective optimization.")

    print("=" * 80)
    print(f"All visualizations saved to: {save_dir}")
    print(f"Total figures saved: {len(saved_paths)}")
    print("=" * 80)

    return saved_paths

# # Example usage with single-objective study (augmentation search)
# try:
#     save_optuna_visualizations(
#         study,
#         STUDY_NAME,
#         objective_names=['Validation F1 Score']
#     )
# except NameError:
#     print("Study object does not exist. Skipping visualization generation.")

#### 3.5.3. Loading Optuna Study

In [31]:
# import optuna
# import pandas as pd

# # Define study details
# STUDY_NAME = 'hbo_mt2_n_epoch_100_n_trials_2500_w_Reduce_lr_scheduler'
# storage_path = f"/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments/20251215/no_class_weighting/{STUDY_NAME}/optuna_study.db"
# storage_name = f"sqlite:///{storage_path}"

# # Load the study from SQLite database
# study = optuna.load_study(
#     study_name=STUDY_NAME,
#     storage=storage_name
# )

# print(f"Study loaded successfully!")
# print(f"Number of trials: {len(study.trials)}")
# print(f"Study directions: {study.directions}")

# best_k_trials_f1 = get_best_k_trials(
#     study.trials_dataframe(),
#     by='value',
#     k=10,
#     mode='hyperparameter',
#     study_name=STUDY_NAME,
#     ascending=False,
#     show_best_trial=True,
#     study_obj=study,
#     trial_id=2105
# )

## 4. Training and Validation Pipeline

### 4.1. Entire Training and Validation Pipeline

In [32]:
import os
import torch
import torch.nn as nn
import numpy as np
import pickle
import shutil
from datetime import datetime
from sklearn import metrics
from sklearn.metrics import confusion_matrix
import torchmetrics

import numpy as np
import torch

# ----------------------------------------------------------------------------
# 1. Utility Functions
# ----------------------------------------------------------------------------

def get_experiment_dir(experiment_name: str, base_dir: str = "experiments", overwrite: bool = False) -> str:
    date_str = datetime.now().strftime("%Y%m%d")
    exp_dir = os.path.join(base_dir, date_str, experiment_name)
    if os.path.exists(exp_dir) and overwrite:
        print(f"Removing existing directory: {exp_dir}")
        shutil.rmtree(exp_dir)
    os.makedirs(exp_dir, exist_ok=True)
    return exp_dir

def save_metrics(metrics_dict, save_dir, filename):
    os.makedirs(save_dir, exist_ok=True)
    file_path = os.path.join(save_dir, f"{filename}.pkl")
    with open(file_path, "wb") as f:
        pickle.dump(metrics_dict, f)

def save_best_model(model, save_dir, model_name):
    os.makedirs(save_dir, exist_ok=True)
    save_path = os.path.join(save_dir, f"{model_name}_best.pt")
    torch.save(model.state_dict(), save_path)
    print(f"• Saved best model weights to {save_path}")
    return save_path

def visualize_training_results(results, save_dir, experiment_name, training_type="holdout", best_epoch=None):
    """
    Visualizes training results and saves figures to the experiment directory.

    Compatible with both holdout and k-fold training results, including early stopping.

    Args:
        results: Results from perform_holdout_training or perform_kfold_training
                 - Holdout (new): tuple of (holdout_metrics, final_summary)
                 - Holdout (old): dict with "history" and "final" keys
                 - K-fold: dict with "overall" and "folds" keys
        save_dir (str): Directory to save the figures
        experiment_name (str): Name of the experiment (for file naming)
        training_type (str): Either "holdout" or "kfold" (default: "holdout")
        best_epoch (int, optional): Best epoch index (0-indexed) to mark on plots.
                                   If None, will try to extract from results.

    Returns:
        dict: Paths to all saved figures
            {
                "loss_plot": str,
                "accuracy_plot": str,
                "f1_plot": str,
                "confusion_matrix_plot": str
            }

    Examples:
        # Holdout training (new structure)
        holdout_metrics, final_summary = perform_holdout_training(...)
        exp_dir = get_experiment_dir("my_experiment")
        paths = visualize_training_results(
            (holdout_metrics, final_summary),
            exp_dir,
            "my_experiment",
            "holdout"
        )

        # K-fold training
        results = perform_kfold_training(...)
        exp_dir = get_experiment_dir("my_kfold")
        paths = visualize_training_results(
            results,
            exp_dir,
            "my_kfold",
            "kfold"
        )
    """
    import matplotlib.pyplot as plt
    import numpy as np
    import seaborn as sns

    # Ensure save directory exists
    os.makedirs(save_dir, exist_ok=True)

    # Extract metrics based on training type and result structure
    if training_type == "holdout":
        # Check if results is a tuple (new structure) or dict (old structure)
        if isinstance(results, tuple):
            # New structure from updated perform_holdout_training
            holdout_metrics, final_summary = results
            metrics = holdout_metrics
            final_cm = np.array(final_summary["confusion_matrix"])
            # Extract best_epoch if not provided
            if best_epoch is None:
                best_epoch = holdout_metrics.get("best_epoch", None)
        else:
            # Old structure (backward compatibility)
            metrics = results.get("history", results)
            final_cm = np.array(results["final"]["confusion_matrix"])

    elif training_type == "kfold":
        # K-fold structure remains compatible
        metrics = results["overall"]
        final_cm = np.array(results["overall"]["confusion_matrix_overall"])
        # For k-fold aggregate plot, best_epoch doesn't apply directly
        # (each fold has its own best epoch)
        best_epoch = None
    else:
        raise ValueError(f"Invalid training_type: {training_type}. Must be 'holdout' or 'kfold'")

    # Extract metric arrays
    train_loss = np.array(metrics["train_loss"])
    val_loss = np.array(metrics["val_loss"])
    train_acc = np.array(metrics["train_accuracy"])
    val_acc = np.array(metrics["val_accuracy"])
    train_f1 = np.array(metrics["train_f1"])
    val_f1 = np.array(metrics["val_f1"])

    # Handle NaN values in k-fold (from variable-length folds due to early stopping)
    if training_type == "kfold":
        # Find first occurrence of NaN to truncate arrays
        valid_mask = ~np.isnan(train_loss)
        if not np.all(valid_mask):
            max_valid_idx = np.where(valid_mask)[0][-1] + 1
            train_loss = train_loss[:max_valid_idx]
            val_loss = val_loss[:max_valid_idx]
            train_acc = train_acc[:max_valid_idx]
            val_acc = val_acc[:max_valid_idx]
            train_f1 = train_f1[:max_valid_idx]
            val_f1 = val_f1[:max_valid_idx]

    # Create epoch array (1-indexed for display)
    epochs = np.arange(1, len(train_loss) + 1)

    # Get early stopping info for titles
    stopped_at = metrics.get("stopped_at_epoch", len(train_loss) - 1)
    early_stop_info = ""
    if best_epoch is not None:
        early_stop_info = f"\n(Best: Epoch {best_epoch+1}, Stopped: Epoch {stopped_at+1})"

    # Dictionary to store saved paths
    saved_paths = {}

    # -------------------------------------------------------------------------
    # Figure 1: Training vs Validation Loss
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, train_loss, 'b-', label='Training Loss', linewidth=2)
    ax.plot(epochs, val_loss, 'r-', label='Validation Loss', linewidth=2)

    # Add best epoch marker
    if best_epoch is not None and best_epoch < len(val_loss):
        ax.axvline(x=best_epoch+1, color='green', linestyle='--',
                   linewidth=2, alpha=0.7, label=f'Best Epoch ({best_epoch+1})')
        ax.plot(best_epoch+1, val_loss[best_epoch],
               'g*', markersize=15, label='Best Val Loss', zorder=5)

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Loss', fontsize=12)
    ax.set_title(f'Training vs Validation Loss - {experiment_name}{early_stop_info}',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)

    loss_path = os.path.join(save_dir, f"{experiment_name}_loss_curves.png")
    plt.savefig(loss_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_paths["loss_plot"] = loss_path

    # -------------------------------------------------------------------------
    # Figure 2: Training vs Validation Accuracy
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, train_acc, 'b-', label='Training Accuracy', linewidth=2)
    ax.plot(epochs, val_acc, 'r-', label='Validation Accuracy', linewidth=2)

    # Add best epoch marker
    if best_epoch is not None and best_epoch < len(val_acc):
        ax.axvline(x=best_epoch+1, color='green', linestyle='--',
                   linewidth=2, alpha=0.7, label=f'Best Epoch ({best_epoch+1})')
        ax.plot(best_epoch+1, val_acc[best_epoch],
               'g*', markersize=15, label='Best Val Acc', zorder=5)

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('Accuracy', fontsize=12)
    ax.set_title(f'Training vs Validation Accuracy - {experiment_name}{early_stop_info}',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    acc_path = os.path.join(save_dir, f"{experiment_name}_accuracy_curves.png")
    plt.savefig(acc_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_paths["accuracy_plot"] = acc_path

    # -------------------------------------------------------------------------
    # Figure 3: Training vs Validation F1-Score
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(10, 6))
    ax.plot(epochs, train_f1, 'b-', label='Training F1-Score', linewidth=2)
    ax.plot(epochs, val_f1, 'r-', label='Validation F1-Score', linewidth=2)

    # Add best epoch marker
    if best_epoch is not None and best_epoch < len(val_f1):
        ax.axvline(x=best_epoch+1, color='green', linestyle='--',
                   linewidth=2, alpha=0.7, label=f'Best Epoch ({best_epoch+1})')
        ax.plot(best_epoch+1, val_f1[best_epoch],
               'g*', markersize=15, label='Best Val F1', zorder=5)

    ax.set_xlabel('Epoch', fontsize=12)
    ax.set_ylabel('F1-Score', fontsize=12)
    ax.set_title(f'Training vs Validation F1-Score - {experiment_name}{early_stop_info}',
                 fontsize=14, fontweight='bold')
    ax.legend(fontsize=11)
    ax.grid(True, alpha=0.3)
    ax.set_ylim([0, 1])

    f1_path = os.path.join(save_dir, f"{experiment_name}_f1_curves.png")
    plt.savefig(f1_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_paths["f1_plot"] = f1_path

    # -------------------------------------------------------------------------
    # Figure 4: Confusion Matrix
    # -------------------------------------------------------------------------
    fig, ax = plt.subplots(figsize=(8, 6))

    # Create heatmap
    sns.heatmap(final_cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=['Healthy (0)', 'Anxiety (1)'],
                yticklabels=['Healthy (0)', 'Anxiety (1)'],
                cbar_kws={'label': 'Count'},
                ax=ax)

    cm_title = f'Final Confusion Matrix - {experiment_name}'
    if best_epoch is not None:
        cm_title += f'\n(From Best Epoch: {best_epoch+1})'
    ax.set_xlabel('Predicted Label', fontsize=12)
    ax.set_ylabel('True Label', fontsize=12)
    ax.set_title(cm_title, fontsize=14, fontweight='bold')

    cm_path = os.path.join(save_dir, f"{experiment_name}_confusion_matrix.png")
    plt.savefig(cm_path, dpi=300, bbox_inches='tight')
    plt.close(fig)
    saved_paths["confusion_matrix_plot"] = cm_path

    print(f"\nVisualization saved to: {save_dir}")
    print(f"  - Loss curves: {os.path.basename(loss_path)}")
    print(f"  - Accuracy curves: {os.path.basename(acc_path)}")
    print(f"  - F1-Score curves: {os.path.basename(f1_path)}")
    print(f"  - Confusion Matrix: {os.path.basename(cm_path)}")

    if best_epoch is not None:
        print(f"\nBest model from epoch {best_epoch+1} (marked with green line and star)")

    return saved_paths

# ----------------------------------------------------------------------------
# 2. Training / Validation Loops
# ----------------------------------------------------------------------------
# ---------- train / val ----------
def train(model, train_loader, optimizer, loss_fn, device,
          epoch=None, n_epochs=None, verbose=True, log_freq=10):
    """
    Performs one full training epoch over the train_loader.

    Args:
        model: The GNN model to train
        train_loader: DataLoader for the training set
        optimizer: Optimizer for updating model parameters
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        device: Device to run training on ('cpu' or 'cuda')
        epoch: Current epoch number (for logging, optional)
        n_epochs: Total number of epochs (for logging, optional)
        verbose: If True, print progress (default: True)
        log_freq: Log every N batches (default: 10)

    Returns:
        tuple: (avg_loss, train_acc, train_f1)
            - avg_loss: Average training loss over all batches
            - train_acc: Training accuracy (computed using torchmetrics)
            - train_f1: Training F1 score (computed using torchmetrics)
    """
    model.train()
    total_loss = 0.0

    # Introspect model signature once
    sig = inspect.signature(model.forward)
    param_names = set(sig.parameters.keys())
    use_x          = ('x'          in param_names)
    use_edge_index = ('edge_index' in param_names)
    use_edge_attr  = ('edge_attr'  in param_names)
    use_batch      = ('batch'      in param_names)

    # Create metrics for this epoch
    train_accuracy_metric = torchmetrics.Accuracy(task='binary').to(device)
    train_f1_metric = torchmetrics.F1Score(task='binary').to(device)

    # Determine total batches for logging
    total_batches = len(train_loader)
    epoch_str = f"{epoch}/{n_epochs}" if (epoch is not None and n_epochs is not None) else "1/1"

    for batch_idx, batch in enumerate(train_loader):
        batch = batch.to(device)

        # Forward pass - build kwargs based on model signature
        optimizer.zero_grad()
        kwargs = {}
        if use_x:          kwargs['x'] = batch.x
        if use_edge_index: kwargs['edge_index'] = batch.edge_index
        if use_edge_attr and hasattr(batch, 'edge_attr'):
            kwargs['edge_attr'] = batch.edge_attr
        if use_batch and hasattr(batch, 'batch'):
            kwargs['batch'] = batch.batch
        logits = model(**kwargs)

        y_gt = batch.y.view(-1).long()

        # Compute loss
        loss = loss_fn(logits, y_gt)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

        # Update metrics with class predictions
        batch_preds = torch.argmax(logits, dim=-1)
        train_accuracy_metric.update(batch_preds, y_gt)
        train_f1_metric.update(batch_preds, y_gt)

        # Log progress at specified frequency
        current_batch = batch_idx + 1
        if verbose and (current_batch % log_freq == 0 or current_batch == total_batches):
            avg_loss = total_loss / current_batch
            current_acc = train_accuracy_metric.compute().item()
            current_f1 = train_f1_metric.compute().item()
            print(f"Epoch [{epoch_str}], Step [{current_batch}/{total_batches}], "
                  f"Loss: {avg_loss:.4f}, Acc: {current_acc:.4f}, F1: {current_f1:.4f}")

    # Compute final metrics for the epoch
    avg_loss = total_loss / len(train_loader)
    train_acc = train_accuracy_metric.compute().item()
    train_f1 = train_f1_metric.compute().item()

    return avg_loss, train_acc, train_f1

def val(model, val_loader, loss_fn, device,
        epoch=None, n_epochs=None, verbose=True, log_final_only=True):
    """
    Performs one full validation epoch over the val_loader.

    Computes all validation metrics internally including precision, recall,
    and confusion matrix, eliminating the need for a separate logger class.

    Args:
        model: The GNN model to evaluate
        val_loader: DataLoader for the validation set
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        device: Device to run evaluation on ('cpu' or 'cuda')
        epoch: Current epoch number (for logging, optional)
        n_epochs: Total number of epochs (for logging, optional)
        verbose: If True, print results (default: True)
        log_final_only: If True, only log final results after all batches (default: True)

    Returns:
        tuple: (avg_loss, val_acc, val_f1, precision, recall, cm, preds, targets)
            - avg_loss: Average validation loss over all batches
            - val_acc: Validation accuracy (computed using torchmetrics)
            - val_f1: Validation F1 score (computed using torchmetrics)
            - precision: Validation precision (computed using torchmetrics)
            - recall: Validation recall (computed using torchmetrics)
            - cm: Confusion matrix as numpy array, shape (2, 2)
            - preds: Numpy array of class predictions {0, 1}
            - targets: Numpy array of ground truth labels {0, 1}
    """
    model.eval()
    total_loss = 0.0
    logit_list, label_list = [], []

    # Introspect model signature once
    sig = inspect.signature(model.forward)
    param_names = set(sig.parameters.keys())
    use_x          = ('x'          in param_names)
    use_edge_index = ('edge_index' in param_names)
    use_edge_attr  = ('edge_attr'  in param_names)
    use_batch      = ('batch'      in param_names)

    # Create metrics for validation
    val_accuracy_metric = torchmetrics.Accuracy(task='binary').to(device)
    val_f1_metric = torchmetrics.F1Score(task='binary').to(device)

    # Determine total batches for logging
    total_batches = len(val_loader)
    epoch_str = f"{epoch}/{n_epochs}" if (epoch is not None and n_epochs is not None) else "1/1"

    with torch.no_grad():
        for batch_idx, batch in enumerate(val_loader):
            batch = batch.to(device)

            # Forward pass - build kwargs based on model signature
            kwargs = {}
            if use_x:          kwargs['x'] = batch.x
            if use_edge_index: kwargs['edge_index'] = batch.edge_index
            if use_edge_attr and hasattr(batch, 'edge_attr'):
                kwargs['edge_attr'] = batch.edge_attr
            if use_batch and hasattr(batch, 'batch'):
                kwargs['batch'] = batch.batch
            logits = model(**kwargs)

            y_gt = batch.y.view(-1).long()

            # Compute loss
            batch_loss = loss_fn(logits, y_gt).item()
            total_loss += batch_loss

            # Get predictions and update metrics
            batch_preds = torch.argmax(logits, dim=-1)
            val_accuracy_metric.update(batch_preds, y_gt)
            val_f1_metric.update(batch_preds, y_gt)

            logit_list.append(batch_preds.detach().cpu().numpy().reshape(-1))
            label_list.append(y_gt.detach().cpu().numpy().reshape(-1))

            # Log per-batch progress if log_final_only=False
            if verbose and not log_final_only:
                current_batch = batch_idx + 1
                current_loss = total_loss / current_batch
                current_acc = val_accuracy_metric.compute().item()
                current_f1 = val_f1_metric.compute().item()
                print(f"Epoch [{epoch_str}], Step [{current_batch}/{total_batches}], "
                      f"Loss: {current_loss:.4f}, Acc: {current_acc:.4f}, F1: {current_f1:.4f}")

    # Compute final metrics
    avg_loss = total_loss / len(val_loader)
    val_acc = val_accuracy_metric.compute().item()
    val_f1 = val_f1_metric.compute().item()
    preds = np.hstack(logit_list).ravel()
    targets = np.hstack(label_list).ravel()

    # Compute precision, recall, and confusion matrix
    preds_tensor = torch.from_numpy(preds).long().to(device)
    targets_tensor = torch.from_numpy(targets).long().to(device)

    precision_metric = torchmetrics.Precision(task='binary').to(device)
    recall_metric = torchmetrics.Recall(task='binary').to(device)

    precision = precision_metric(preds_tensor, targets_tensor).item()
    recall = recall_metric(preds_tensor, targets_tensor).item()
    cm = confusion_matrix(targets, preds, labels=[0, 1])

    # Log final results only once (or only time if log_final_only=True)
    if verbose and log_final_only:
        print(f"Epoch [{epoch_str}], Step [{total_batches}/{total_batches}], "
              f"Loss: {avg_loss:.4f}, Acc: {val_acc:.4f}, F1: {val_f1:.4f}")

    return avg_loss, val_acc, val_f1, precision, recall, cm, preds, targets

class EarlyStopping:
    """
    Early stopping to stop training when validation metric stops improving.

    Args:
        patience: Number of epochs to wait for improvement
        min_delta: Minimum change to qualify as improvement
        mode: 'min' for loss, 'max' for accuracy/F1
        verbose: Print messages when stopping
    """
    def __init__(self, patience=10, min_delta=0.0001, mode='min', verbose=True):
        self.patience = patience
        self.min_delta = min_delta
        self.mode = mode
        self.verbose = verbose
        self.counter = 0
        self.best_score = None
        self.early_stop = False
        self.best_epoch = 0

    def __call__(self, current_score, epoch):
        """
        Args:
            current_score: Current validation metric value
            epoch: Current epoch number

        Returns:
            bool: True if training should stop
        """
        if self.best_score is None:
            self.best_score = current_score
            self.best_epoch = epoch
            return False

        improved = False
        if self.mode == 'min':
            improved = current_score < (self.best_score - self.min_delta)
        else:  # mode == 'max'
            improved = current_score > (self.best_score + self.min_delta)

        if improved:
            self.best_score = current_score
            self.best_epoch = epoch
            self.counter = 0
        else:
            self.counter += 1
            if self.verbose:
                print(f"  EarlyStopping: {self.counter}/{self.patience} epochs without improvement")

        if self.counter >= self.patience:
            if self.verbose:
                print(f"  EarlyStopping: Stopping training at epoch {epoch}")
                print(f"  Best score: {self.best_score:.6f} at epoch {self.best_epoch}")
            self.early_stop = True

        return self.early_stop

def perform_holdout_training(
    *,
    train_loader,
    val_loader,
    model_fn,
    optimizer_fn,
    loss_fn,
    device,
    epochs: int,
    experiment_name: str,
    scheduler_fn=None,
    scheduler=None,
    patience=10,              # NEW: early stopping patience
    min_delta=0.0001,        # NEW: minimum improvement threshold
    early_stop_metric='loss' # NEW: 'loss', 'f1', or 'accuracy'
):
    """
    Args:
        train_loader: DataLoader for training set
        val_loader: DataLoader for validation set
        model_fn: Callable that returns a model instance
        optimizer_fn: Callable that takes model and returns optimizer
        loss_fn: Loss function (e.g., CrossEntropyLoss)
        device: Device to run on ('cpu' or 'cuda')
        epochs: Number of training epochs
        experiment_name: Name for saving results
        scheduler_fn: Optional callable that takes optimizer and returns scheduler
        patience: Number of epochs to wait for improvement before stopping
        min_delta: Minimum change to qualify as improvement
        early_stop_metric: Metric to monitor ('loss', 'f1', 'accuracy')
    Returns:
        dict: Holdout training results and metrics
    """
    model = model_fn().to(device)
    optimizer = optimizer_fn(model)
    scheduler = scheduler_fn(optimizer) if scheduler_fn else None

    exp_dir = get_experiment_dir(experiment_name)

    # Initialize tracking lists
    train_losses = []
    train_accs = []
    train_f1s = []
    val_losses = []
    val_accs = []
    val_f1s = []
    precisions = []
    recalls = []
    confusion_matrices = []

    # Track best model based on early_stop_metric
    if early_stop_metric == 'loss':
        best_metric_value = float('inf')
    elif early_stop_metric in ['f1', 'accuracy']:
        best_metric_value = 0.0
    else:
        raise ValueError(f"Invalid early_stop_metric: {early_stop_metric}. Must be 'loss', 'f1', or 'accuracy'")

    best_model_state = None
    best_epoch = 0

    # NEW: Initialize early stopping
    mode = 'min' if early_stop_metric == 'loss' else 'max'
    early_stopper = EarlyStopping(patience=patience, min_delta=min_delta,
                                  mode=mode, verbose=True)

    print(f"\n=== Holdout Training: {experiment_name} ===")
    print(f"Early stopping: patience={patience}, metric={early_stop_metric}")

    for epoch in range(epochs):
        # Training phase
        tr_loss, tr_acc, tr_f1 = train(
            model, train_loader, optimizer, loss_fn, device,
            epoch=epoch + 1, n_epochs=epochs, verbose=True, log_freq=10
        )

        # Validation phase
        vl_loss, vl_acc, vl_f1, precision, recall, cm, vl_preds, vl_targets = val(
            model, val_loader, loss_fn, device,
            epoch=epoch + 1, n_epochs=epochs, verbose=False, log_final_only=True
        )

        # Update learning rate scheduler
        if scheduler:
            scheduler.step(vl_f1)

        # Append metrics to history
        train_losses.append(tr_loss)
        train_accs.append(tr_acc)
        train_f1s.append(tr_f1)
        val_losses.append(vl_loss)
        val_accs.append(vl_acc)
        val_f1s.append(vl_f1)
        precisions.append(precision)
        recalls.append(recall)
        confusion_matrices.append(cm.tolist())

        # Track best model by early_stop_metric (consistent with early stopping)
        current_metric = {'loss': vl_loss, 'accuracy': vl_acc, 'f1': vl_f1}[early_stop_metric]

        is_better = False
        if early_stop_metric == 'loss':
            is_better = current_metric < best_metric_value
        else:  # f1 or accuracy
            is_better = current_metric > best_metric_value

        if is_better:
            best_metric_value = current_metric
            best_model_state = model.state_dict()
            best_epoch = epoch
            metric_name = {'loss': 'Loss', 'accuracy': 'Accuracy', 'f1': 'F1'}[early_stop_metric]
            print(f"  ★ New best model! Val {metric_name}: {current_metric:.6f}")

        # Print epoch summary
        print(
            f"Epoch {epoch+1}/{epochs}  "
            f"TR L:{tr_loss:.4f} A:{tr_acc:.4f} F1:{tr_f1:.4f} | "
            f"VL L:{vl_loss:.4f} A:{vl_acc:.4f} F1:{vl_f1:.4f}"
        )

        # NEW: Check early stopping
        stop_metric = {'loss': vl_loss, 'accuracy': vl_acc, 'f1': vl_f1}[early_stop_metric]
        if early_stopper(stop_metric, epoch):
            print(f"\nEarly stopping triggered at epoch {epoch+1}")
            print(f"Best model was at epoch {best_epoch+1}")
            break

    # Save per-epoch history
    holdout_metrics = {
        "train_loss": train_losses,
        "train_accuracy": train_accs,
        "train_f1": train_f1s,
        "val_loss": val_losses,
        "val_accuracy": val_accs,
        "val_f1": val_f1s,
        "precision": precisions,
        "recall": recalls,
        "confusion_matrix": confusion_matrices,
        "best_epoch": best_epoch,  # NEW: save which epoch was best
        "stopped_at_epoch": len(train_losses) - 1,  # NEW: track when training stopped
    }
    save_metrics(holdout_metrics, exp_dir, f"{experiment_name}_holdout")

    # Load and save best model
    if best_model_state is not None:
        model.load_state_dict(best_model_state)
        save_best_model(model, exp_dir, f"{experiment_name}_holdout")

    # FIXED: Create final summary from BEST epoch, not last epoch
    best_cm = np.array(confusion_matrices[best_epoch])

    final_summary = {
        "val_accuracy": val_accs[best_epoch],       # FIXED: use best_epoch
        "val_f1": val_f1s[best_epoch],              # FIXED: use best_epoch
        "precision": precisions[best_epoch],        # FIXED: use best_epoch
        "recall": recalls[best_epoch],              # FIXED: use best_epoch
        "confusion_matrix": best_cm.tolist(),       # FIXED: use best_epoch
        "best_epoch": best_epoch + 1,               # NEW: 1-indexed for readability
        "total_epochs_trained": len(train_losses),  # NEW: how many epochs ran
    }
    save_metrics(final_summary, exp_dir, f"{experiment_name}_holdout_final")

    print(f"\n{'='*70}")
    print(f"Training Complete: {experiment_name}")
    print(f"{'='*70}")
    print(f"Best model from epoch {best_epoch+1} (stopped at epoch {len(train_losses)})")
    print(f"Selection criterion: {early_stop_metric}")
    print(f"Best validation metrics:")
    print(f"  Accuracy:  {val_accs[best_epoch]:.4f}")
    print(f"  F1 Score:  {val_f1s[best_epoch]:.4f}")
    print(f"  Loss:      {val_losses[best_epoch]:.4f}")

    return holdout_metrics, final_summary

def perform_kfold_training(
    *,
    fold_loaders,
    model_fn,
    optimizer_fn,
    loss_fn,
    device,
    epochs: int,
    experiment_name: str,
    scheduler_fn=None,
    patience=10,
    min_delta=0.0001,
    early_stop_metric='loss'
):
    """
    Performs K-fold cross-validation training without using MetricsLogger.

    All metric tracking is done explicitly with lists, making the code more
    readable and eliminating ambiguity.

    Args:
        fold_loaders: List of (train_loader, val_loader) pairs from get_kfold_subject_loaders_v2
        model_fn: Callable that returns a new model instance
        optimizer_fn: Callable that takes model and returns optimizer
        loss_fn: Loss function (e.g., nn.CrossEntropyLoss)
        device: Device to train on ('cpu' or 'cuda')
        epochs: Number of epochs per fold
        experiment_name: Name for saving results
        scheduler_fn: Optional callable that takes optimizer and returns scheduler
        patience: Number of epochs to wait for improvement before stopping
        min_delta: Minimum change to qualify as improvement
        early_stop_metric: Metric to monitor ('loss', 'f1', 'accuracy')
    Returns:
        dict: K-fold training results and metrics

    """
    exp_dir = get_experiment_dir(experiment_name)
    all_fold_metrics = []
    final_cms = []
    best_epochs_per_fold = []

    for fold_idx, (train_loader, val_loader) in enumerate(fold_loaders, start=1):
        print(f"\n{'='*70}")
        print(f"Fold {fold_idx}/{len(fold_loaders)}")
        print(f"{'='*70}")

        # Initialize model, optimizer, scheduler
        model = model_fn().to(device)
        optimizer = optimizer_fn(model)
        scheduler = scheduler_fn(optimizer) if scheduler_fn else None

        # Initialize tracking lists
        train_losses = []
        train_accs = []
        train_f1s = []
        val_losses = []
        val_accs = []
        val_f1s = []
        precisions = []
        recalls = []
        confusion_matrices = []

        # Track best model for this fold based on early_stop_metric
        if early_stop_metric == 'loss':
            best_metric_value = float('inf')
        elif early_stop_metric in ['f1', 'accuracy']:
            best_metric_value = 0.0
        else:
            raise ValueError(f"Invalid early_stop_metric: {early_stop_metric}. Must be 'loss', 'f1', or 'accuracy'")

        best_model_state = None
        best_epoch = 0

        # NEW: Initialize early stopping for this fold
        mode = 'min' if early_stop_metric == 'loss' else 'max'
        early_stopper = EarlyStopping(patience=patience, min_delta=min_delta,
                                      mode=mode, verbose=True)

        for epoch in range(epochs):
            # Training phase
            tr_loss, tr_acc, tr_f1 = train(
                model, train_loader, optimizer, loss_fn, device,
                epoch=epoch + 1, n_epochs=epochs, verbose=True, log_freq=10
            )

            # Validation phase
            vl_loss, vl_acc, vl_f1, precision, recall, cm, vl_preds, vl_targets = val(
                model, val_loader, loss_fn, device,
                epoch=epoch + 1, n_epochs=epochs, verbose=False, log_final_only=True
            )

            # Step scheduler
            if scheduler:
                scheduler.step(vl_f1)

            # Append metrics
            train_losses.append(tr_loss)
            train_accs.append(tr_acc)
            train_f1s.append(tr_f1)
            val_losses.append(vl_loss)
            val_accs.append(vl_acc)
            val_f1s.append(vl_f1)
            precisions.append(precision)
            recalls.append(recall)
            confusion_matrices.append(cm.tolist())

            # Track best model by early_stop_metric (consistent with early stopping)
            current_metric = {'loss': vl_loss, 'accuracy': vl_acc, 'f1': vl_f1}[early_stop_metric]

            is_better = False
            if early_stop_metric == 'loss':
                is_better = current_metric < best_metric_value
            else:  # f1 or accuracy
                is_better = current_metric > best_metric_value

            if is_better:
                best_metric_value = current_metric
                best_model_state = model.state_dict()
                best_epoch = epoch
                metric_name = {'loss': 'Loss', 'accuracy': 'Accuracy', 'f1': 'F1'}[early_stop_metric]
                print(f"  ★ New best model! Val {metric_name}: {current_metric:.6f}")

            # Print summary
            print(
                f"[Fold {fold_idx}] Epoch {epoch+1}/{epochs}  "
                f"TR L:{tr_loss:.4f} Acc:{tr_acc:.4f} F1:{tr_f1:.4f} | "
                f"VL L:{vl_loss:.4f} Acc:{vl_acc:.4f} F1:{vl_f1:.4f}"
            )

            # Check early stopping
            stop_metric = {'loss': vl_loss, 'accuracy': vl_acc, 'f1': vl_f1}[early_stop_metric]
            if early_stopper(stop_metric, epoch):
                print(f"\n[Fold {fold_idx}] Early stopping at epoch {epoch+1}")
                print(f"[Fold {fold_idx}] Best model was at epoch {best_epoch+1}")
                break

        # Save per-fold metrics
        fold_metrics = {
            "train_loss": train_losses,
            "train_accuracy": train_accs,
            "train_f1": train_f1s,
            "val_loss": val_losses,
            "val_accuracy": val_accs,
            "val_f1": val_f1s,
            "precision": precisions,
            "recall": recalls,
            "confusion_matrix": confusion_matrices,
            "best_epoch": best_epoch,
            "stopped_at_epoch": len(train_losses) - 1,
        }
        save_metrics(fold_metrics, exp_dir, f"{experiment_name}_fold_{fold_idx}")
        all_fold_metrics.append(fold_metrics)
        best_epochs_per_fold.append(best_epoch)

        # Load and save best model
        if best_model_state is not None:
            model.load_state_dict(best_model_state)
            save_best_model(model, exp_dir, f"{experiment_name}_fold_{fold_idx}")

        # FIXED: Use confusion matrix from BEST epoch, not last
        final_cms.append(np.array(confusion_matrices[best_epoch]))

        print(f"\n[Fold {fold_idx}] Summary:")
        print(f"  Best epoch: {best_epoch+1}/{len(train_losses)}")
        print(f"  Best val loss: {val_losses[best_epoch]:.4f}")
        print(f"  Best val accuracy: {val_accs[best_epoch]:.4f}")
        print(f"  Best val F1: {val_f1s[best_epoch]:.4f}")

    # Aggregate overall metrics
    # For curves, pad shorter folds with NaN to handle different stopping points
    max_epochs = max(len(m["train_loss"]) for m in all_fold_metrics)

    def pad_to_max(arr, max_len):
        """Pad array with NaN to max length"""
        padded = np.full(max_len, np.nan)
        padded[:len(arr)] = arr
        return padded

    tl = np.array([pad_to_max(m["train_loss"], max_epochs) for m in all_fold_metrics])
    ta = np.array([pad_to_max(m["train_accuracy"], max_epochs) for m in all_fold_metrics])
    tf = np.array([pad_to_max(m["train_f1"], max_epochs) for m in all_fold_metrics])
    vl = np.array([pad_to_max(m["val_loss"], max_epochs) for m in all_fold_metrics])
    va = np.array([pad_to_max(m["val_accuracy"], max_epochs) for m in all_fold_metrics])
    vf = np.array([pad_to_max(m["val_f1"], max_epochs) for m in all_fold_metrics])

    # Use nanmean to ignore NaN from early stopped folds
    overall_metrics = {
        "train_loss": np.nanmean(tl, axis=0).tolist(),
        "train_accuracy": np.nanmean(ta, axis=0).tolist(),
        "train_f1": np.nanmean(tf, axis=0).tolist(),
        "val_loss": np.nanmean(vl, axis=0).tolist(),
        "val_accuracy": np.nanmean(va, axis=0).tolist(),
        "val_f1": np.nanmean(vf, axis=0).tolist(),
        "best_epochs_per_fold": [e+1 for e in best_epochs_per_fold],
    }

    best_fold_precisions = [all_fold_metrics[i]["precision"][best_epochs_per_fold[i]]
                            for i in range(len(all_fold_metrics))]
    best_fold_recalls = [all_fold_metrics[i]["recall"][best_epochs_per_fold[i]]
                        for i in range(len(all_fold_metrics))]

    best_fold_accuracies = [all_fold_metrics[i]["val_accuracy"][best_epochs_per_fold[i]]
                            for i in range(len(all_fold_metrics))]
    best_fold_f1s = [all_fold_metrics[i]["val_f1"][best_epochs_per_fold[i]]
                    for i in range(len(all_fold_metrics))]

    # Aggregate confusion matrix (sum across folds)
    agg_cm = np.sum(final_cms, axis=0) if final_cms else np.zeros((2, 2), int)

    # Calculate overall metrics from aggregated CM
    if agg_cm.shape == (2, 2) and agg_cm.sum() > 0:
        tn, fp, fn, tp = agg_cm.ravel()
        overall_accuracy = (tp + tn) / (tp + tn + fp + fn) if (tp + tn + fp + fn) > 0 else 0.0
        overall_precision = tp / (tp + fp) if (tp + fp) > 0 else 0.0
        overall_recall = tp / (tp + fn) if (tp + fn) > 0 else 0.0
        overall_f1 = 2 * (overall_precision * overall_recall) / (overall_precision + overall_recall) \
                     if (overall_precision + overall_recall) > 0 else 0.0
    else:
        overall_precision = overall_recall = overall_f1 = 0.0

    overall_metrics.update({
        "accuracy_mean": np.mean(best_fold_accuracies),
        "f1_mean": np.mean(best_fold_f1s),
        "precision_mean": np.mean(best_fold_precisions),
        "recall_mean": np.mean(best_fold_recalls),
        "accuracy_overall": overall_accuracy,
        "precision_overall": overall_precision,
        "recall_overall": overall_recall,
        "f1_overall": overall_f1,
        "confusion_matrix_overall": agg_cm.tolist(),
    })
    save_metrics(overall_metrics, exp_dir, f"{experiment_name}_kfold_overall")

    # Print summary
    print(f"\n{'='*70}")
    print(f"K-Fold Training Complete: {experiment_name}")
    print(f"{'='*70}")
    print(f"Best epochs per fold: {overall_metrics['best_epochs_per_fold']}")
    print(f"Overall Metrics (from best epochs across {len(fold_loaders)} folds):")
    print(f"  Precision: {overall_precision:.4f}")
    print(f"  Recall:    {overall_recall:.4f}")
    print(f"  F1 Score:  {overall_f1:.4f}")
    print(f"  Confusion Matrix:\n{agg_cm}")

    return {"folds": all_fold_metrics, "overall": overall_metrics}

### 4.2. Run Training/Validation

#### 4.2.1. Holdout

In [33]:
# # Setup
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# n_epochs = 50

# train_loader_v2, val_loader_v2 = helper_utils.get_holdout_subject_loaders_v2(
#         dataset,
#         batch_size=8,
#         shuffle_train=True,
#         num_workers=0,
#         pin_memory=False,
#         val_ratio=0.2,
#         random_state=42,
#         show_subjects=False,
#         transform=transform,
#         verbose=False
#     )

# candidate_hyperparams = {
#     "candidate_1": {
#         "n_layers": 3,
#         "n_filters": [192, 384, 512],
#         "fc_size": 448,
#         "dropout_rate": 0.4,
#         "learning_rate": 0.0000101,
#     },
# }

# # Pick candidate and unpack the hyperparameters
# candidate = "candidate_1"
# selected_candidate = candidate_hyperparams[candidate]
# n_layers = selected_candidate["n_layers"]
# n_filters = selected_candidate["n_filters"]
# fc_size = selected_candidate["fc_size"]
# dropout_rate = selected_candidate["dropout_rate"]
# learning_rate = selected_candidate["learning_rate"]

# # Define model factory function
# def model_fn():
#     return FlexibleGINNet(
#         in_channels=6, n_layers=n_layers, n_filters=n_filters,
#         fc_size=fc_size, dropout_rate=dropout_rate, edge_dim=2, n_classes=2
#     )

# # Define optimizer factory function
# def optimizer_fn(model):
#     return torch.optim.Adam(model.parameters(), lr=learning_rate)

# # Define scheduler factory function (optional)
# def scheduler_fn(optimizer):
#     return torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='max', factor=0.5, patience=5, verbose=False
#     )

# # Loss function
# class_weights = calculate_class_weights(train_loader_v2, device)
# loss_fn = nn.CrossEntropyLoss(weight=class_weights)

# # Run holdout training
# # CRITICAL: Use n_epochs (set to match Optuna's epoch count) instead of hardcoded 100
# EXPERIMENT_NAME = f"gine_holdout_{candidate}"
# results = perform_holdout_training(
#     train_loader=train_loader_v2,
#     val_loader=val_loader_v2,
#     model_fn=model_fn,
#     optimizer_fn=optimizer_fn,
#     loss_fn=loss_fn,
#     device=device,
#     epochs=n_epochs,
#     experiment_name=EXPERIMENT_NAME,
#     scheduler_fn=scheduler_fn,
#     patience=99999,
#     min_delta=0.0001,
#     early_stop_metric='f1'
# )

# # Visualize results
# exp_dir = get_experiment_dir(EXPERIMENT_NAME)
# visualize_training_results(
#     results=results,
#     save_dir=exp_dir,
#     experiment_name=EXPERIMENT_NAME,
#     training_type="holdout"
# )

#### 4.2.2 K-Fold

##### 4.2.2.1. K-Fold Hyperparameters - Without Focal Loss

In [34]:
# ==========================================
# DEVICE CONFIGURATION
# ==========================================
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# ==========================================
# DATA LOADING AND PREPROCESSING
# ==========================================
# Dataset configuration
root_dir = r'/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/data/processed-new/GNG'
DATA_TYPE = 'hbt'
MAX_TRIALS = 4

# Load dataset
dataset = fNIRSGraphDatasetNonRecurrent(
    root=root_dir,
    data_type=DATA_TYPE,
    max_trials=MAX_TRIALS,
    directed=True,
    corr_threshold=0.1,
    self_loops=True
)

# Calculate normalization statistics
mean_dict, std_dict = get_mean_std(dataset)

# Define data transformations
val_transform = get_transformations(
        mean=mean_dict,
        std=std_dict,
        augment=False,
        use_positional_encoding=False,
    )

# ==========================================
# K-FOLD CROSS-VALIDATION SETUP
# ==========================================
fold_loaders_v2 = helper_utils.get_kfold_subject_loaders_v2(
    dataset,
    n_splits=5,
    batch_size=8,
    shuffle_train=True,
    num_workers=0,
    pin_memory=False,
    random_state=42,
    show_subjects=True,
    transform=val_transform
)

# ==========================================
# LOSS FUNCTION CONFIGURATION
# ==========================================
# Calculate class weights for handling class imbalance
class_weights = calculate_class_weights(fold_loaders_v2, device, use_sqrt=False)
loss_fn = nn.CrossEntropyLoss()

# ==========================================
# HYPERPARAMETER CANDIDATES
# ==========================================
candidate_hyperparams = {
    "candidate_1": {
        "n_layers": 3,
        "n_filters": [128, 80, 112],
        "heads": [2, 6, 4],
        "dropout": 0.7,
        "fc_size": 96,
        "use_norm": True,
        "norm_type": "batch",
        "learning_rate": 0.018468,
    },
    "candidate_2": {
        "n_layers": 2,
        "n_filters": [48, 80],
        "heads": [2, 6],
        "dropout": 0.7,
        "fc_size": 96,
        "use_norm": True,
        "norm_type": "batch",
        "learning_rate": 0.052206,
    },
    "candidate_3": {
        "n_layers": 2,
        "n_filters": [96, 64],
        "heads": [6, 4],
        "dropout": 0.3,
        "fc_size": 96,
        "use_norm": True,
        "norm_type": "batch",
        "learning_rate": 0.039745,
    },
    "candidate_4": {
        "n_layers": 2,
        "n_filters": [112, 32],
        "heads": [6, 4],
        "dropout": 0.4,
        "fc_size": 96,
        "use_norm": True,
        "norm_type": "batch",
        "learning_rate": 0.036652,
    },
}

# ==========================================
# HYPERPARAMETER SELECTION
# ==========================================
# Select candidate and extract hyperparameters
candidate = "candidate_4"
selected_candidate = candidate_hyperparams[candidate]
n_layers = selected_candidate["n_layers"]
n_filters = selected_candidate["n_filters"]
heads = selected_candidate["heads"]
dropout = selected_candidate["dropout"]
fc_size = selected_candidate["fc_size"]
use_norm = selected_candidate["use_norm"]
learning_rate = selected_candidate["learning_rate"]

# ==========================================
# TRAINING CONFIGURATION
# ==========================================
n_epochs = 100

# Model factory function
def model_fn():
    return FlexibleGATNet(
        in_channels=6,
        n_layers=n_layers,
        n_filters=n_filters,
        heads=heads,
        dropout=dropout,
        fc_size=fc_size,
        use_norm=use_norm,
        edge_dim=2,
        n_classes=2,
        use_gine_first_layer=True
    )

# Optimizer factory function
def optimizer_fn(model):
    return torch.optim.Adam(model.parameters(), lr=learning_rate)

# Learning rate scheduler factory function
def scheduler_fn(optimizer):
    return torch.optim.lr_scheduler.ReduceLROnPlateau(
        optimizer, mode='max', factor=0.5, patience=3, verbose=False
    )

# ==========================================
# MODEL TRAINING
# ==========================================
EXPERIMENT_NAME = f"gat_kfold_{DATA_TYPE}_mt_{MAX_TRIALS}_{candidate}"

results = perform_kfold_training(
    fold_loaders=fold_loaders_v2,
    model_fn=model_fn,
    optimizer_fn=optimizer_fn,
    loss_fn=loss_fn,
    device=device,
    epochs=n_epochs,
    experiment_name=EXPERIMENT_NAME,
    scheduler_fn=scheduler_fn,
    patience=9999,
    min_delta=0.0001,
    early_stop_metric='f1'
)

# ==========================================
# RESULTS VISUALIZATION
# ==========================================
exp_dir = get_experiment_dir(EXPERIMENT_NAME)
visualize_training_results(
    results=results,
    save_dir=exp_dir,
    experiment_name=EXPERIMENT_NAME,
    training_type="kfold"
)

Processing...
Done!


=== Dataset Overview ===
Total graphs           : 248
Unique subjects        : 62
Node feature x shape   : (23, 6) | dtype=torch.float32
Edge_attr shape        : (485, 2) | dtype=torch.float32
Per-graph label counts : {0: 132, 1: 116}
Per-subject label cnts : {0: 33, 1: 29}
------------------------
=== K-Fold 1/5 (by subject) with Transform ===
Train
  graphs: 196 | subjects: 49
  label counts (graphs): {0: 104, 1: 92}
  label counts (subjects): {0: 26, 1: 23}
  subjects: ['AA011', 'AA013', 'AA041', 'AA056', 'AA064', 'AA089', 'AA090', 'AA093', 'AA094', 'AA097', 'AA098', 'AA099', 'AH017', 'AH018', 'AH019', 'AH020', 'AH021', 'AH022', 'AH023', 'AH024', 'AH025', 'AH026', 'AH027', 'AH030', 'AH031', 'AH034', 'AH037', 'AH038', 'AH039', 'AH040', 'AH043', 'AH044', 'AH045', 'AH046', 'AH047', 'AH048', 'AH049', 'AH050', 'LA042', 'LA052', 'LA053', 'LA054', 'LA057', 'LA058', 'LA059', 'LA063', 'LA091', 'LA095', 'LA096']
Val  
  graphs: 52 | subjects: 13
  label counts (graphs): {0: 28, 1: 24}
  label

{'loss_plot': 'experiments/20260428/gat_kfold_hbt_mt_4_candidate_4/gat_kfold_hbt_mt_4_candidate_4_loss_curves.png',
 'accuracy_plot': 'experiments/20260428/gat_kfold_hbt_mt_4_candidate_4/gat_kfold_hbt_mt_4_candidate_4_accuracy_curves.png',
 'f1_plot': 'experiments/20260428/gat_kfold_hbt_mt_4_candidate_4/gat_kfold_hbt_mt_4_candidate_4_f1_curves.png',
 'confusion_matrix_plot': 'experiments/20260428/gat_kfold_hbt_mt_4_candidate_4/gat_kfold_hbt_mt_4_candidate_4_confusion_matrix.png'}

##### 4.2.2.2. K-Fold Hyperparameters - With Focal Loss

In [35]:
# # ==========================================
# # DEVICE CONFIGURATION
# # ==========================================
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# # ==========================================
# # DATA LOADING AND PREPROCESSING
# # ==========================================
# # Dataset configuration
# root_dir = r'/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/data/processed_data/GNG'
# DATA_TYPE = 'hbt'
# MAX_TRIALS = 4

# # Load dataset
# dataset = fNIRSGraphDatasetNonRecurrent(
#     root=root_dir,
#     data_type=DATA_TYPE,
#     max_trials=MAX_TRIALS,
#     directed=True,
#     corr_threshold=0.1,
#     self_loops=True
# )

# # Calculate normalization statistics
# mean_dict, std_dict = get_mean_std(dataset)

# # Define data transformations
# val_transform = get_transformations(
#         mean=mean_dict,
#         std=std_dict,
#         augment=False,
#         use_positional_encoding=False,
#     )

# # ==========================================
# # K-FOLD CROSS-VALIDATION SETUP
# # ==========================================
# fold_loaders_v2 = helper_utils.get_kfold_subject_loaders_v2(
#     dataset,
#     n_splits=5,
#     batch_size=8,
#     shuffle_train=True,
#     num_workers=0,
#     pin_memory=False,
#     random_state=42,
#     show_subjects=True,
#     transform=val_transform
# )

# # ==========================================
# # HYPERPARAMETER CANDIDATES
# # ==========================================
# candidate_hyperparams = {
#     "candidate_1": {
#         "n_layers": 1,
#         "n_filters": [48],
#         "heads": [6],
#         "dropout": 0.7,
#         "fc_size": 128,
#         "use_norm": True,
#         "norm_type": "layer",
#         "learning_rate": 0.076526,
#         "focal_alpha": 0.707753,
#         "focal_gamma": 1.216092
#     },
#     "candidate_2": {
#         "n_layers": 1,
#         "n_filters": [96],
#         "heads": [4],
#         "dropout": 0.7,
#         "fc_size": 128,
#         "use_norm": True,
#         "norm_type": "batch",
#         "learning_rate": 0.065112,
#         "focal_alpha": 0.720368,
#         "focal_gamma": 2.546075
#     },
#     "candidate_3": {
#         "n_layers": 3,
#         "n_filters": [48, 80, 16],
#         "heads": [4, 2, 4],
#         "dropout": 0.5,
#         "fc_size": 96,
#         "use_norm": False,
#         "learning_rate": 0.014917,
#         "focal_alpha": 0.458534,
#         "focal_gamma": 1.181633
#     },
#     "candidate_4": {
#         "n_layers": 2,
#         "n_filters": [32, 64],
#         "heads": [6, 4],
#         "dropout": 0.5,
#         "fc_size": 96,
#         "use_norm": True,
#         "norm_type": "batch",
#         "learning_rate": 0.018819,
#         "focal_alpha": 0.611054,
#         "focal_gamma": 2.002332
#     },
# }

# # ==========================================
# # HYPERPARAMETER SELECTION
# # ==========================================
# # Select candidate and extract hyperparameters
# candidate = "candidate_4"
# selected_candidate = candidate_hyperparams[candidate]
# n_layers = selected_candidate["n_layers"]
# n_filters = selected_candidate["n_filters"]
# heads = selected_candidate["heads"]
# dropout = selected_candidate["dropout"]
# fc_size = selected_candidate["fc_size"]
# use_norm = selected_candidate["use_norm"]
# # Only extract norm_type if use_norm is True
# if use_norm:
#     norm_type = selected_candidate["norm_type"]
# learning_rate = selected_candidate["learning_rate"]
# focal_alpha = selected_candidate["focal_alpha"]
# focal_gamma = selected_candidate["focal_gamma"]

# # ==========================================
# # LOSS FUNCTION CONFIGURATION
# # ==========================================
# loss_fn = FocalLoss(
#             alpha=focal_alpha,
#             gamma=focal_gamma,
#             reduction='mean'
# )

# # ==========================================
# # TRAINING CONFIGURATION
# # ==========================================
# n_epochs = 50

# # Model factory function
# if use_norm:
#     def model_fn():
#         return FlexibleGATNet(
#             in_channels=6,
#             n_layers=n_layers,
#             n_filters=n_filters,
#             heads=heads,
#             dropout=dropout,
#             fc_size=fc_size,
#             use_norm=True,
#             norm_type=norm_type,
#             edge_dim=2,
#             n_classes=2,
#             use_gine_first_layer=True
#         )
# else:
#     def model_fn():
#         return FlexibleGATNet(
#             in_channels=6,
#             n_layers=n_layers,
#             n_filters=n_filters,
#             heads=heads,
#             dropout=dropout,
#             fc_size=fc_size,
#             use_norm=False,
#             edge_dim=2,
#             n_classes=2,
#             use_gine_first_layer=True
#         )

# # Optimizer factory function
# def optimizer_fn(model):
#     return torch.optim.Adam(model.parameters(), lr=learning_rate)

# # Learning rate scheduler factory function
# def scheduler_fn(optimizer):
#     return torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode='max', factor=0.5, patience=3, verbose=False
#     )

# # ==========================================
# # MODEL TRAINING
# # ==========================================
# EXPERIMENT_NAME = f"gat_kfold_{DATA_TYPE}_mt_{MAX_TRIALS}_{candidate}"

# results = perform_kfold_training(
#     fold_loaders=fold_loaders_v2,
#     model_fn=model_fn,
#     optimizer_fn=optimizer_fn,
#     loss_fn=loss_fn,
#     device=device,
#     epochs=n_epochs,
#     experiment_name=EXPERIMENT_NAME,
#     scheduler_fn=scheduler_fn,
#     patience=9999,
#     min_delta=0.001,
#     early_stop_metric='f1'
# )

# # ==========================================
# # RESULTS VISUALIZATION
# # ==========================================
# exp_dir = get_experiment_dir(EXPERIMENT_NAME)
# visualize_training_results(
#     results=results,
#     save_dir=exp_dir,
#     experiment_name=EXPERIMENT_NAME,
#     training_type="kfold"
# )

##### 4.2.2.3. K-Fold Augmentation

In [36]:
# # Setup
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# n_epochs = 50

# root_dir = (
#     r"/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/data/processed_data/GNG"
# )
# DATA_TYPE = "hbt"
# MAX_TRIALS = 4
# dataset = fNIRSGraphDatasetNonRecurrent(
#     root=root_dir,
#     data_type=DATA_TYPE,
#     max_trials=MAX_TRIALS,
#     directed=True,
#     corr_threshold=0.1,
#     self_loops=True,
# )

# mean_dict, std_dict = get_mean_std(dataset)

# candidate_augmentations = {
#     "candidate_1": {
#         "use_positional_encoding": False,
#         "feature_mask_p": 0.1,
#         "feature_mask_mode": "col",
#         "edge_dropout_p": 0.25,
#     },
#     "candidate_2": {
#         "use_positional_encoding": False,
#         "feature_mask_p": 0.05,
#         "feature_mask_mode": "row",
#         "edge_dropout_p": 0.05,
#     },
#     "candidate_3": {
#         "use_positional_encoding": True,
#         "pe_walk_length": 3,
#         "feature_mask_p": 0.05,
#         "feature_mask_mode": "col",
#         "edge_dropout_p": 0.1,
#     },
#     "candidate_4": {
#         "use_positional_encoding": True,
#         "pe_walk_length": 5,
#         "feature_mask_p": 0.1,
#         "feature_mask_mode": "all",
#         "edge_dropout_p": 0.3,
#     },
# }

# candidate = 'candidate_4'
# aug_params = candidate_augmentations[candidate]
# use_positional_encoding = aug_params["use_positional_encoding"]
# pe_walk_length = aug_params.get("pe_walk_length", None)  # Use .get() to avoid KeyError
# feature_mask_p = aug_params["feature_mask_p"]
# feature_mask_mode = aug_params["feature_mask_mode"]
# edge_dropout_p = aug_params["edge_dropout_p"]

# # Training transform (with augmentation)
# if use_positional_encoding:
#     train_transform = get_transformations(
#         mean=mean_dict,
#         std=std_dict,
#         augment=True,  # Enable training augmentation
#         edge_dropout_p=edge_dropout_p,
#         feature_mask_p=feature_mask_p,
#         feature_mask_mode=feature_mask_mode,
#         use_positional_encoding=use_positional_encoding,
#         pe_walk_length=pe_walk_length,
#     )
#     # Validation transform (NO training augmentation, but same PE)
#     val_transform = get_transformations(
#         mean=mean_dict,
#         std=std_dict,
#         augment=False,  # NO training augmentation for validation
#         use_positional_encoding=use_positional_encoding,
#         pe_walk_length=pe_walk_length,
#     )
# else:
#     train_transform = get_transformations(
#         mean=mean_dict,
#         std=std_dict,
#         augment=True,  # Enable training augmentation
#         edge_dropout_p=edge_dropout_p,
#         feature_mask_p=feature_mask_p,
#         feature_mask_mode=feature_mask_mode,
#         use_positional_encoding=use_positional_encoding,
#     )
#     # Validation transform (NO training augmentation, no PE)
#     val_transform = get_transformations(
#         mean=mean_dict,
#         std=std_dict,
#         augment=False,  # NO training augmentation for validation
#         use_positional_encoding=use_positional_encoding,
#     )

# fold_loaders_v2 = helper_utils.get_kfold_subject_loaders_v2(
#     dataset,
#     n_splits=5,
#     batch_size=8,
#     shuffle_train=True,
#     num_workers=0,
#     pin_memory=False,
#     random_state=42,
#     show_subjects=True,
#     train_transform=train_transform,
#     val_transform=val_transform,
# )

# # Calculate class weights
# class_weights = calculate_class_weights(fold_loaders_v2, device)

# # Base input channels + positional encoding if used
# base_in_channels = 6
# if use_positional_encoding and pe_walk_length is not None:
#     in_channels = base_in_channels + pe_walk_length
# else:
#     in_channels = base_in_channels

# n_layers = 2
# n_filters = [112, 32]
# heads = [6, 4]
# dropout = 0.4
# fc_size = 96
# use_norm = True
# norm_type = 'batch'
# learning_rate = 0.036652

# # Define model factory function
# def model_fn():
#     return FlexibleGATNet(
#         in_channels=in_channels,
#         n_layers=n_layers,
#         n_filters=n_filters,
#         heads=heads,
#         dropout=dropout,
#         fc_size=fc_size,
#         use_norm=use_norm,
#         edge_dim=2,
#         n_classes=2,
#         use_gine_first_layer=True,
#     )


# # Define optimizer factory function
# def optimizer_fn(model):
#     return torch.optim.Adam(model.parameters(), lr=learning_rate)


# # Define scheduler factory function (optional)
# def scheduler_fn(optimizer):
#     return torch.optim.lr_scheduler.ReduceLROnPlateau(
#         optimizer, mode="max", factor=0.5, patience=3, verbose=False
#     )

# # Loss function
# class_weights = calculate_class_weights(fold_loaders_v2, device)
# loss_fn = nn.CrossEntropyLoss()

# # Run K-fold training
# EXPERIMENT_NAME = f"gat_kfold_augmentations_{DATA_TYPE}_mt_{MAX_TRIALS}_{candidate}"
# results = perform_kfold_training(
#     fold_loaders=fold_loaders_v2,
#     model_fn=model_fn,
#     optimizer_fn=optimizer_fn,
#     loss_fn=loss_fn,
#     device=device,
#     epochs=n_epochs,
#     experiment_name=EXPERIMENT_NAME,
#     scheduler_fn=scheduler_fn,
#     patience=9999,
#     min_delta=0.0001,
#     early_stop_metric="f1",
# )

# # Visualize results
# exp_dir = get_experiment_dir(EXPERIMENT_NAME)
# visualize_training_results(
#     results=results,
#     save_dir=exp_dir,
#     experiment_name=EXPERIMENT_NAME,
#     training_type="kfold",
# )


## 5. Analyze Training and Validation Performance

### 5.1. Holdout

### 5.2. K-Fold

In [37]:
CANDIDATE = 4
MAX_TRIALS = 2
DATA_TYPE = 'hbt'
DATE = '20260428'

result_path = f'/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/src/notebook/experiments/{DATE}/gat_kfold_{DATA_TYPE}_mt_{MAX_TRIALS}_candidate_{CANDIDATE}'
helper_utils.report_training_results(result_path)


K-FOLD CROSS-VALIDATION RESULTS
Experiment: gat_kfold_hbt_mt_2_candidate_4
Folder: /home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/src/notebook/experiments/20260428/gat_kfold_hbt_mt_2_candidate_4
Number of Folds: 5

PER-FOLD BEST EPOCH PERFORMANCE
+--------+------------+----------+--------+-----------+--------+
| Fold   | Best Epoch | Accuracy |     F1 | Precision | Recall |
+--------+------------+----------+--------+-----------+--------+
| Fold 1 |         67 |   0.8077 | 0.7826 |    0.8182 | 0.7500 |
| Fold 2 |          2 |   0.6154 | 0.7059 |    0.5455 | 1.0000 |
| Fold 3 |         16 |   0.7917 | 0.8148 |    0.7333 | 0.9167 |
| Fold 4 |         14 |   0.7083 | 0.7742 |    0.6316 | 1.0000 |
| Fold 5 |         20 |   0.6667 | 0.6667 |    0.5714 | 0.8000 |
+--------+------------+----------+--------+-----------+--------+

PER-FOLD FINAL EPOCH PERFORMANCE
+--------+----------+--------+-----------+--------+
| Fold   | Accuracy |     F1 | Precision | Recall |
+--------+-----

### 5.3. Automated CSV Reporting

Export all experiment results to CSV format for easy analysis and comparison.

In [38]:
def export_experiments_to_csv(experiments_date_path, output_csv_filename='results_summary.csv'):
    """
    Scan experiment folders and export overall k-fold metrics to CSV.

    Supports both experiment naming patterns:
    1. Original: gat_kfold_{data_type}_mt_{max_trials}_candidate_{num}
    2. Augmentation: gat_kfold_augmentations_{data_type}_mt_{max_trials}_candidate_{num}

    Args:
        experiments_date_path: Path to the date folder (e.g., experiments/20251216/)
        output_csv_filename: Name of the output CSV file

    Returns:
        DataFrame with all experiment results
    """
    import os
    import pickle
    import pandas as pd
    from pathlib import Path
    import re

    results = []

    # Extract date from path
    date = Path(experiments_date_path).name

    # Walk through the directory structure: Date -> Candidate -> Experiment
    for candidate_folder in sorted(os.listdir(experiments_date_path)):
        candidate_path = os.path.join(experiments_date_path, candidate_folder)

        # Skip if not a directory or doesn't match pattern
        if not os.path.isdir(candidate_path) or not candidate_folder.startswith('candidate_'):
            continue

        # Extract candidate number
        try:
            candidate_num = int(candidate_folder.split('_')[1])
        except (IndexError, ValueError):
            continue

        # Iterate through experiment folders
        for experiment_folder in sorted(os.listdir(candidate_path)):
            experiment_path = os.path.join(candidate_path, experiment_folder)

            # Skip if not a directory
            if not os.path.isdir(experiment_path):
                continue

            # Parse experiment folder name supporting two patterns:
            # Pattern 1: gat_kfold_{data_type}_mt_{max_trials}_candidate_{num}
            # Pattern 2: gat_kfold_augmentations_{data_type}_mt_{max_trials}_candidate_{num}

            data_type = None
            max_trials = None
            experiment_type = "standard"  # Track experiment type

            # Try regex pattern first for robustness
            pattern = r'^gat_kfold(?:_augmentations)?_([a-z]+)_mt_(\d+)_candidate_\d+$'
            match = re.match(pattern, experiment_folder)

            if match:
                data_type = match.group(1)  # hbo, hbr, or hbt
                max_trials = int(match.group(2))  # number after 'mt_'

                # Detect if this is an augmentation experiment
                if '_augmentations_' in experiment_folder:
                    experiment_type = "augmentation"
            else:
                # Fallback: try splitting for backward compatibility
                parts = experiment_folder.split('_')
                if len(parts) < 5 or parts[0] != 'gat' or parts[1] != 'kfold':
                    continue

                try:
                    # Handle both patterns by checking for 'augmentations' in parts
                    if 'augmentations' in parts:
                        # Pattern: gat_kfold_augmentations_hbo_mt_2_candidate_1
                        data_type = parts[3]  # data type is at index 3
                        max_trials = int(parts[5])  # number after 'mt_'
                        experiment_type = "augmentation"
                    else:
                        # Pattern: gat_kfold_hbo_mt_2_candidate_1
                        data_type = parts[2]  # data type is at index 2
                        max_trials = int(parts[4])  # number after 'mt_'
                except (IndexError, ValueError):
                    continue

            if data_type is None or max_trials is None:
                continue

            # Look for the overall k-fold results file
            overall_pkl = os.path.join(experiment_path, f"{experiment_folder}_kfold_overall.pkl")

            if not os.path.exists(overall_pkl):
                print(f"⚠ Warning: {overall_pkl} not found, skipping...")
                continue

            # Load the overall metrics
            try:
                with open(overall_pkl, 'rb') as f:
                    metrics = pickle.load(f)

                # Extract the required overall metrics
                result_row = {
                    'Date': date,
                    'Candidate': candidate_num,
                    'Experiment_Type': experiment_type,
                    'Data_Type': data_type.upper(),
                    'Max_Trials': max_trials,
                    'Overall_Accuracy': round(metrics.get('accuracy_overall', 0), 4),
                    'Overall_Precision': round(metrics.get('precision_overall', 0), 4),
                    'Overall_Recall': round(metrics.get('recall_overall', 0), 4),
                    'Overall_F1': round(metrics.get('f1_overall', 0), 4),
                    'Experiment_Folder': experiment_folder
                }

                results.append(result_row)
                print(f"✓ Loaded: {experiment_folder} ({experiment_type})")

            except Exception as e:
                print(f"✗ Error loading {overall_pkl}: {e}")
                continue

    # Create DataFrame
    if not results:
        print("No results found!")
        return None

    df = pd.DataFrame(results)

    # Sort by candidate, experiment type, and data type for better readability
    df = df.sort_values(['Candidate', 'Experiment_Type', 'Data_Type', 'Max_Trials']).reset_index(drop=True)

    # Save to CSV
    output_path = os.path.join(experiments_date_path, output_csv_filename)
    df.to_csv(output_path, index=False)
    print(f"\n✓ Results exported to: {output_path}")
    print(f"Total experiments: {len(df)}")
    print(f"  - Standard: {(df['Experiment_Type'] == 'standard').sum()}")
    print(f"  - Augmentation: {(df['Experiment_Type'] == 'augmentation').sum()}")

    return df

In [39]:
# # Export all experiments from 20251216 to CSV
# DATE = '20260428'
# experiments_path = f'/home/user/jeffrymahbuubi/PROJECTS/2-fNIRS-Graph-Base-Method/src/notebook/experiments/{DATE}/gat_kfold_hbt_mt_2_candidate_4'

# df_results = export_experiments_to_csv(experiments_path)

# # Display the results
# if df_results is not None:
#     print("\n" + "="*80)
#     print("EXPERIMENT RESULTS SUMMARY")
#     print("="*80)
#     display(df_results)

### 5.4. Analyze Best Candidate

In [40]:
def analyze_best_candidates(csv_path):
    """
    Analyze results_summary.csv to determine which candidate performs best
    for max_trials=2 and max_trials=4.

    Args:
        csv_path: Path to the results_summary.csv file

    Returns:
        Dictionary with analysis results for each max_trials setting
    """
    import pandas as pd
    import numpy as np
    from prettytable import PrettyTable

    # Load the CSV
    df = pd.read_csv(csv_path)

    print("="*90)
    print("BEST CANDIDATE ANALYSIS BY MAX_TRIALS")
    print("="*90)

    analysis_results = {}

    # Analyze for each max_trials setting
    for max_trials in sorted(df['Max_Trials'].unique()):
        print(f"\n{'='*90}")
        print(f"MAX_TRIALS = {max_trials}")
        print(f"{'='*90}\n")

        # Filter data for this max_trials
        df_mt = df[df['Max_Trials'] == max_trials].copy()

        # Group by Candidate and Data_Type, then calculate mean metrics
        grouped = df_mt.groupby(['Candidate', 'Data_Type']).agg({
            'Overall_Accuracy': 'mean',
            'Overall_Precision': 'mean',
            'Overall_Recall': 'mean',
            'Overall_F1': 'mean'
        }).reset_index()

        # Calculate average across all data types for each candidate
        candidate_avg = df_mt.groupby('Candidate').agg({
            'Overall_Accuracy': 'mean',
            'Overall_Precision': 'mean',
            'Overall_Recall': 'mean',
            'Overall_F1': 'mean'
        }).round(4)

        # Display per-candidate average performance
        table = PrettyTable()
        table.field_names = ["Candidate", "Avg Accuracy", "Avg Precision", "Avg Recall", "Avg F1"]
        table.align = "l"
        table.align["Avg Accuracy"] = "r"
        table.align["Avg Precision"] = "r"
        table.align["Avg Recall"] = "r"
        table.align["Avg F1"] = "r"

        for candidate in sorted(candidate_avg.index):
            table.add_row([
                f"Candidate {candidate}",
                f"{candidate_avg.loc[candidate, 'Overall_Accuracy']:.4f}",
                f"{candidate_avg.loc[candidate, 'Overall_Precision']:.4f}",
                f"{candidate_avg.loc[candidate, 'Overall_Recall']:.4f}",
                f"{candidate_avg.loc[candidate, 'Overall_F1']:.4f}"
            ])

        print(table)

        # Find best candidate for each metric
        best_accuracy_candidate = candidate_avg['Overall_Accuracy'].idxmax()
        best_precision_candidate = candidate_avg['Overall_Precision'].idxmax()
        best_recall_candidate = candidate_avg['Overall_Recall'].idxmax()
        best_f1_candidate = candidate_avg['Overall_F1'].idxmax()

        print(f"\n📊 Best Performers:")
        print(f"   • Highest Accuracy:  Candidate {best_accuracy_candidate} ({candidate_avg.loc[best_accuracy_candidate, 'Overall_Accuracy']:.4f})")
        print(f"   • Highest Precision: Candidate {best_precision_candidate} ({candidate_avg.loc[best_precision_candidate, 'Overall_Precision']:.4f})")
        print(f"   • Highest Recall:    Candidate {best_recall_candidate} ({candidate_avg.loc[best_recall_candidate, 'Overall_Recall']:.4f})")
        print(f"   • Highest F1:        Candidate {best_f1_candidate} ({candidate_avg.loc[best_f1_candidate, 'Overall_F1']:.4f})")

        # Detailed breakdown by data type
        print(f"\n📋 Performance by Data Type:")
        for data_type in sorted(df_mt['Data_Type'].unique()):
            df_dtype = df_mt[df_mt['Data_Type'] == data_type]
            best_candidate = df_dtype.loc[df_dtype['Overall_F1'].idxmax(), 'Candidate']
            best_f1 = df_dtype['Overall_F1'].max()
            print(f"   • {data_type}: Candidate {int(best_candidate)} (F1={best_f1:.4f})")

        # Store results
        analysis_results[max_trials] = {
            'candidate_averages': candidate_avg,
            'best_accuracy': best_accuracy_candidate,
            'best_precision': best_precision_candidate,
            'best_recall': best_recall_candidate,
            'best_f1': best_f1_candidate,
            'detailed_by_datatype': grouped
        }

        # Overall recommendation
        print(f"\n🏆 RECOMMENDATION for Max_Trials={max_trials}:")
        print(f"   Candidate {best_f1_candidate} shows the best overall performance (highest F1 score)")

    print(f"\n{'='*90}\n")

    return analysis_results

In [41]:
# # Analyze the results to find the best candidates
# DATE = '20251218'
# csv_path = f'/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/non_recurrent_gnn/flexible_gatconv2/experiments/{DATE}/no_class_weight/results_summary.csv'

# analysis = analyze_best_candidates(csv_path)

### 5.5. Analyze Best Class Weight

In [42]:
def parse_experiment_name(experiment_folder):
    """
    Parse experiment folder name to extract metadata.

    Expected format: gat_kfold_{data_type}_mt_{max_trials}_candidate_{num}

    Args:
        experiment_folder (str): Experiment folder name

    Returns:
        tuple: (data_type, max_trials, candidate_num) or (None, None, None) if parsing fails
    """
    import re

    pattern = r'^gat_kfold_([a-z]+)_mt_(\d+)_candidate_(\d+)$'
    match = re.match(pattern, experiment_folder)

    if match:
        data_type = match.group(1)
        max_trials = int(match.group(2))
        candidate_num = int(match.group(3))
        return data_type, max_trials, candidate_num

    return None, None, None

def extract_confusion_matrix(metrics):
    """
    Extract confusion matrix from metrics dictionary.

    Args:
        metrics (dict): Metrics dictionary loaded from pickle file

    Returns:
        np.ndarray or None: Confusion matrix as numpy array, or None if not found
    """
    import numpy as np

    cm_data = metrics.get('confusion_matrix_overall', None)

    if cm_data is None:
        return None

    # Handle different storage formats
    if isinstance(cm_data, list):
        return np.array(cm_data)
    elif isinstance(cm_data, np.ndarray):
        return cm_data

    return None

def parse_confusion_matrix(cm):
    """
    Parse confusion matrix to extract TN, FP, FN, TP.

    Confusion matrix format:
    [[TN, FP],
     [FN, TP]]

    Args:
        cm (np.ndarray): 2x2 confusion matrix

    Returns:
        tuple: (TN, FP, FN, TP) or (0, 0, 0, 0) if parsing fails
    """
    if cm is None:
        return 0, 0, 0, 0

    # Ensure correct shape
    if cm.shape == (2, 2):
        TN = int(cm[0, 0])
        FP = int(cm[0, 1])
        FN = int(cm[1, 0])
        TP = int(cm[1, 1])
        return TN, FP, FN, TP

    return 0, 0, 0, 0

def compare_class_weight_experiments(class_weight_experiments_path,
                                     output_csv_filename='class_weight_comparison.csv'):
    """
    Compare experiments across different class weighting strategies.

    This function scans three subdirectories (balanced, sqrt, baseline) in the
    class_weight experiments folder and compiles results from each strategy
    into a single comparison CSV file.

    Directory structure expected:
    class_weight_experiments_path/
    ├── balanced/
    │   ├── gat_kfold_hbo_mt_2_candidate_4/
    │   │   └── gat_kfold_hbo_mt_2_candidate_4_kfold_overall.pkl
    │   └── ...
    ├── sqrt/
    │   └── (same structure as balanced)
    └── baseline/
        └── (same structure as balanced)

    Args:
        class_weight_experiments_path (str):
            Path to the class_weight experiments folder
            Example: '.../experiments/20251217/class_weight/'
        output_csv_filename (str, optional):
            Output CSV filename. Default: 'class_weight_comparison.csv'

    Returns:
        pd.DataFrame or None: DataFrame with comparison results, or None if no results found

    Columns:
        - Class_Weight_Type: 'balanced', 'sqrt', or 'baseline'
        - Data_Type: 'HBO', 'HBR', or 'HBT'
        - Max_Trials: Number of maximum trials used
        - Candidate: Candidate number
        - Overall_Accuracy: Overall accuracy across all folds
        - Overall_Precision: Overall precision across all folds
        - Overall_Recall: Overall recall across all folds
        - Overall_F1: Overall F1 score across all folds
        - Specificity: True negative rate (1 - FP_Rate)
        - FP_Rate: False positive rate
        - FN_Rate: False negative rate
        - TN, FP, FN, TP: Confusion matrix components
        - Experiment_Folder: Experiment folder name
    """
    import os
    import pickle
    import pandas as pd
    import numpy as np
    from pathlib import Path

    results = []

    # Define class weight strategies to compare
    strategies = ['balanced', 'sqrt', 'baseline']

    print("=" * 80)
    print("CLASS WEIGHT EXPERIMENTS COMPARISON")
    print("=" * 80)

    # Iterate through each class weight strategy
    for strategy in strategies:
        strategy_path = os.path.join(class_weight_experiments_path, strategy)

        if not os.path.exists(strategy_path):
            print(f"\n⚠ Warning: '{strategy}' folder not found at {strategy_path}")
            print(f"   Skipping {strategy} strategy...")
            continue

        print(f"\n--- Processing '{strategy}' strategy ---")
        strategy_count = 0

        # Iterate through experiment folders in this strategy
        for experiment_folder in sorted(os.listdir(strategy_path)):
            experiment_path = os.path.join(strategy_path, experiment_folder)

            # Skip if not a directory or if it's a file
            if not os.path.isdir(experiment_path):
                continue

            # Parse experiment folder name
            data_type, max_trials, candidate_num = parse_experiment_name(experiment_folder)

            # Skip if parsing failed
            if not all([data_type, max_trials, candidate_num]):
                print(f"  ⚠ Skipping (invalid name): {experiment_folder}")
                continue

            # Construct path to overall metrics file
            overall_pkl = os.path.join(
                experiment_path,
                f"{experiment_folder}_kfold_overall.pkl"
            )

            # Check if file exists
            if not os.path.exists(overall_pkl):
                print(f"  ⚠ Skipping (no metrics file): {experiment_folder}")
                continue

            # Try to load and process metrics
            try:
                with open(overall_pkl, 'rb') as f:
                    metrics = pickle.load(f)

                # Extract standard metrics from overall statistics
                accuracy = metrics.get('accuracy_overall', 0)
                precision = metrics.get('precision_overall', 0)
                recall = metrics.get('recall_overall', 0)
                f1 = metrics.get('f1_overall', 0)

                # Extract and parse confusion matrix
                cm = extract_confusion_matrix(metrics)
                TN, FP, FN, TP = parse_confusion_matrix(cm)

                # Calculate derived metrics from confusion matrix
                # Specificity (True Negative Rate)
                specificity = TN / (TN + FP) if (TN + FP) > 0 else 0
                # False Positive Rate
                fp_rate = FP / (TN + FP) if (TN + FP) > 0 else 0
                # False Negative Rate
                fn_rate = FN / (TP + FN) if (TP + FN) > 0 else 0

                # Build result row
                result_row = {
                    'Class_Weight_Type': strategy,
                    'Data_Type': data_type.upper(),
                    'Max_Trials': max_trials,
                    'Candidate': candidate_num,
                    'Overall_Accuracy': round(accuracy, 4),
                    'Overall_Precision': round(precision, 4),
                    'Overall_Recall': round(recall, 4),
                    'Overall_F1': round(f1, 4),
                    'Specificity': round(specificity, 4),
                    'FP_Rate': round(fp_rate, 4),
                    'FN_Rate': round(fn_rate, 4),
                    'TN': TN,
                    'FP': FP,
                    'FN': FN,
                    'TP': TP,
                    'Experiment_Folder': experiment_folder
                }

                results.append(result_row)
                strategy_count += 1

                # Log progress
                print(f"  ✓ {experiment_folder}")

            except Exception as e:
                print(f"  ✗ Error loading {overall_pkl}:")
                print(f"     {str(e)}")
                continue

        print(f"  → Loaded {strategy_count} experiments")

    # Check if we found any results
    if not results:
        print("\n✗ No results found!")
        return None

    # Create DataFrame from results
    df = pd.DataFrame(results)

    # Sort for easy comparison: by data type, max trials, then class weight type
    df = df.sort_values(
        ['Data_Type', 'Max_Trials', 'Class_Weight_Type']
    ).reset_index(drop=True)

    # Save to CSV
    output_path = os.path.join(class_weight_experiments_path, output_csv_filename)
    df.to_csv(output_path, index=False)

    # Print summary
    print("\n" + "=" * 80)
    print("SUMMARY")
    print("=" * 80)
    print(f"✓ Results exported to: {output_path}")
    print(f"\nTotal experiments: {len(df)}")

    for strategy in strategies:
        count = (df['Class_Weight_Type'] == strategy).sum()
        if count > 0:
            print(f"  • {strategy:10s}: {count:2d} experiments")

    # Show data type distribution
    print(f"\nData types:")
    for data_type in sorted(df['Data_Type'].unique()):
        count = (df['Data_Type'] == data_type).sum()
        print(f"  • {data_type}: {count} experiments")

    print("\n" + "=" * 80)

    return df

In [43]:
# # Run the comparison analysis
# class_weight_path = '/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/non_recurrent_gnn/flexible_gatconv2/experiments/20251217/class_weight'

# df_comparison = compare_class_weight_experiments(class_weight_path)

## 6. Explainable AI

In [44]:
# from torch_geometric.explain import Explainer, GNNExplainer
# import matplotlib.pyplot as plt

# helper_utils.set_seed(42)
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# fold_loaders_v2 = helper_utils.get_kfold_subject_loaders_v2(
#     dataset,
#     n_splits=5,
#     batch_size=8,
#     shuffle_train=True,
#     num_workers=0,
#     pin_memory=False,
#     random_state=42,
#     show_subjects=True,
#     transform=transform
# )

# class ModelWrapper(torch.nn.Module):
#     """Wraps model to output probabilities instead of logits for binary classification."""
#     def __init__(self, model):
#         super().__init__()
#         self.model = model

#     def forward(self, x, edge_index, edge_attr=None, batch=None):
#         logits = self.model(x, edge_index, edge_attr)
#         return torch.sigmoid(logits)

# def load_model_from_checkpoint(model_path: str, model_fn, device: torch.device):
#     """Load model weights from checkpoint."""
#     model = model_fn().to(device)
#     checkpoint = torch.load(model_path, map_location=device)

#     if isinstance(checkpoint, dict) and 'model_state_dict' in checkpoint:
#         model.load_state_dict(checkpoint['model_state_dict'])
#     else:
#         model.load_state_dict(checkpoint)

#     return model

# def visualize_feature_importance(
#     explanation,
#     node_feature_names: Optional[List[str]] = None,
#     top_k: int = 10,
#     save_path: Optional[str] = None,
# ) -> None:
#     """
#     Visualize feature importance scores ranking node and edge features.

#     Args:
#         explanation: Explanation object from explainer
#         node_feature_names: Names of node features
#         top_k: Top k features to display
#         save_path: Path to save the figure
#     """
#     node_mask = explanation.node_mask

#     if node_mask is None:
#         print("No node mask available for feature importance visualization")
#         return

#     if node_mask.dim() == 1:
#         print("Node mask is 1D (object-level), cannot compute feature importance")
#         return

#     feature_importance = node_mask.sum(dim=0).detach().cpu().numpy()
#     feature_importance = np.abs(feature_importance)

#     top_indices = np.argsort(feature_importance)[-top_k:][::-1]
#     top_importance = feature_importance[top_indices]

#     if node_feature_names is None:
#         top_labels = [f'Feature {i}' for i in top_indices]
#     else:
#         top_labels = [node_feature_names[i] for i in top_indices]

#     plt.figure(figsize=(10, 6))
#     bars = plt.barh(range(len(top_labels)), top_importance, color='steelblue')
#     plt.yticks(range(len(top_labels)), top_labels)
#     plt.xlabel('Importance Score')
#     plt.title('Feature Importance Ranking (Top {})'.format(top_k))
#     plt.gca().invert_yaxis()

#     for i, v in enumerate(top_importance):
#         plt.text(v + 0.01, i, f'{v:.4f}', va='center')

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300, bbox_inches='tight')
#         print(f"Feature importance plot saved to {save_path}")
#     else:
#         plt.show()

#     plt.close()

# def visualize_node_importance(
#     explanation,
#     edge_index: torch.Tensor,
#     node_labels: Optional[List[str]] = None,
#     top_k: Optional[int] = None,
#     save_path: Optional[str] = None,
# ) -> None:
#     """
#     Visualize node importance scores showing which nodes contribute to prediction.

#     Args:
#         explanation: Explanation object from explainer
#         edge_index: Edge indices of the graph
#         node_labels: Labels/names for nodes
#         top_k: Top k nodes to highlight (if None, show all)
#         save_path: Path to save the figure
#     """
#     node_mask = explanation.node_mask

#     if node_mask is None:
#         print("No node mask available for node importance visualization")
#         return

#     if node_mask.dim() == 2:
#         node_importance = node_mask.sum(dim=1).detach().cpu().numpy()
#     else:
#         node_importance = node_mask.detach().cpu().numpy()

#     node_importance = np.abs(node_importance)
#     num_nodes = len(node_importance)

#     if top_k is None or top_k >= num_nodes:
#         top_k = num_nodes

#     top_indices = np.argsort(node_importance)[-top_k:][::-1]
#     top_importance = node_importance[top_indices]

#     if node_labels is None:
#         top_labels = [f'Channel {i}' for i in top_indices]
#     else:
#         top_labels = [node_labels[i] for i in top_indices]

#     fig, ax = plt.subplots(figsize=(12, 6))

#     colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(top_importance)))
#     bars = ax.barh(range(len(top_labels)), top_importance, color=colors)

#     ax.set_yticks(range(len(top_labels)))
#     ax.set_yticklabels(top_labels)
#     ax.set_xlabel('Importance Score', fontsize=12)
#     ax.set_title(f'Node Importance Scores (Top {top_k})', fontsize=14, fontweight='bold')
#     ax.invert_yaxis()

#     for i, (idx, v) in enumerate(zip(top_indices, top_importance)):
#         ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300, bbox_inches='tight')
#         print(f"Node importance plot saved to {save_path}")
#     else:
#         plt.show()

#     plt.close()

# def visualize_edge_importance(
#     explanation,
#     edge_index: torch.Tensor,
#     node_labels: Optional[List[str]] = None,
#     top_k: Optional[int] = None,
#     save_path: Optional[str] = None,
# ) -> None:
#     """
#     Visualize edge importance scores showing which edges contribute to prediction.

#     Args:
#         explanation: Explanation object from explainer
#         edge_index: Edge indices of the graph [2, num_edges]
#         node_labels: Labels/names for nodes
#         top_k: Top k edges to display (if None, show all)
#         save_path: Path to save the figure
#     """
#     edge_mask = explanation.edge_mask

#     if edge_mask is None:
#         print("No edge mask available for edge importance visualization")
#         return

#     if edge_mask.dim() == 2:
#         edge_importance = edge_mask.sum(dim=1).detach().cpu().numpy()
#     else:
#         edge_importance = edge_mask.detach().cpu().numpy()

#     edge_importance = np.abs(edge_importance)
#     num_edges = len(edge_importance)

#     if top_k is None or top_k >= num_edges:
#         top_k = min(num_edges, 20)

#     top_indices = np.argsort(edge_importance)[-top_k:][::-1]
#     top_importance = edge_importance[top_indices]

#     edge_index_np = edge_index.detach().cpu().numpy()

#     if node_labels is None:
#         top_labels = [f'Edge {edge_index_np[0, i]}-{edge_index_np[1, i]}'
#                      for i in top_indices]
#     else:
#         top_labels = [f'{node_labels[edge_index_np[0, i]]}-{node_labels[edge_index_np[1, i]]}'
#                      for i in top_indices]

#     fig, ax = plt.subplots(figsize=(12, 6))

#     colors = plt.cm.YlOrRd(np.linspace(0.3, 0.9, len(top_importance)))
#     bars = ax.barh(range(len(top_labels)), top_importance, color=colors)

#     ax.set_yticks(range(len(top_labels)))
#     ax.set_yticklabels(top_labels, fontsize=10)
#     ax.set_xlabel('Importance Score', fontsize=12)
#     ax.set_title(f'Edge Importance Scores (Top {top_k})', fontsize=14, fontweight='bold')
#     ax.invert_yaxis()

#     for i, v in enumerate(top_importance):
#         ax.text(v + 0.01, i, f'{v:.4f}', va='center', fontsize=10)

#     plt.tight_layout()

#     if save_path:
#         plt.savefig(save_path, dpi=300, bbox_inches='tight')
#         print(f"Edge importance plot saved to {save_path}")
#     else:
#         plt.show()

#     plt.close()

# def explain_subject_samples(
#     data_loader,
#     subject_id,
#     explainer: Explainer,
#     model: torch.nn.Module,
#     device: torch.device,
# ) -> Dict:
#     """
#     Explain all samples from a specific subject in the dataloader.

#     Args:
#         data_loader: DataLoader containing batched data
#         subject_id: Subject ID to retrieve and explain
#         explainer: Configured Explainer object from torch_geometric.explain
#         model: The graph neural network model
#         device: Device to run on (cpu/cuda)

#     Returns:
#         Dictionary containing:
#         - 'subject_id': The queried subject ID
#         - 'num_samples': Number of samples for this subject
#         - 'samples': List of dicts, each containing:
#             - 'trial_idx': Trial index
#             - 'prediction': Predicted class label (0 or 1)
#             - 'prediction_label': Prediction label string
#             - 'prediction_prob': Prediction probability
#             - 'ground_truth': Ground truth label (0 or 1)
#             - 'ground_truth_label': Ground truth label string
#             - 'node_importance': Node importance scores (np.array)
#             - 'edge_importance': Edge importance scores (np.array)
#             - 'feature_importance': Feature importance scores (np.array)
#             - 'explanation': Explanation object from explainer
#             - 'edge_index': Edge indices of the graph
#     """

#     subject_samples = []
#     label_map = {0: 'Healthy', 1: 'Anxiety'}

#     for batch in data_loader:
#         batch = batch.to(device)

#         num_graphs = batch.num_graphs if hasattr(batch, 'num_graphs') else 1

#         for graph_idx in range(num_graphs):
#             try:
#                 if num_graphs == 1:
#                     data_single = batch
#                 else:
#                     data_single = batch.get_example(graph_idx) if hasattr(batch, 'get_example') else batch

#                 data_single = data_single.to(device)

#                 current_subject_id = str(data_single.subject_id) if hasattr(data_single, 'subject_id') else None

#                 if current_subject_id is None or current_subject_id != str(subject_id):
#                     continue

#                 trial_idx = int(data_single.trial_idx) if hasattr(data_single, 'trial_idx') else None

#                 explanation = explainer(
#                     x=data_single.x,
#                     edge_index=data_single.edge_index,
#                     edge_attr=data_single.edge_attr if hasattr(data_single, 'edge_attr') else None,
#                     target=data_single.y.unsqueeze(0) if data_single.y.dim() == 0 else data_single.y,
#                     index=None
#                 )

#                 node_mask = explanation.node_mask.detach().cpu().numpy() if explanation.node_mask is not None else np.ones(data_single.x.shape[0])
#                 edge_mask = explanation.edge_mask.detach().cpu().numpy() if explanation.edge_mask is not None else np.ones(data_single.edge_index.shape[1])

#                 node_importance = np.abs(node_mask).mean(axis=1) if node_mask.ndim > 1 else np.abs(node_mask)
#                 edge_importance = np.abs(edge_mask).mean(axis=1) if edge_mask.ndim > 1 else np.abs(edge_mask)

#                 if node_mask.ndim == 2:
#                     feature_importance = np.abs(node_mask).sum(axis=0)
#                 else:
#                     feature_importance = np.zeros(data_single.x.shape[1])

#                 with torch.no_grad():
#                     pred_prob = model(
#                         data_single.x.to(device),
#                         data_single.edge_index.to(device),
#                         data_single.edge_attr.to(device) if hasattr(data_single, 'edge_attr') else None
#                     )

#                     if isinstance(pred_prob, torch.Tensor):
#                         if pred_prob.shape[-1] > 1:
#                             pred_prob_value = pred_prob.max().item()
#                             pred_class = pred_prob.argmax().item()
#                         else:
#                             pred_prob_value = pred_prob.item()
#                             pred_class = 1 if pred_prob_value > 0.5 else 0
#                     else:
#                         pred_prob_value = 0.0
#                         pred_class = 0

#                 ground_truth = int(data_single.y.item()) if hasattr(data_single, 'y') else None

#                 sample_result = {
#                     'trial_idx': trial_idx,
#                     'prediction': pred_class,
#                     'prediction_label': label_map.get(pred_class, 'Unknown'),
#                     'prediction_prob': float(pred_prob_value),
#                     'ground_truth': ground_truth,
#                     'ground_truth_label': label_map.get(ground_truth, 'Unknown') if ground_truth is not None else None,
#                     'node_importance': node_importance,
#                     'edge_importance': edge_importance,
#                     'feature_importance': feature_importance,
#                     'explanation': explanation,
#                     'edge_index': data_single.edge_index.detach().cpu(),
#                 }

#                 subject_samples.append(sample_result)

#             except Exception as e:
#                 print(f"Error processing sample in subject {subject_id}: {e}")
#                 continue

#     return {
#         'subject_id': str(subject_id),
#         'num_samples': len(subject_samples),
#         'samples': subject_samples,
#     }

# def get_subject_summary(
#     subject_results: Dict,
# ) -> Dict:
#     """
#     Generate summary statistics for a subject's explanation results.

#     Args:
#         subject_results: Dictionary returned from explain_subject_samples()

#     Returns:
#         Dictionary containing summary statistics
#     """

#     if subject_results['num_samples'] == 0:
#         return {
#             'subject_id': subject_results['subject_id'],
#             'num_samples': 0,
#             'accuracy': None,
#             'mean_node_importance': None,
#             'mean_edge_importance': None,
#             'predictions': [],
#             'ground_truths': [],
#         }

#     samples = subject_results['samples']

#     predictions = [s['prediction'] for s in samples]
#     ground_truths = [s['ground_truth'] for s in samples]
#     node_importances = [s['node_importance'] for s in samples]
#     edge_importances = [s['edge_importance'] for s in samples]

#     accuracy = None
#     if all(gt is not None for gt in ground_truths):
#         accuracy = sum(1 for p, gt in zip(predictions, ground_truths) if p == gt) / len(predictions)

#     mean_node_imp = np.mean([np.mean(ni) for ni in node_importances])
#     mean_edge_imp = np.mean([np.mean(ei) for ei in edge_importances])

#     return {
#         'subject_id': subject_results['subject_id'],
#         'num_samples': subject_results['num_samples'],
#         'accuracy': accuracy,
#         'mean_node_importance': float(mean_node_imp),
#         'mean_edge_importance': float(mean_edge_imp),
#         'predictions': predictions,
#         'ground_truths': ground_truths,
#         'trial_indices': [s['trial_idx'] for s in samples],
#     }

### 6.1. Explainer Setup

In [45]:
# candidate_3 = {
#     "n_layers": 5,
#     "n_filters": [384, 128, 256, 384, 256],
#     "fc_size": 64,
#     "dropout_rate": 0.3,
#     "learning_rate": 0.0000618,
# }

# def model_fn():
#     return FlexibleGINNet(
#         in_channels=6,              # 6 statistical features per channel
#         n_layers=candidate_3["n_layers"],
#         n_filters=candidate_3["n_filters"],
#         fc_size=candidate_3["fc_size"],
#         dropout_rate=candidate_3["dropout_rate"],
#         edge_dim=2,                 # Pearson correlation + coherence
#         n_classes=2                 # Binary: healthy vs anxiety
#     )

# FOLD_INDEX = 0

# # Construct checkpoint path
# checkpoint_dir = '/home/user/jeffrymahbuubi/1-GNN-Development/fNIRS-Project/notebook/experiments/20251130/candidate_3/mt_2/gine_kfold_hbo_mt_2_candidate_3'
# checkpoint_name = f'gine_kfold_hbo_mt_2_candidate_3_fold_{FOLD_INDEX + 1}_best.pt'
# model_path = os.path.join(checkpoint_dir, checkpoint_name)

# # Load model
# model = load_model_from_checkpoint(model_path, model_fn, device)
# model = ModelWrapper(model)  # Wrap for probability output
# model.eval()

# # Get validation loader for the selected fold
# train_loader, val_loader = fold_loaders_v2[FOLD_INDEX]

# print(f"Loaded model from: {model_path}")
# print(f"Fold {FOLD_INDEX + 1} - Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

# def setup_explainer(
#         model: torch.nn.Module,
#         device: torch.device,
# ) -> Explainer:

#     model = model.to(device)

#     explainer = Explainer(
#         model=model,
#         algorithm=GNNExplainer(epochs=50, lr=0.005),
#         explanation_type='model',
#         node_mask_type='attributes',
#         edge_mask_type='object',
#         model_config=dict(
#             mode='binary_classification',
#             task_level='graph',
#             return_type='probs'
#         )
#     )

#     return explainer

# # Define feature names based on dataset configuration
# FEATURE_NAMES = ["mean", "min", "max", "skewness", "kurtosis", "variance"]

### 6.2. Single-Subject

In [46]:
# # Setup explainer
# explainer = setup_explainer(model=model, device=device)

# # Option 1: Explain specific subject
# subject_id = 'AA056'  # User specifies
# subject_results = explain_subject_samples(
#     data_loader=val_loader,
#     subject_id=subject_id,
#     explainer=explainer,
#     model=model,
#     device=device,
# )

# # Print results
# print(f"\n{'='*60}")
# print(f"SUBJECT {subject_results['subject_id']} EXPLANATION")
# print(f"{'='*60}")
# print(f"Total samples: {subject_results['num_samples']}")

# for sample in subject_results['samples']:
#     print(f"\nTrial {sample['trial_idx']}:")
#     print(f"  Prediction: {sample['prediction_label']} (prob: {sample['prediction_prob']:.4f})")
#     print(f"  Ground Truth: {sample['ground_truth_label']}")
#     print(f"  Correct: {'✓' if sample['prediction'] == sample['ground_truth'] else '✗'}")

# # Get summary statistics
# summary = get_subject_summary(subject_results)

# print(f"\n{'='*60}")
# print(f"SUBJECT {summary['subject_id']} SUMMARY")
# print(f"{'='*60}")
# print(f"Number of samples: {summary['num_samples']}")
# if summary['accuracy'] is not None:
#     print(f"Accuracy: {summary['accuracy']:.2%}")
# print(f"Mean node importance: {summary['mean_node_importance']:.4f}")
# print(f"Mean edge importance: {summary['mean_edge_importance']:.4f}")
# print(f"Trials: {summary['trial_indices']}")

# # Visualize the first sample
# sample = subject_results['samples'][0]

# visualize_feature_importance(
#     sample['explanation'],
#     node_feature_names=FEATURE_NAMES,
#     top_k=6,  # Show all 6 features
# )

# visualize_node_importance(
#     sample['explanation'],
#     sample['edge_index'],
#     top_k=10,  # Show top 10 out of 23 channels
# )

# visualize_edge_importance(
#     sample['explanation'],
#     sample['edge_index'],
#     top_k=10,  # Show top 10 important connections
# )